# Repository copy
This is a lightweight copy of the executable research notebook with cell outputs removed. The scientific source code is preserved. Some cells contain the original Google Colab/Drive paths; for exact reproduction, use the archived data/splits/annotations in this repository and update only path variables as documented in `REPRODUCIBILITY.md`.


# **Mount Drive + set paths**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random, shutil, zipfile
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================
# Old/original dataset used for the first training run
OLD_IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
OLD_COCO_ALL_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/1.json"

# New CVAT export ZIPs (upload to Colab /content, or place on Drive and update the paths)
# Order matters: later zips override earlier ones if filenames overlap.
NEW_CVAT_ZIPS = [
    "/content/coco_new.zip",
    "/content/coco_new2_RENAMED.zip",
]

# Output root for the merged + retrained experiment
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"
os.makedirs(OUT_ROOT, exist_ok=True)

# Working folders
WORK_ROOT = os.path.join(OUT_ROOT, "merged_dataset_work")
EXTRACT_ROOT = os.path.join(WORK_ROOT, "cvat_exports_extracted")
MERGED_IMG_DIR = os.path.join(WORK_ROOT, "merged_images")
MERGED_JSON_DIR = os.path.join(WORK_ROOT, "merged_annotations")
os.makedirs(EXTRACT_ROOT, exist_ok=True)
os.makedirs(MERGED_IMG_DIR, exist_ok=True)
os.makedirs(MERGED_JSON_DIR, exist_ok=True)

# Downstream output folders
FIG_DIR   = os.path.join(OUT_ROOT, "figures")
PRED_DIR  = os.path.join(OUT_ROOT, "pred_masks")
OVR_DIR   = os.path.join(OUT_ROOT, "overlays")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(OVR_DIR, exist_ok=True)

print("OLD_IMG_DIR exists:", os.path.isdir(OLD_IMG_DIR), "| #files:", len(os.listdir(OLD_IMG_DIR)) if os.path.isdir(OLD_IMG_DIR) else 0)
print("OLD_COCO_ALL_JSON exists:", os.path.exists(OLD_COCO_ALL_JSON))
print("NEW_CVAT_ZIPS:")
for zp in NEW_CVAT_ZIPS:
    print("  ", zp, "| exists:", os.path.exists(zp))
print("OUT_ROOT:", OUT_ROOT)


# **Install Detectron2**

In [ ]:
!python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available(), 'cuda ver', torch.version.cuda)"

# If CUDA is available but detectron2 not installed, install detectron2
try:
    import detectron2
    print("detectron2:", detectron2.__version__)
except Exception as e:
    print("Installing detectron2...")
    !pip -q install 'git+https://github.com/facebookresearch/detectron2.git'
    import detectron2
    print("detectron2 installed:", detectron2.__version__)

# **Merge old gamma'_ annotations + new CVAT export, then create train/val COCO splits**

In [ ]:
# ============================================================
# Merge old gamma_prime annotations + multiple corrected CVAT exports
# Later exports override earlier ones on overlapping filenames.
# ============================================================

def _norm_name(x):
    return os.path.basename(str(x)).replace('\\', '/').strip().lower()

def _basename_keep_case(x):
    return os.path.basename(str(x)).replace('\\', '/').strip()

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f)

def keep_single_class_coco(coco, keep_name='gamma_prime'):
    cats = coco.get('categories', [])
    name_to_id = {c['name']: c['id'] for c in cats}
    if keep_name not in name_to_id:
        raise ValueError(f"Category '{keep_name}' not found. Found: {list(name_to_id.keys())}")
    keep_old_id = name_to_id[keep_name]

    anns = []
    for ann in coco.get('annotations', []):
        if ann.get('category_id') == keep_old_id:
            a = dict(ann)
            a['category_id'] = 1
            anns.append(a)

    keep_img_ids = {a['image_id'] for a in anns}
    images = [dict(im) for im in coco.get('images', []) if im['id'] in keep_img_ids]

    out = {
        'licenses': coco.get('licenses', []),
        'info': coco.get('info', {}),
        'images': images,
        'annotations': anns,
        'categories': [{'id': 1, 'name': keep_name, 'supercategory': ''}],
    }
    return out

def find_cvat_json_and_image_root(zip_path, extract_dir):
    if os.path.isdir(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

    json_candidates = list(Path(extract_dir).rglob('instances_default.json'))
    if not json_candidates:
        json_candidates = list(Path(extract_dir).rglob('*.json'))
    if not json_candidates:
        raise FileNotFoundError(f'No annotation JSON found inside the CVAT export zip: {zip_path}')
    ann_json = str(json_candidates[0])

    img_candidates = []
    for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']:
        img_candidates.extend(Path(extract_dir).rglob(ext))
    if not img_candidates:
        raise FileNotFoundError(f'No images found inside the CVAT export zip: {zip_path}')

    image_root = os.path.commonpath([str(p.parent) for p in img_candidates])
    return ann_json, image_root

def build_filename_lookup(folder):
    lookup = {}
    for fn in os.listdir(folder):
        full = os.path.join(folder, fn)
        if os.path.isfile(full):
            lookup[_norm_name(fn)] = full
    return lookup

def build_dataset_entry_from_coco(coco, image_lookup, source_tag):
    anns_by_img = {}
    for ann in coco['annotations']:
        anns_by_img.setdefault(ann['image_id'], []).append(ann)

    entries = {}
    for im in coco['images']:
        nm = _norm_name(im['file_name'])
        entries[nm] = {
            'norm_name': nm,
            'file_name': _basename_keep_case(im['file_name']),
            'img_record': dict(im),
            'anns': [dict(a) for a in anns_by_img.get(im['id'], [])],
            'src_path': image_lookup.get(nm),
            'source': source_tag,
        }
    return entries

def load_new_zip_dataset(zip_path, extract_root, zip_idx):
    extract_dir = os.path.join(extract_root, f"zip_{zip_idx:02d}")
    ann_json, img_root = find_cvat_json_and_image_root(zip_path, extract_dir)
    coco_full = load_json(ann_json)
    coco = keep_single_class_coco(coco_full, keep_name='gamma_prime')

    lookup = {}
    for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']:
        for p in Path(img_root).rglob(ext):
            lookup[_norm_name(p.name)] = str(p)

    return {
        'zip_path': zip_path,
        'ann_json': ann_json,
        'img_root': img_root,
        'coco': coco,
        'lookup': lookup,
        'source': f'new_zip_{zip_idx}',
    }

def merge_old_and_new_gamma_prime(old_json_path, old_img_dir, new_zip_paths, out_img_dir, out_json_path, extract_root):
    old_coco_full = load_json(old_json_path)
    old_coco = keep_single_class_coco(old_coco_full, keep_name='gamma_prime')
    old_lookup = build_filename_lookup(old_img_dir)

    old_entries = build_dataset_entry_from_coco(old_coco, old_lookup, 'old_original')

    new_datasets = []
    for idx, zp in enumerate(new_zip_paths, start=1):
        if not os.path.exists(zp):
            raise FileNotFoundError(f"Expected CVAT zip not found: {zp}")
        new_datasets.append(load_new_zip_dataset(zp, extract_root, idx))

    # Override policy: old -> zip1 -> zip2 -> ... (later wins on overlap)
    merged_entries = dict(old_entries)
    replacement_log = []

    for ds in new_datasets:
        ds_entries = build_dataset_entry_from_coco(ds['coco'], ds['lookup'], ds['source'])
        for nm, entry in ds_entries.items():
            prev_source = merged_entries[nm]['source'] if nm in merged_entries else None
            if prev_source is not None:
                replacement_log.append({
                    'file_name': entry['file_name'],
                    'replaced_source': prev_source,
                    'new_source': ds['source'],
                    'zip_path': ds['zip_path'],
                })
            merged_entries[nm] = entry

    if os.path.isdir(out_img_dir):
        shutil.rmtree(out_img_dir)
    os.makedirs(out_img_dir, exist_ok=True)

    merged = {
        'licenses': old_coco.get('licenses', []),
        'info': {
            'description': 'Merged gamma_prime dataset: old training set + multiple corrected CVAT exports. Later exports override earlier ones on overlapping filenames.',
            'old_json': old_json_path,
            'new_zips': list(new_zip_paths),
        },
        'images': [],
        'annotations': [],
        'categories': [{'id': 1, 'name': 'gamma_prime', 'supercategory': ''}],
    }

    next_img_id = 1
    next_ann_id = 1
    merge_rows = []

    for nm in sorted(merged_entries.keys()):
        entry = merged_entries[nm]
        img_record = entry['img_record']
        anns_for_img = entry['anns']
        src_path = entry['src_path']

        if src_path is None or not os.path.exists(src_path):
            raise FileNotFoundError(f'Missing image file for {img_record.get("file_name")}: {src_path}')

        out_name = entry['file_name']
        out_path = os.path.join(out_img_dir, out_name)
        shutil.copy2(src_path, out_path)

        new_img = dict(img_record)
        new_img['id'] = next_img_id
        new_img['file_name'] = out_name
        merged['images'].append(new_img)

        for ann in anns_for_img:
            a = dict(ann)
            a['id'] = next_ann_id
            a['image_id'] = next_img_id
            a['category_id'] = 1
            merged['annotations'].append(a)
            next_ann_id += 1

        merge_rows.append({
            'file_name': out_name,
            'final_source': entry['source'],
            'height': img_record.get('height'),
            'width': img_record.get('width'),
            'n_annotations': len(anns_for_img),
            'was_in_old': nm in old_entries,
            'is_truly_new_vs_old': nm not in old_entries,
        })
        next_img_id += 1

    save_json(merged, out_json_path)

    merge_report = pd.DataFrame(merge_rows)
    report_csv = os.path.join(os.path.dirname(out_json_path), 'merge_report.csv')
    merge_report.to_csv(report_csv, index=False)

    replacement_df = pd.DataFrame(replacement_log)
    replacement_csv = os.path.join(os.path.dirname(out_json_path), 'replacement_log.csv')
    replacement_df.to_csv(replacement_csv, index=False)

    per_zip_summary = []
    old_name_set = set(old_entries.keys())
    seen_before = set(old_entries.keys())
    for ds in new_datasets:
        ds_entries = build_dataset_entry_from_coco(ds['coco'], ds['lookup'], ds['source'])
        ds_names = set(ds_entries.keys())
        truly_new = ds_names - seen_before
        overlaps_with_previous = ds_names & seen_before
        overlaps_with_old = ds_names & old_name_set
        per_zip_summary.append({
            'source': ds['source'],
            'zip_path': ds['zip_path'],
            'images_gamma_prime': len(ds['coco']['images']),
            'annotations_gamma_prime': len(ds['coco']['annotations']),
            'truly_new_images_added_vs_previous_sources': len(truly_new),
            'overlap_with_previous_sources': len(overlaps_with_previous),
            'overlap_with_original_old_dataset': len(overlaps_with_old),
        })
        seen_before |= ds_names

    final_new_vs_old = [r for r in merge_rows if r['is_truly_new_vs_old']]
    final_corrected_old = [r for r in merge_rows if (r['was_in_old'] and r['final_source'] != 'old_original')]

    summary = {
        'old_images_gamma_prime': len(old_coco['images']),
        'old_annotations_gamma_prime': len(old_coco['annotations']),
        'n_new_zip_exports': len(new_datasets),
        'per_zip_summary': per_zip_summary,
        'final_corrected_old_images': len(final_corrected_old),
        'final_truly_new_images_added_vs_old': len(final_new_vs_old),
        'merged_images_total': len(merged['images']),
        'merged_annotations_total': len(merged['annotations']),
        'merged_json': out_json_path,
        'merged_img_dir': out_img_dir,
        'merge_report_csv': report_csv,
        'replacement_log_csv': replacement_csv,
    }
    return merged, summary, merge_report, replacement_df

MERGED_COCO_JSON = os.path.join(MERGED_JSON_DIR, 'merged_gamma_prime_coco.json')
merged_coco, merge_summary, merge_report_df, replacement_df = merge_old_and_new_gamma_prime(
    old_json_path=OLD_COCO_ALL_JSON,
    old_img_dir=OLD_IMG_DIR,
    new_zip_paths=NEW_CVAT_ZIPS,
    out_img_dir=MERGED_IMG_DIR,
    out_json_path=MERGED_COCO_JSON,
    extract_root=EXTRACT_ROOT,
)

# IMPORTANT: downstream cells will now use the merged dataset automatically
IMG_DIR = MERGED_IMG_DIR
COCO_ALL_JSON = MERGED_COCO_JSON

print('Merge summary:')
print(json.dumps(merge_summary, indent=2))
display(merge_report_df.head())
if len(replacement_df) > 0:
    print("Examples of replacements made by newer sources:")
    display(replacement_df.head(10))

# ============================================================
# Split merged COCO into train/val JSONs
# ============================================================
def split_coco_json(in_json_path, out_train_path, out_val_path, val_ratio=0.2, seed=42):
    """Split a single COCO JSON into train/val JSONs by image, preserving ids."""
    with open(in_json_path, 'r') as f:
        coco = json.load(f)

    images = coco.get('images', [])
    anns   = coco.get('annotations', [])

    img_ids = [im['id'] for im in images]
    rnd = random.Random(seed)
    rnd.shuffle(img_ids)

    n_val = max(1, int(len(img_ids) * val_ratio))
    val_ids   = set(img_ids[:n_val])
    train_ids = set(img_ids[n_val:])

    train_images = [im for im in images if im['id'] in train_ids]
    val_images   = [im for im in images if im['id'] in val_ids]

    train_anns = [a for a in anns if a['image_id'] in train_ids]
    val_anns   = [a for a in anns if a['image_id'] in val_ids]

    base = {k: v for k, v in coco.items() if k not in ['images', 'annotations']}
    train_coco = dict(base); train_coco['images'] = train_images; train_coco['annotations'] = train_anns
    val_coco   = dict(base); val_coco['images']   = val_images;   val_coco['annotations']   = val_anns

    os.makedirs(os.path.dirname(out_train_path), exist_ok=True)
    with open(out_train_path, 'w') as f:
        json.dump(train_coco, f)
    with open(out_val_path, 'w') as f:
        json.dump(val_coco, f)

    return {
        'in_json': in_json_path,
        'train_json': out_train_path,
        'val_json': out_val_path,
        'n_images': len(images),
        'n_train_images': len(train_images),
        'n_val_images': len(val_images),
        'n_anns': len(anns),
        'n_train_anns': len(train_anns),
        'n_val_anns': len(val_anns),
    }

SPLIT_DIR = os.path.join(OUT_ROOT, 'coco_splits')
COCO_TRAIN_JSON = os.path.join(SPLIT_DIR, 'train.json')
COCO_VAL_JSON   = os.path.join(SPLIT_DIR, 'val.json')

split_info = split_coco_json(
    COCO_ALL_JSON,
    COCO_TRAIN_JSON,
    COCO_VAL_JSON,
    val_ratio=0.2,
    seed=42,
)
print('Split info:', split_info)

# These names are kept for backward compatibility with the rest of the notebook
SINGLE_TRAIN_JSON = COCO_TRAIN_JSON
SINGLE_VAL_JSON   = COCO_VAL_JSON

# **Verify image files exist for the COCO split**

In [ ]:
def coco_list_files(coco_json):
    coco = json.load(open(coco_json, "r"))
    return [im["file_name"] for im in coco["images"]]

train_files = coco_list_files(SINGLE_TRAIN_JSON)
val_files   = coco_list_files(SINGLE_VAL_JSON)

missing_train = [f for f in train_files if not os.path.exists(os.path.join(IMG_DIR, f))]
missing_val   = [f for f in val_files   if not os.path.exists(os.path.join(IMG_DIR, f))]

print("Single-class TRAIN images:", len(train_files), "| missing:", len(missing_train))
print("Single-class VAL   images:", len(val_files),   "| missing:", len(missing_val))

if missing_train[:5]:
    print("Example missing train:", missing_train[:5])
if missing_val[:5]:
    print("Example missing val:", missing_val[:5])

assert len(missing_val) == 0, "VAL split has missing images in IMG_DIR. Fix IMG_DIR or COCO file_name."

# **Register datasets (Detectron2)**

In [ ]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog

DATASET_TRAIN = "gamma_prime_train"
DATASET_VAL   = "gamma_prime_val"

# If it is already registered, remove and re-register cleanly
for name in [DATASET_TRAIN, DATASET_VAL]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

register_coco_instances(DATASET_TRAIN, {}, SINGLE_TRAIN_JSON, IMG_DIR)
register_coco_instances(DATASET_VAL,   {}, SINGLE_VAL_JSON,   IMG_DIR)

meta = MetadataCatalog.get(DATASET_TRAIN)
meta.thing_classes = ["gamma_prime"]

print("Registered datasets:", DatasetCatalog.list())
print("Train samples:", len(DatasetCatalog.get(DATASET_TRAIN)))
print("Val samples:", len(DatasetCatalog.get(DATASET_VAL)))

# **Ground Truth visualization (overlay) + save figure**

In [ ]:
import cv2
import matplotlib.pyplot as plt

from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import DatasetCatalog, MetadataCatalog

def show_gt_examples(dataset_name, n=6, save_path=None):
    dset = DatasetCatalog.get(dataset_name)
    meta = MetadataCatalog.get(dataset_name)

    picks = random.sample(dset, min(n, len(dset)))

    cols = 3
    rows = int(np.ceil(len(picks)/cols))
    plt.figure(figsize=(5*cols, 5*rows))

    for i, rec in enumerate(picks):
        img_path = rec["file_name"]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        v = Visualizer(img, metadata=meta, scale=1.0, instance_mode=ColorMode.IMAGE)
        out = v.draw_dataset_dict(rec)

        ax = plt.subplot(rows, cols, i+1)
        ax.imshow(out.get_image())
        ax.set_title(f"{os.path.basename(img_path)} | GT inst={len(rec['annotations'])}")
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()

gt_save = os.path.join(FIG_DIR, "fig_gt_overlays_val.png")
show_gt_examples(DATASET_VAL, n=6, save_path=gt_save)

# **Train Mask R-CNN (γ′ 1-class) + save model**

In [ ]:
import math
import os
import torch
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, hooks
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import build_detection_train_loader, DatasetMapper
from detectron2.data import transforms as T
from detectron2.evaluation import COCOEvaluator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ============================================================
# Sigma-push training config for dense/small gamma-prime particles
# ============================================================
MODEL_CFG = "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"

train_samples = len(DatasetCatalog.get(DATASET_TRAIN))
val_samples = len(DatasetCatalog.get(DATASET_VAL))
batch_size = 2
iters_per_epoch = max(1, math.ceil(train_samples / batch_size))

# Train longer now that the dataset is larger
target_epochs = 260
max_iter = min(36000, max(18000, target_epochs * iters_per_epoch))
step1 = int(max_iter * 0.70)
step2 = int(max_iter * 0.88)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(MODEL_CFG))

cfg.DATASETS.TRAIN = (DATASET_TRAIN,)
cfg.DATASETS.TEST  = (DATASET_VAL,)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.DEVICE = DEVICE
cfg.OUTPUT_DIR = os.path.join(OUT_ROOT, "detectron2_output")
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(MODEL_CFG)

# Solver / LR schedule
cfg.SOLVER.IMS_PER_BATCH = batch_size
cfg.SOLVER.BASE_LR = 1.0e-4
cfg.SOLVER.MAX_ITER = max_iter
cfg.SOLVER.STEPS = (step1, step2)
cfg.SOLVER.GAMMA = 0.1
cfg.SOLVER.WARMUP_ITERS = min(1500, max(500, int(0.08 * max_iter)))
cfg.SOLVER.CHECKPOINT_PERIOD = max(500, iters_per_epoch * 5)
cfg.SOLVER.AMP.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0
cfg.SOLVER.CLIP_GRADIENTS.NORM_TYPE = 2.0

cfg.TEST.EVAL_PERIOD = max(500, iters_per_epoch * 5)

# Small-object / dense-instance tuning
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[4, 8, 16, 32, 64, 128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]

cfg.MODEL.RPN.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.RPN.NMS_THRESH = 0.7
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 12000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 6000
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST  = 6000
cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 4000

cfg.TEST.DETECTIONS_PER_IMAGE = 3000
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.35

# Higher resolution to preserve tiny particles
cfg.INPUT.MIN_SIZE_TRAIN = (768, 896, 1024)
cfg.INPUT.MAX_SIZE_TRAIN = 1280
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 1280

BEST_METRIC = "segm/AP50"
BEST_WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_best_ap50.pth")

class GammaPrimeTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        output_folder = output_folder or os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, output_dir=output_folder)

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(
            cfg,
            is_train=True,
            augmentations=[
                T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
                T.RandomFlip(prob=0.5, horizontal=False, vertical=True),
                T.RandomRotation(angle=[0, 90, 180, 270], sample_style="choice"),
                T.RandomBrightness(0.9, 1.1),
                T.RandomContrast(0.9, 1.1),
            ],
        )
        return build_detection_train_loader(cfg, mapper=mapper)

    def build_hooks(self):
        hooks_list = super().build_hooks()
        hooks_list.insert(
            -1,
            hooks.BestCheckpointer(
                self.cfg.TEST.EVAL_PERIOD,
                DetectionCheckpointer(self.model, self.cfg.OUTPUT_DIR),
                BEST_METRIC,
                mode="max",
                file_prefix="model_best_ap50",
            ),
        )
        return hooks_list

print(f"Train images: {train_samples} | Val images: {val_samples}")
print(f"Iterations/epoch: {iters_per_epoch}")
print(f"MAX_ITER: {max_iter} | STEPS: {(step1, step2)}")
print("Backbone config:", MODEL_CFG)
print("Train size:", cfg.INPUT.MIN_SIZE_TRAIN, "| Test size:", cfg.INPUT.MIN_SIZE_TEST)
print("Best metric checkpoint:", BEST_WEIGHTS)

trainer = GammaPrimeTrainer(cfg)
trainer.resume_or_load(resume=False)


In [ ]:
trainer.train()

print("Training finished. Output:", cfg.OUTPUT_DIR)
print("Final weights:", os.path.join(cfg.OUTPUT_DIR, "model_final.pth"))

# **Evaluate on VAL (COCO AP) and save results JSON**

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

FINAL_WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
print("Evaluating weights:", FINAL_WEIGHTS)

# Make sure the trainer model uses the chosen weights
DetectionCheckpointer(trainer.model).load(FINAL_WEIGHTS)

evaluator = COCOEvaluator(DATASET_VAL, output_dir=cfg.OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, DATASET_VAL)

results = inference_on_dataset(trainer.model, val_loader, evaluator)
print(results)

# Save as json for figures
res_path = os.path.join(OUT_ROOT, "coco_eval_results.json")
with open(res_path, "w") as f:
    json.dump(results, f, indent=2)
print("Saved:", res_path)

summary = {
    "final_weights_used": FINAL_WEIGHTS,
    "best_metric": BEST_METRIC,
    "segm_AP": results.get("segm", {}).get("AP", None),
    "segm_AP50": results.get("segm", {}).get("AP50", None),
    "segm_AP75": results.get("segm", {}).get("AP75", None),
}
with open(os.path.join(OUT_ROOT, "evaluation_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Saved:", os.path.join(OUT_ROOT, "evaluation_summary.json"))


# **COCO AP50/AP75 bar chart**

In [ ]:
import matplotlib.pyplot as plt

with open(os.path.join(OUT_ROOT, "coco_eval_results.json"), "r") as f:
    R = json.load(f)

ap50 = R["segm"]["AP50"]
ap75 = R["segm"]["AP75"]
ap   = R["segm"]["AP"]

vals = [ap, ap50, ap75]
labels = ["AP(50:95)", "AP50", "AP75"]

plt.figure(figsize=(6,4))
x = np.arange(len(vals))
plt.bar(x, vals)
plt.xticks(x, labels)
plt.ylabel("Segmentation AP")
plt.title("Mask R-CNN (γ′) — COCO Segmentation AP")
for i,v in enumerate(vals):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout()

save_path = os.path.join(FIG_DIR, "fig_coco_ap_bars.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

# **Prediction overlays (VAL)**

In [ ]:
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer, ColorMode

cfg_pred = cfg.clone()
cfg_pred.MODEL.WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg_pred.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.20
predictor = DefaultPredictor(cfg_pred)

def show_pred_examples(dataset_name, n=6, save_path=None):
    dset = DatasetCatalog.get(dataset_name)
    meta = MetadataCatalog.get(dataset_name)

    picks = random.sample(dset, min(n, len(dset)))

    cols = 3
    rows = int(np.ceil(len(picks)/cols))
    plt.figure(figsize=(5*cols, 5*rows))

    for i, rec in enumerate(picks):
        img_path = rec["file_name"]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        out = predictor(img_rgb)
        inst = out["instances"].to("cpu")

        v = Visualizer(img_rgb, metadata=meta, scale=1.0, instance_mode=ColorMode.IMAGE)
        vis = v.draw_instance_predictions(inst)

        ax = plt.subplot(rows, cols, i+1)
        ax.imshow(vis.get_image())
        ax.set_title(f"{os.path.basename(img_path)} | pred inst={len(inst)}")
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()

pred_save = os.path.join(FIG_DIR, "fig_pred_overlays_val.png")
show_pred_examples(DATASET_VAL, n=6, save_path=pred_save)


# **Threshold sweep (0.05 -> 0.5): mean Dice + mean instance count**

In [ ]:
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from skimage.morphology import remove_small_objects

coco = COCO(SINGLE_VAL_JSON)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def gt_union_mask(img_id):
    img = coco.loadImgs(img_id)[0]
    H, W = img["height"], img["width"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union, img["file_name"]

def dice(a, b, eps=1e-7):
    a = (a > 0).astype(np.uint8)
    b = (b > 0).astype(np.uint8)
    inter = (a & b).sum()
    return (2*inter + eps) / (a.sum() + b.sum() + eps)

def apply_union_postprocess(mask01, min_size=16):
    mask01 = (mask01 > 0)
    if min_size and min_size > 1:
        mask01 = remove_small_objects(mask01, min_size=min_size)
    return mask01.astype(np.uint8)

img_ids = coco.getImgIds()
print("VAL images:", len(img_ids))

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
rows = []

for thr in thresholds:
    cfg_thr = cfg.clone()
    cfg_thr.MODEL.WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    cfg_thr.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(thr)
    predictor_thr = DefaultPredictor(cfg_thr)

    dices = []
    inst_counts = []

    for img_id in img_ids:
        gt, fname = gt_union_mask(img_id)
        img_path = os.path.join(IMG_DIR, fname)
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        out = predictor_thr(img_rgb)
        inst = out["instances"].to("cpu")
        inst_counts.append(int(len(inst)))

        if len(inst) == 0:
            pred_union = np.zeros_like(gt, dtype=np.uint8)
        else:
            pm = inst.pred_masks.numpy().astype(np.uint8)  # [N,H,W]
            pred_union = (np.any(pm, axis=0)).astype(np.uint8)

        pred_union = apply_union_postprocess(pred_union, min_size=16)
        dices.append(dice(gt, pred_union))

    rows.append({
        "score_thresh": thr,
        "mean_dice": float(np.mean(dices)),
        "std_dice": float(np.std(dices)),
        "mean_pred_inst": float(np.mean(inst_counts)),
    })

df_thr = pd.DataFrame(rows).sort_values(["mean_dice", "score_thresh"], ascending=[False, True]).reset_index(drop=True)
print(df_thr)

best_row = df_thr.iloc[0].to_dict()
BEST_SCORE_THRESH = float(best_row["score_thresh"])
print("Selected BEST_SCORE_THRESH:", BEST_SCORE_THRESH)

# plot
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(df_thr["score_thresh"], df_thr["mean_dice"], marker="o")
plt.xlabel("Score threshold")
plt.ylabel("Mean Dice (union mask)")
plt.title("Threshold sweep — Dice vs score threshold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "fig_threshold_sweep_dice.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

df_thr.to_csv(os.path.join(OUT_ROOT, "threshold_sweep.csv"), index=False)
with open(os.path.join(OUT_ROOT, "threshold_sweep_best.json"), "w") as f:
    json.dump({"BEST_SCORE_THRESH": BEST_SCORE_THRESH, "best_row": best_row}, f, indent=2)
print("Saved CSV:", os.path.join(OUT_ROOT, "threshold_sweep.csv"))
print("Saved JSON:", os.path.join(OUT_ROOT, "threshold_sweep_best.json"))


# **Deployment: run on ALL images -> save masks + overlays + CSV**

In [ ]:
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects

def compute_shape_metrics_from_union(mask01):
    mask01 = (mask01 > 0).astype(np.uint8)

    area_frac = mask01.mean()

    cc = label(mask01)
    props = regionprops(cc)

    if len(props) == 0:
        return {
            "gamma_prime_area_fraction": float(area_frac),
            "gamma_prime_cc_count": 0,
            "gamma_prime_mean_eq_diameter_px": 0.0,
            "gamma_prime_mean_aspect_ratio": 0.0,
            "gamma_prime_mean_circularity": 0.0,
        }

    eqd = []
    ar  = []
    circ = []

    for p in props:
        eqd.append(p.equivalent_diameter)
        maj = p.major_axis_length if p.major_axis_length > 1e-9 else np.nan
        mino = p.minor_axis_length if p.minor_axis_length > 1e-9 else np.nan
        ar.append(float(maj/mino) if (mino and not np.isnan(mino) and not np.isnan(maj)) else np.nan)

        per = p.perimeter if p.perimeter > 1e-9 else np.nan
        a = p.area
        circ.append(float(4*np.pi*a/(per*per)) if (per and not np.isnan(per)) else np.nan)

    eqd = np.array(eqd, dtype=float)
    ar  = np.array(ar, dtype=float)
    circ= np.array(circ, dtype=float)

    return {
        "gamma_prime_area_fraction": float(area_frac),
        "gamma_prime_cc_count": int(len(props)),
        "gamma_prime_mean_eq_diameter_px": float(np.nanmean(eqd)),
        "gamma_prime_mean_aspect_ratio": float(np.nanmean(ar)),
        "gamma_prime_mean_circularity": float(np.nanmean(circ)),
    }

def overlay_pred_mask(img_rgb, mask01, fill_alpha=0.35, edge_thickness=2):
    # create colored overlay in green
    img = img_rgb.copy()
    H,W = mask01.shape
    m = (mask01 > 0)

    # fill
    color = np.array([0, 255, 0], dtype=np.uint8)
    img_f = img.astype(np.float32)
    img_f[m] = img_f[m]*(1-fill_alpha) + color*(fill_alpha)

    out = np.clip(img_f, 0, 255).astype(np.uint8)

    # edge
    edges = cv2.Canny((mask01*255).astype(np.uint8), 50, 150)
    ys, xs = np.where(edges > 0)
    for y,x in zip(ys,xs):
        y0 = max(0, y-edge_thickness); y1 = min(H, y+edge_thickness+1)
        x0 = max(0, x-edge_thickness); x1 = min(W, x+edge_thickness+1)
        out[y0:y1, x0:x1] = [255, 255, 0]  # yellow boundary
    return out

# predictor for deployment
cfg_deploy = cfg.clone()
cfg_deploy.MODEL.WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg_deploy.MODEL.ROI_HEADS.SCORE_THRESH_TEST = BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20
deploy_pred = DefaultPredictor(cfg_deploy)

# run on all images in IMG_DIR
image_paths = []
for fn in sorted(os.listdir(IMG_DIR)):
    if fn.lower().endswith((".png",".jpg",".jpeg",".tif",".tiff")):
        image_paths.append(os.path.join(IMG_DIR, fn))

print("Images found:", len(image_paths))
print("Deployment threshold:", cfg_deploy.MODEL.ROI_HEADS.SCORE_THRESH_TEST)

rows = []
for img_path in image_paths:
    fname = os.path.basename(img_path)

    img = cv2.imread(img_path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    out = deploy_pred(img_rgb)
    inst = out["instances"].to("cpu")

    if len(inst) == 0:
        union01 = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
    else:
        pm = inst.pred_masks.numpy().astype(np.uint8)
        union01 = (np.any(pm, axis=0)).astype(np.uint8)

    # Light post-processing to suppress tiny noise
    union01 = remove_small_objects(union01.astype(bool), min_size=16).astype(np.uint8)

    m = compute_shape_metrics_from_union(union01)

    mask_path = os.path.join(PRED_DIR, fname.rsplit(".",1)[0] + "_pred_mask.png")
    save_mask = (union01*255).astype(np.uint8)
    cv2.imwrite(mask_path, save_mask)

    ov = overlay_pred_mask(img_rgb, union01)
    ov_path = os.path.join(OVR_DIR, fname.rsplit(".",1)[0] + "_overlay.png")
    cv2.imwrite(ov_path, cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

    rows.append({
        "file": fname,
        "pred_mask_path": mask_path,
        "overlay_path": ov_path,
        **m
    })

df = pd.DataFrame(rows)
csv_out = os.path.join(OUT_ROOT, "deployment_predictions_and_metrics.csv")
df.to_csv(csv_out, index=False)
print("Saved:", csv_out)
df.head()


# **Distributions of extracted microstructure metrics**

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(os.path.join(OUT_ROOT, "deployment_predictions_and_metrics.csv"))

plots = [
    ("gamma_prime_area_fraction", "Area fraction (γ′)"),
    ("gamma_prime_cc_count", "Connected components (γ′ union)"),
    ("gamma_prime_mean_eq_diameter_px", "Mean equiv. diameter (px)"),
    ("gamma_prime_mean_aspect_ratio", "Mean aspect ratio"),
    ("gamma_prime_mean_circularity", "Mean circularity"),
]

for col, title in plots:
    plt.figure(figsize=(6,4))
    plt.hist(df[col].values, bins=25)
    plt.xlabel(title)
    plt.ylabel("Count")
    plt.title(f"Distribution: {title}")
    plt.tight_layout()

    sp = os.path.join(FIG_DIR, f"fig_hist_{col}.png")
    plt.savefig(sp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", sp)

# **Saved overlay images**

In [ ]:
import matplotlib.pyplot as plt

overlay_files = [os.path.join(OVR_DIR, f) for f in sorted(os.listdir(OVR_DIR)) if f.lower().endswith(".png")]
print("Overlays:", len(overlay_files))

# choose a good spread
k = min(12, len(overlay_files))
picks = np.linspace(0, len(overlay_files)-1, k, dtype=int)
picks = [overlay_files[i] for i in picks]

cols = 4
rows = int(np.ceil(k/cols))
plt.figure(figsize=(4.5*cols, 4.5*rows))

for i, p in enumerate(picks):
    img = cv2.imread(p)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax = plt.subplot(rows, cols, i+1)
    ax.imshow(img)
    ax.set_title(os.path.basename(p).replace("_overlay.png",""))
    ax.axis("off")

plt.tight_layout()
save_path = os.path.join(FIG_DIR, "fig_montage_deployment_overlays.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

# **Workflow diagram**

In [ ]:
# !pip -q install graphviz
# from graphviz import Digraph

# dot = Digraph("workflow", format="png")
# dot.attr(rankdir="LR", fontsize="12")

# dot.node("A", "Input micrographs\n(SEM/optical)\n(640×640)")
# dot.node("B", "COCO annotations\n(CVAT/Roboflow)")
# dot.node("C", "Single-class COCO\n(γ′ only)")
# dot.node("D", "Mask R-CNN training\n(Detectron2)")
# dot.node("E", "Evaluation\nCOCO AP50/AP75\nDice/IoU + Threshold sweep")
# dot.node("F", "Deployment inference\nNew micrographs → masks")
# dot.node("G", "Microstructure metrics\nArea fraction, count,\nEq. diameter, AR, circularity")
# dot.node("H", "Outputs\nOverlays + masks + CSV")

# dot.edges([("A","B"), ("B","C"), ("C","D"), ("D","E"), ("D","F"), ("F","G"), ("G","H")])

# workflow_path = os.path.join(FIG_DIR, "fig_workflow")
# dot.render(workflow_path, cleanup=True)
# print("Saved:", workflow_path + ".png")

# # show
# import matplotlib.pyplot as plt
# img = cv2.imread(workflow_path + ".png")
# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# plt.figure(figsize=(14,4))
# plt.imshow(img)
# plt.axis("off")
# plt.show()

# **Setup helpers (Ground Truth union mask + error overlay)**

In [ ]:
import os, json
import numpy as np
import cv2
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# This mUST match our earlier paths (IMG_DIR, SINGLE_VAL_JSON, FIG_DIR, cfg_pred, predictor)
print("IMG_DIR:", IMG_DIR)
print("SINGLE_VAL_JSON:", SINGLE_VAL_JSON)
print("FIG_DIR:", FIG_DIR)

coco_val = COCO(SINGLE_VAL_JSON)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygon list
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def gt_union_mask_from_coco(img_id):
    img = coco_val.loadImgs(img_id)[0]
    H, W = img["height"], img["width"]
    fname = img["file_name"]
    ann_ids = coco_val.getAnnIds(imgIds=[img_id])
    anns = coco_val.loadAnns(ann_ids)

    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))

    return union, fname, len(anns)

def pred_union_mask(img_rgb, predictor):
    out = predictor(img_rgb)
    inst = out["instances"].to("cpu")
    if len(inst) == 0:
        return np.zeros(img_rgb.shape[:2], dtype=np.uint8), 0
    pm = inst.pred_masks.numpy().astype(np.uint8)  # [N,H,W]
    union = (np.any(pm, axis=0)).astype(np.uint8)
    return union, int(len(inst))

def edges_from_mask(mask01, thickness=2):
    m = (mask01 > 0).astype(np.uint8) * 255
    e = cv2.Canny(m, 50, 150)
    if thickness > 1:
        k = np.ones((thickness, thickness), np.uint8)
        e = cv2.dilate(e, k, iterations=1)
    return (e > 0)

def draw_outline(img_rgb, mask01, color=(0,255,0), thickness=2):
    img = img_rgb.copy()
    e = edges_from_mask(mask01, thickness=thickness)
    img[e] = np.array(color, dtype=np.uint8)
    return img

def error_map_overlay(img_rgb, gt01, pr01, alpha=0.45):
    """
    TP: green, FN: red, FP: blue
    """
    gt = (gt01 > 0)
    pr = (pr01 > 0)

    tp = gt & pr
    fn = gt & (~pr)
    fp = (~gt) & pr

    overlay = img_rgb.copy().astype(np.float32)

    # colors
    green = np.array([0,255,0], np.float32)
    red   = np.array([255,0,0], np.float32)
    blue  = np.array([0,0,255], np.float32)

    for mask, col in [(tp, green), (fn, red), (fp, blue)]:
        overlay[mask] = overlay[mask]*(1-alpha) + col*alpha

    return np.clip(overlay, 0, 255).astype(np.uint8)

import json
import zipfile
import shutil


# **Ground Truth / Pred / Error grid**

In [ ]:
import random

def make_gt_pred_error_grid(n_images=6, seed=42, score_thresh=BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20, save_name="fig_gt_pred_error_grid.png"):
    random.seed(seed)

    # ensure predictor threshold
    cfg_pred.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(score_thresh)
    from detectron2.engine import DefaultPredictor
    predictor_local = DefaultPredictor(cfg_pred)

    img_ids = coco_val.getImgIds()
    picks = random.sample(img_ids, min(n_images, len(img_ids)))

    rows = len(picks)
    cols = 3

    plt.figure(figsize=(4.2*cols, 4.2*rows))

    for r, img_id in enumerate(picks):
        gt01, fname, gt_count = gt_union_mask_from_coco(img_id)
        img_path = os.path.join(IMG_DIR, fname)

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print("[WARN] cannot read:", img_path)
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        pr01, pred_count = pred_union_mask(img_rgb, predictor_local)

        # panels
        gt_panel   = draw_outline(img_rgb, gt01, color=(0,255,0), thickness=2)     # green
        pred_panel = draw_outline(img_rgb, pr01, color=(255,255,0), thickness=2)   # yellow
        err_panel  = error_map_overlay(img_rgb, gt01, pr01, alpha=0.45)

        # plot GT
        ax = plt.subplot(rows, cols, r*cols + 1)
        ax.imshow(gt_panel)
        ax.set_title(f"{fname} | GT (inst={gt_count})", fontsize=10)
        ax.axis("off")

        # plot Pred
        ax = plt.subplot(rows, cols, r*cols + 2)
        ax.imshow(pred_panel)
        ax.set_title(f"{fname} | Pred (inst={pred_count})", fontsize=10)
        ax.axis("off")

        # plot Error
        ax = plt.subplot(rows, cols, r*cols + 3)
        ax.imshow(err_panel)
        ax.set_title(f"{fname} | Error (TP=G, FN=R, FP=B)", fontsize=10)
        ax.axis("off")

    plt.tight_layout()

    save_path = os.path.join(FIG_DIR, save_name)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

make_gt_pred_error_grid(
    n_images=6,          # we can change this to 9 or 12
    seed=7,
    score_thresh=BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20,
    save_name="fig_gt_pred_error_grid.png"
)

# **To check if we are using the right COCO + image folder**

In [ ]:

import os
from pycocotools.coco import COCO

COCO_JSON = COCO_ALL_JSON
IMG_DIR   = IMG_DIR

print("COCO_JSON:", COCO_JSON)
print("IMG_DIR  :", IMG_DIR, "| exists:", os.path.isdir(IMG_DIR))

coco = COCO(COCO_JSON)

print("COCO images:", len(coco.dataset["images"]))
print("COCO anns  :", len(coco.dataset["annotations"]))
print("COCO cats  :", coco.dataset["categories"])

# show 10 example filenames + whether they exist in IMG_DIR
for img in coco.dataset["images"][:10]:
    p = os.path.join(IMG_DIR, img["file_name"])
    print(img["file_name"], " | exists:", os.path.exists(p))


# **Select 5 images that have Ground Truth and exist on drive**

In [ ]:
import random

def gt_count_for_image(img_id):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    return len(ann_ids)

candidates = []
for img in coco.dataset["images"]:
    img_id = img["id"]
    fname = img["file_name"]
    img_path = os.path.join(IMG_DIR, fname)
    if os.path.exists(img_path):
        n = gt_count_for_image(img_id)
        if n > 0:
            candidates.append((img_id, fname, n))

print("Candidates (exists on disk + GT>0):", len(candidates))
print("First 10 candidates:", candidates[:10])

selected = random.sample(candidates, 5) if len(candidates) >= 5 else candidates
print("\nSelected:")
for img_id, fname, n in selected:
    print(fname, "| GT anns:", n)

# **COCO mask decode (works for polygon + RLE)**

In [ ]:
import numpy as np
from pycocotools import mask as maskUtils

def ann_to_mask(ann, height, width):
    seg = ann.get("segmentation", None)
    if seg is None:
        return np.zeros((height, width), dtype=np.uint8)

    # Polygon
    if isinstance(seg, list):
        rles = maskUtils.frPyObjects(seg, height, width)
        rle = maskUtils.merge(rles)
        m = maskUtils.decode(rle)

    # RLE dict
    elif isinstance(seg, dict) and "counts" in seg:
        rle = seg
        m = maskUtils.decode(rle)

    else:
        return np.zeros((height, width), dtype=np.uint8)

    if m.ndim == 3:
        m = m[..., 0]
    return (m > 0).astype(np.uint8)

# **(Original | GT union | GT overlay)**

In [ ]:
import cv2, matplotlib.pyplot as plt

def overlay_fill_and_edge(img_rgb, mask01, fill_alpha=0.35):
    """
    img_rgb: HxWx3 uint8
    mask01:  HxW uint8 (0/1)
    """
    img = img_rgb.copy().astype(np.float32)

    # Fill (green)
    fill = np.array([0, 255, 0], dtype=np.float32)
    m = (mask01 > 0)
    img[m] = img[m] * (1 - fill_alpha) + fill * fill_alpha

    # Edge (white) from contours
    edges = np.zeros(mask01.shape, dtype=np.uint8)
    cnts, _ = cv2.findContours(mask01.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(edges, cnts, -1, 255, 1)
    img[edges > 0] = np.array([255, 255, 255], dtype=np.float32)

    return np.clip(img, 0, 255).astype(np.uint8)

def make_triptych(img_id, fname):
    img_info = coco.loadImgs([img_id])[0]
    H, W = img_info["height"], img_info["width"]
    img_path = os.path.join(IMG_DIR, fname)

    bgr = cv2.imread(img_path)
    if bgr is None:
        print("[WARN] Cannot read:", img_path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    union = np.zeros((H, W), dtype=np.uint8)
    nonempty = 0
    for ann in anns:
        m = ann_to_mask(ann, H, W)
        if m.sum() > 0:
            nonempty += 1
        union = np.maximum(union, m)

    overlay = overlay_fill_and_edge(rgb, union, fill_alpha=0.35)

    plt.figure(figsize=(14,4))
    plt.subplot(1,3,1); plt.imshow(rgb); plt.title(f"Original\n{fname}")
    plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(union, cmap="gray"); plt.title(f"GT union mask\nanns={len(anns)} | decoded_nonempty={nonempty}")
    plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(overlay); plt.title("GT overlay (fill+edge)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Show 5 images

In [ ]:
for img_id, fname, n in selected:
    make_triptych(img_id, fname)

In [ ]:

SAVE_DIR = os.path.join(OUT_ROOT, "gt_visualisations")
os.makedirs(SAVE_DIR, exist_ok=True)

def save_triptych(img_id, fname):
    img_info = coco.loadImgs([img_id])[0]
    H, W = img_info["height"], img_info["width"]
    img_path = os.path.join(IMG_DIR, fname)

    bgr = cv2.imread(img_path)
    if bgr is None:
        print("[WARN] Cannot read:", img_path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        m = ann_to_mask(ann, H, W)
        union = np.maximum(union, m)

    overlay = overlay_fill_and_edge(rgb, union, fill_alpha=0.35)
    base = os.path.splitext(fname)[0]

    cv2.imwrite(os.path.join(SAVE_DIR, f"{base}_original.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(SAVE_DIR, f"{base}_mask.png"), union * 255)
    cv2.imwrite(os.path.join(SAVE_DIR, f"{base}_overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    fig = plt.figure(figsize=(14, 4))
    plt.subplot(1, 3, 1); plt.imshow(rgb); plt.axis("off")
    plt.subplot(1, 3, 2); plt.imshow(union, cmap="gray"); plt.axis("off")
    plt.subplot(1, 3, 3); plt.imshow(overlay); plt.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, f"{base}_triptych.png"))
    plt.close(fig)

    print(f"[SAVED] {base} → {SAVE_DIR}")

for img_id, fname, n in selected:
    save_triptych(img_id, fname)


# **COCO evaluation bar chart (AP / AP50 / AP75)**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert isinstance(results, dict), "Expected `results` to be a dict from Detectron2 evaluation."
assert "bbox" in results or "segm" in results, "Expected results to contain 'bbox' and/or 'segm'."

OUT_DIR = FIG_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# Build tidy dataframe

rows = []
for task in ["bbox", "segm"]:
    if task not in results:
        continue
    for metric in ["AP", "AP50", "AP75"]:
        if metric in results[task]:
            rows.append({"task": task, "metric": metric, "value": float(results[task][metric])})

df = pd.DataFrame(rows)

if df.empty:
    raise ValueError("No metrics found in results. Check results keys and metric names.")

# Keep nice ordering
metric_order = ["AP", "AP50", "AP75"]
task_order = [t for t in ["bbox", "segm"] if t in df["task"].unique()]

# Pivot for plotting
pivot = (
    df.pivot(index="metric", columns="task", values="value")
      .reindex(metric_order)
      .loc[metric_order]
)

# Plot + display
x = np.arange(len(pivot.index))
width = 0.35

plt.figure(figsize=(8, 4.8))

for j, task in enumerate(task_order):
    vals = pivot[task].values
    plt.bar(x + (j - (len(task_order)-1)/2)*width, vals, width=width, label=task)

plt.xticks(x, pivot.index)
plt.ylabel("COCO metric value")
plt.title("COCO evaluation summary (1-class γ′)")
plt.legend()
plt.tight_layout()

# Save plot
plot_path = os.path.join(OUT_DIR, "coco_eval_summary_ap_ap50_ap75.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

display(df)

csv_path = os.path.join(OUT_DIR, "coco_eval_summary_table.csv")
df.to_csv(csv_path, index=False)
print("Saved table to:", csv_path)

# **DEPLOYMENT**

# Install/Imports (run once)

In [ ]:
# To be skipped if detectron2 is already installed.
# !pip install --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git'
!pip -q install gradio opencv-python-headless pycocotools

import os, cv2, json, math
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

import torch
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.model_zoo import get_config_file
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model

# **Set paths + build predictor**

In [ ]:
# Detectron2 config name (Mask R-CNN)
CFG_NAME = "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"

# Trained weights (.pth) – prefer the best validation checkpoint
WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")

# Image folder to run deployment on
DEPLOY_IMG_DIR = IMG_DIR

# Where to save outputs
OUT_DIR = OUT_ROOT
PRED_MASK_DIR = os.path.join(OUT_DIR, "pred_masks")
OVERLAY_DIR   = os.path.join(OUT_DIR, "overlays")
CSV_PATH      = os.path.join(OUT_DIR, "deployment_predictions_and_metrics.csv")
BATCH_EXPORT_ROOT = os.path.join(OUT_DIR, "batch_exports")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(BATCH_EXPORT_ROOT, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SCORE_THRESH = BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20
MIN_COMPONENT_SIZE = 16

from detectron2.config import get_cfg
from detectron2.model_zoo import get_config_file
from detectron2.engine import DefaultPredictor

cfg = get_cfg()
cfg.merge_from_file(get_config_file(CFG_NAME))

cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[4, 8, 16, 32, 64, 128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]

cfg.MODEL.RPN.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.RPN.NMS_THRESH = 0.7
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST  = 6000
cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 4000

cfg.TEST.DETECTIONS_PER_IMAGE = 3000
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 1280

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.35
cfg.MODEL.WEIGHTS = WEIGHTS
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH
cfg.MODEL.DEVICE = DEVICE

predictor = DefaultPredictor(cfg)

print("Device:", DEVICE)
print("Weights exists:", os.path.exists(WEIGHTS))
print("Saving to:", OUT_DIR)
print("Batch CVAT exports root:", BATCH_EXPORT_ROOT)
print("Deployment SCORE_THRESH:", SCORE_THRESH)
print("MIN_COMPONENT_SIZE:", MIN_COMPONENT_SIZE)


# **Helper functions (overlay + measurements)**

In [ ]:
def postprocess_instance_masks(masks_bool, min_area=MIN_COMPONENT_SIZE if "MIN_COMPONENT_SIZE" in globals() else 16):
    cleaned = []
    for m in masks_bool:
        m2 = remove_small_objects(m.astype(bool), min_size=min_area)
        if np.any(m2):
            cleaned.append(m2.astype(bool))
    return cleaned


def read_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def save_mask_png(mask01, save_path):
    # mask01: uint8 0/1 or bool
    m = (mask01.astype(np.uint8) * 255)
    cv2.imwrite(save_path, m)

def overlay_fill_and_boundary(img_rgb, mask01, fill_color=(0,255,0), edge_color=(255,255,255),
                              fill_alpha=0.35, edge_thickness=2):
    """
    img_rgb: HxWx3 uint8
    mask01: HxW (bool or 0/1)
    """
    img = img_rgb.copy()
    m = mask01.astype(bool)

    # Fill
    if m.any():
        color = np.array(fill_color, dtype=np.float32)
        img_f = img.astype(np.float32)
        img_f[m] = img_f[m] * (1 - fill_alpha) + color * fill_alpha
        img = np.clip(img_f, 0, 255).astype(np.uint8)

    # Boundary via contours
    m8 = (mask01.astype(np.uint8) * 255)
    cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) > 0:
        cv2.drawContours(img, cnts, -1, edge_color, thickness=edge_thickness)

    return img

def compute_instance_metrics_from_masks(masks_bool):
    """
    masks_bool: list of HxW bool arrays (one per instance)
    returns dict of aggregate stats
    """
    if len(masks_bool) == 0:
        return dict(
            pred_instance_count=0,
            area_fraction=0.0,
            cc_count=0,
            mean_area_px=np.nan,
            mean_eq_diameter_px=np.nan,
            mean_aspect_ratio=np.nan,
            mean_circularity=np.nan,
            mean_perimeter_px=np.nan,
            mean_nn_spacing_px=np.nan,
        )

    H, W = masks_bool[0].shape
    union = np.zeros((H, W), dtype=np.uint8)
    areas = []
    perims = []
    eqds = []
    ars = []
    circs = []
    centroids = []

    for m in masks_bool:
        m8 = (m.astype(np.uint8) * 255)
        union = np.maximum(union, m8)

        area = float(m.sum())
        areas.append(area)

        cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(cnts) == 0:
            perims.append(0.0)
            eqds.append(0.0)
            ars.append(np.nan)
            circs.append(np.nan)
            centroids.append((np.nan, np.nan))
            continue

        cnt = max(cnts, key=cv2.contourArea)
        perim = float(cv2.arcLength(cnt, True))
        perims.append(perim)

        # Equivalent diameter: sqrt(4A/pi)
        eqd = math.sqrt(4.0 * area / math.pi) if area > 0 else 0.0
        eqds.append(eqd)

        # Aspect ratio: from min-area rectangle
        rect = cv2.minAreaRect(cnt)  # ((cx,cy),(w,h),theta)
        (cx, cy), (rw, rh), _ = rect
        if rw > 0 and rh > 0:
            ar = max(rw, rh) / min(rw, rh)
        else:
            ar = np.nan
        ars.append(ar)

        # Circularity: 4πA / P^2
        circ = (4.0 * math.pi * area / (perim**2)) if perim > 1e-6 else np.nan
        circs.append(circ)

        centroids.append((float(cx), float(cy)))

    # Connected components on UNION mask
    union01 = (union > 0).astype(np.uint8)
    num_cc, _ = cv2.connectedComponents(union01)
    cc_count = int(max(0, num_cc - 1))

    area_fraction = float(union01.sum()) / float(H * W)

    # Nearest-neighbour spacing (centroid-to-centroid)
    pts = np.array([[x, y] for (x, y) in centroids if np.isfinite(x) and np.isfinite(y)], dtype=np.float32)
    if len(pts) >= 2:
        dists = []
        for i in range(len(pts)):
            diff = pts - pts[i]
            dd = np.sqrt((diff[:,0]**2 + diff[:,1]**2))
            dd[i] = np.inf
            dists.append(np.min(dd))
        mean_nn = float(np.mean(dists))
    else:
        mean_nn = np.nan

    return dict(
        pred_instance_count=int(len(masks_bool)),
        area_fraction=float(area_fraction),
        cc_count=cc_count,
        mean_area_px=float(np.nanmean(areas)),
        mean_eq_diameter_px=float(np.nanmean(eqds)),
        mean_aspect_ratio=float(np.nanmean(ars)),
        mean_circularity=float(np.nanmean(circs)),
        mean_perimeter_px=float(np.nanmean(perims)),
        mean_nn_spacing_px=float(mean_nn),
    )

def mask_to_coco_segmentation(mask_bool, approx_epsilon=1.5):
    """Convert a binary instance mask to COCO polygon segmentation."""
    m8 = (mask_bool.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    segmentation = []
    for cnt in contours:
        if approx_epsilon and approx_epsilon > 0:
            cnt = cv2.approxPolyDP(cnt, approx_epsilon, True)
        if cnt.shape[0] < 3:
            continue
        poly = cnt.reshape(-1, 2).astype(float).flatten().tolist()
        if len(poly) >= 6:
            segmentation.append(poly)
    return segmentation

def build_coco_annotation(mask_bool, image_id, ann_id, category_id=1):
    segmentation = mask_to_coco_segmentation(mask_bool)
    if len(segmentation) == 0:
        return None
    ys, xs = np.where(mask_bool)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    bbox = [float(x0), float(y0), float(x1 - x0 + 1), float(y1 - y0 + 1)]
    area = float(mask_bool.sum())
    return {
        "id": int(ann_id),
        "image_id": int(image_id),
        "category_id": int(category_id),
        "segmentation": segmentation,
        "area": area,
        "bbox": bbox,
        "iscrowd": 0,
    }

def make_run_root(base_root=BATCH_EXPORT_ROOT, prefix="batch_cvat_export"):
    ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    run_root = os.path.join(base_root, f"{prefix}_{ts}")
    os.makedirs(run_root, exist_ok=True)
    return run_root

def prepare_export_dirs(run_root):
    dirs = {
        "run_root": run_root,
        "images_dir": os.path.join(run_root, "images"),
        "masks_dir": os.path.join(run_root, "pred_masks"),
        "overlays_dir": os.path.join(run_root, "overlays"),
        "annotations_dir": os.path.join(run_root, "annotations"),
    }
    for p in dirs.values():
        os.makedirs(p, exist_ok=True)
    return dirs

def zip_dir(src_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(src_dir):
            for fname in files:
                fpath = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, src_dir)
                zf.write(fpath, arcname)
    return zip_path

def infer_one_image(img_path, save_outputs=True, export_dirs=None, image_id=None, ann_start_id=1, copy_image=False):
    """
    Runs Detectron2 inference, saves mask+overlay, returns metrics dict + paths.
    If export_dirs is provided, also prepares per-instance COCO annotations for CVAT import.
    """
    img_rgb = read_rgb(img_path)
    H, W = img_rgb.shape[:2]

    outputs = predictor(img_rgb)
    inst = outputs["instances"].to("cpu")

    masks = []
    if inst.has("pred_masks"):
        pm = inst.pred_masks.numpy()  # [N,H,W] bool
        for i in range(pm.shape[0]):
            masks.append(pm[i].astype(bool))

    masks = postprocess_instance_masks(masks)

    # Union mask
    if len(masks) > 0:
        union = np.zeros((H, W), dtype=np.uint8)
        for m in masks:
            union = np.maximum(union, m.astype(np.uint8))
        union01 = union.astype(np.uint8)
    else:
        union01 = np.zeros((H, W), dtype=np.uint8)

    fname = os.path.basename(img_path)
    base = os.path.splitext(fname)[0]
    if export_dirs is None:
        mask_path = os.path.join(PRED_MASK_DIR, f"{base}_pred_mask.png")
        overlay_path = os.path.join(OVERLAY_DIR, f"{base}_overlay.png")
        image_copy_path = img_path
    else:
        mask_path = os.path.join(export_dirs["masks_dir"], f"{base}_pred_mask.png")
        overlay_path = os.path.join(export_dirs["overlays_dir"], f"{base}_overlay.png")
        image_copy_path = os.path.join(export_dirs["images_dir"], fname)

    if save_outputs:
        save_mask_png(union01, mask_path)
        ov = overlay_fill_and_boundary(img_rgb, union01, fill_color=(0,255,0), edge_color=(255,255,255))
        cv2.imwrite(overlay_path, cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))
        if export_dirs is not None and copy_image:
            shutil.copy2(img_path, image_copy_path)
    else:
        mask_path = ""
        overlay_path = ""
        image_copy_path = img_path

    m = compute_instance_metrics_from_masks(masks)
    row = dict(
        file=fname,
        image_path=image_copy_path,
        source_image_path=img_path,
        pred_mask_path=mask_path,
        overlay_path=overlay_path,
        **m
    )

    image_record = None
    annotations = []
    next_ann_id = ann_start_id
    if export_dirs is not None and image_id is not None:
        image_record = {
            "id": int(image_id),
            "file_name": fname,
            "width": int(W),
            "height": int(H),
        }
        for mask_bool in masks:
            ann = build_coco_annotation(mask_bool, image_id=image_id, ann_id=next_ann_id, category_id=1)
            if ann is not None:
                annotations.append(ann)
                next_ann_id += 1

    return row, image_record, annotations, next_ann_id


# **Run deployment on a folder (and export CSV)**

In [ ]:

def deploy_folder(img_dir, csv_path=CSV_PATH, export_cvat=False, run_root=None):
    image_paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"):
        image_paths += glob(os.path.join(img_dir, ext))
    image_paths = sorted(image_paths)

    if len(image_paths) == 0:
        raise ValueError(f"No images found in: {img_dir}")

    export_dirs = None
    zip_path = None
    annotations_json_path = None
    if export_cvat:
        if run_root is None:
            run_root = make_run_root()
        export_dirs = prepare_export_dirs(run_root)
        csv_path = os.path.join(run_root, "deployment_predictions_and_metrics.csv")

    rows = []
    coco_images = []
    coco_annotations = []
    ann_id = 1

    for image_id, p in enumerate(tqdm(image_paths, desc="Running deployment"), start=1):
        try:
            row, image_record, anns, ann_id = infer_one_image(
                p,
                save_outputs=True,
                export_dirs=export_dirs,
                image_id=image_id if export_cvat else None,
                ann_start_id=ann_id,
                copy_image=export_cvat,
            )
            rows.append(row)
            if export_cvat:
                coco_images.append(image_record)
                coco_annotations.extend(anns)
        except Exception as e:
            rows.append(dict(
                file=os.path.basename(p),
                image_path=p,
                source_image_path=p,
                pred_mask_path="",
                overlay_path="",
                pred_instance_count=0,
                area_fraction=np.nan,
                cc_count=np.nan,
                mean_area_px=np.nan,
                mean_eq_diameter_px=np.nan,
                mean_aspect_ratio=np.nan,
                mean_circularity=np.nan,
                mean_perimeter_px=np.nan,
                mean_nn_spacing_px=np.nan,
                error=str(e)
            ))

    df = pd.DataFrame(rows)

    if export_cvat:
        df.to_csv(csv_path, index=False)
        coco = {
            "info": {
                "description": "Predicted gamma_prime masks for CVAT correction",
                "version": "1.0",
            },
            "licenses": [],
            "images": coco_images,
            "annotations": coco_annotations,
            "categories": [{"id": 1, "name": "gamma_prime", "supercategory": "microstructure"}],
        }
        annotations_json_path = os.path.join(export_dirs["annotations_dir"], "instances_default.json")
        with open(annotations_json_path, "w") as f:
            json.dump(coco, f)

        readme_path = os.path.join(run_root, "README_for_CVAT.txt")
        with open(readme_path, "w") as f:
            f.write(
                "1. In CVAT, create a segmentation task and upload the same images from the images folder.\n"
                "2. Then use Actions -> Upload annotations.\n"
                "3. Select format COCO 1.0 and upload annotations/instances_default.json.\n"
                "4. The pred_masks folder contains union masks for quick inspection; CVAT uses the COCO JSON.\n"
            )

        zip_path = os.path.join(run_root, "cvat_predictions_bundle.zip")
        zip_dir(run_root, zip_path)
        print("Saved CVAT bundle:", zip_path)
    else:
        if os.path.exists(csv_path):
            old = pd.read_csv(csv_path)
            df_all = pd.concat([old, df], ignore_index=True)
            df_all.to_csv(csv_path, index=False)
        else:
            df.to_csv(csv_path, index=False)

    print("Saved:", csv_path)
    return df, csv_path, zip_path, run_root, annotations_json_path

df_results, latest_csv_path, latest_zip_path, latest_run_root, latest_annotations_json = deploy_folder(
    DEPLOY_IMG_DIR,
    csv_path=CSV_PATH,
    export_cvat=False,
)
df_results.head()


# **Quick visualization to show 5 examples (Original | Mask | Overlay)**

In [ ]:
import matplotlib.pyplot as plt

def show_examples(df, n=5, seed=42):
    df2 = df.sample(n=min(n, len(df)), random_state=seed)
    for _, r in df2.iterrows():
        img = read_rgb(r["image_path"])
        m = cv2.imread(r["pred_mask_path"], cv2.IMREAD_GRAYSCALE) if os.path.exists(r["pred_mask_path"]) else None
        ov = cv2.imread(r["overlay_path"], cv2.IMREAD_COLOR) if os.path.exists(r["overlay_path"]) else None
        if ov is not None:
            ov = cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(14,4))
        plt.subplot(1,3,1); plt.title(f"Original: {r['file']}"); plt.imshow(img); plt.axis("off")
        plt.subplot(1,3,2); plt.title("Pred union mask"); plt.imshow(m, cmap="gray"); plt.axis("off")
        plt.subplot(1,3,3); plt.title("Overlay (fill + boundary)"); plt.imshow(ov); plt.axis("off")
        plt.tight_layout()
        plt.show()

show_examples(df_results, n=5)

# **Gradio Web App (single image + batch folder)**

In [ ]:
import gradio as gr

def gr_infer_single(img):
    """
    img is a numpy array RGB from gradio
    """
    if img is None:
        return None, None, pd.DataFrame()

    img_rgb = img.astype(np.uint8)
    H, W = img_rgb.shape[:2]

    outputs = predictor(img_rgb)
    inst = outputs["instances"].to("cpu")

    masks = []
    if inst.has("pred_masks"):
        pm = inst.pred_masks.numpy()
        for i in range(pm.shape[0]):
            masks.append(pm[i].astype(bool))

    if len(masks) > 0:
        union01 = np.zeros((H, W), dtype=np.uint8)
        for m in masks:
            union01 = np.maximum(union01, m.astype(np.uint8))
    else:
        union01 = np.zeros((H, W), dtype=np.uint8)

    overlay = overlay_fill_and_boundary(img_rgb, union01, fill_color=(0,255,0), edge_color=(255,255,255))
    m = compute_instance_metrics_from_masks(masks)
    df = pd.DataFrame([m])
    mask_vis = (union01 * 255).astype(np.uint8)
    return overlay, mask_vis, df

def gr_deploy_folder(img_dir):
    df, csv_path, zip_path, run_root, annotations_json_path = deploy_folder(
        img_dir,
        csv_path=CSV_PATH,
        export_cvat=True,
        run_root=None,
    )
    status = (
        f"Done. Export folder: {run_root}\n"
        f"CSV: {csv_path}\n"
        f"COCO JSON: {annotations_json_path}\n"
        f"CVAT ZIP bundle: {zip_path}"
    )
    return df, csv_path, zip_path, status

with gr.Blocks() as demo:
    gr.Markdown(
        """## γ′ Deployment + Measurement Pipeline (Mask R-CNN)

Upload an image or run a whole folder batch.

Batch export saves:
- overlay
- union masks
- CSV
- images
- COCO JSON
- CVAT-ready ZIP bundle
"""
    )

    with gr.Tab("Single Image"):
        inp = gr.Image(type="numpy", label="Upload micrograph")
        btn = gr.Button("Run Inference + Measurements")
        out_overlay = gr.Image(type="numpy", label="Overlay (fill + boundary)")
        out_mask = gr.Image(type="numpy", label="Predicted union mask (PNG-style)")
        out_table = gr.Dataframe(label="Measurements (per image)")

        btn.click(fn=gr_infer_single, inputs=[inp], outputs=[out_overlay, out_mask, out_table])

    with gr.Tab("Batch Folder"):
        folder_in = gr.Textbox(value=IMG_DIR, label="Folder path containing images")
        btn2 = gr.Button("Run Batch Deployment + Save CSV + Export CVAT Bundle")
        out_df = gr.Dataframe(label="Batch Results (preview)")
        out_csv = gr.File(label="Download CSV")
        out_zip = gr.File(label="Download CVAT ZIP bundle")
        out_status = gr.Textbox(label="Export status", lines=5)

        btn2.click(fn=gr_deploy_folder, inputs=[folder_in], outputs=[out_df, out_csv, out_zip, out_status])

demo.launch(share=True, debug=True)

# **FIGURES FOR PAPER 1**

In [ ]:
# ==========================================
# FIGURE 2 PANELS FROM REAL COCO MASK CODE
# (a) SEM image
# (b) annotation
# (c) ground truth
# (d) mask
# (e) overlay
# ==========================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# -----------------------------
# REQUIRED:
# - coco = COCO(YOUR_JSON)
# - ann_to_mask(ann, H, W) already defined
# -----------------------------

# ===== YOUR EXACT IMAGE PATH =====
TARGET_IMAGE_PATH = r"/content/image_55.png"

# COCO JSON path
# Replace this with your real annotation JSON path
COCO_JSON_PATH = r"/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/1.json"

# Output folder
OUT_DIR = r"/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure2_real_panels"

os.makedirs(OUT_DIR, exist_ok=True)

# Load COCO
coco = COCO(COCO_JSON_PATH)

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def get_img_info_from_filename(coco, target_image_path):
    target_file_name = os.path.basename(target_image_path)  # image_55.jpg
    all_imgs = coco.loadImgs(coco.getImgIds())

    matches = [x for x in all_imgs if os.path.basename(x["file_name"]) == target_file_name]
    if not matches:
        raise ValueError(
            f"File name '{target_file_name}' not found in COCO JSON.\n"
            f"Check whether the JSON stores the same filename."
        )
    return matches[0]

def build_union_mask(coco, img_id):
    info = coco.loadImgs([img_id])[0]
    H, W = info["height"], info["width"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        m = ann_to_mask(ann, H, W)
        union = np.maximum(union, m.astype(np.uint8))
    return union, anns

def clean_mask(mask01, min_area=20):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask01.astype(np.uint8), connectivity=8
    )
    cleaned = np.zeros_like(mask01, dtype=np.uint8)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            cleaned[labels == i] = 1
    return cleaned

def make_overlay(rgb, mask01, fill_color=(255, 140, 0), alpha=0.45,
                 edge_color=(255, 255, 0), edge_thickness=2):
    out = rgb.copy()

    # fill mask
    fill = np.zeros_like(out, dtype=np.uint8)
    fill[:, :] = fill_color
    mask_bool = mask01.astype(bool)

    out[mask_bool] = (
        (1 - alpha) * out[mask_bool] + alpha * fill[mask_bool]
    ).astype(np.uint8)

    # draw contours
    contours, _ = cv2.findContours(mask01.astype(np.uint8),
                                   cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    out_bgr = cv2.cvtColor(out, cv2.COLOR_RGB2BGR)
    cv2.drawContours(out_bgr, contours, -1, edge_color[::-1], edge_thickness)
    out = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
    return out

def draw_polygon_annotation(ax, rgb, anns, linewidth=1.5, edgecolor='yellow'):
    ax.imshow(rgb, cmap='gray')
    patches = []

    for ann in anns:
        seg = ann.get("segmentation", None)

        if isinstance(seg, list):  # polygon annotations
            for poly in seg:
                pts = np.array(poly).reshape(-1, 2)
                if len(pts) >= 3:
                    patches.append(Polygon(pts, closed=True))

    if patches:
        p = PatchCollection(
            patches,
            facecolor='none',
            edgecolor=edgecolor,
            linewidth=linewidth
        )
        ax.add_collection(p)

    ax.axis("off")

# -----------------------------
# ann_to_mask must already exist
# Example fallback below if needed
# -----------------------------
def ann_to_mask(ann, H, W):
    from pycocotools import mask as maskUtils

    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg

    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

# -----------------------------
# Load target image + masks
# -----------------------------
info = get_img_info_from_filename(coco, TARGET_IMAGE_PATH)
img_id = info["id"]
file_name = info["file_name"]

rgb = read_rgb(TARGET_IMAGE_PATH)
gt_mask, anns = build_union_mask(coco, img_id)
cleaned_mask = clean_mask(gt_mask, min_area=20)
overlay = make_overlay(rgb, cleaned_mask,
                       fill_color=(255, 140, 0),
                       alpha=0.45,
                       edge_color=(255, 255, 0),
                       edge_thickness=2)

# -----------------------------
# Save individual panels
# -----------------------------
sem_path = os.path.join(OUT_DIR, "panel_a_sem.png")
ann_path = os.path.join(OUT_DIR, "panel_b_annotation.png")
gt_path  = os.path.join(OUT_DIR, "panel_c_ground_truth.png")
msk_path = os.path.join(OUT_DIR, "panel_d_mask.png")
ovl_path = os.path.join(OUT_DIR, "panel_e_overlay.png")
fig2_path = os.path.join(OUT_DIR, "figure2_5panel.png")

plt.imsave(sem_path, rgb)

fig = plt.figure(figsize=(6, 6))
ax = plt.gca()
draw_polygon_annotation(ax, rgb, anns, linewidth=1.4, edgecolor='yellow')
plt.tight_layout()
plt.savefig(ann_path, dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.close(fig)

plt.imsave(gt_path, gt_mask, cmap="gray")
plt.imsave(msk_path, cleaned_mask, cmap="gray")
plt.imsave(ovl_path, overlay)

# -----------------------------
# Create final 5-panel figure
# -----------------------------
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

# (a) SEM
axes[0].imshow(rgb, cmap='gray')
axes[0].set_title("(a) SEM image", fontsize=14)
axes[0].axis("off")

# (b) Annotation
draw_polygon_annotation(axes[1], rgb, anns, linewidth=1.4, edgecolor='yellow')
axes[1].set_title("(b) Annotation", fontsize=14)

# (c) Ground truth
axes[2].imshow(gt_mask, cmap='gray')
axes[2].set_title("(c) Ground truth", fontsize=14)
axes[2].axis("off")

# (d) Mask
axes[3].imshow(cleaned_mask, cmap='gray')
axes[3].set_title("(d) Mask", fontsize=14)
axes[3].axis("off")

# (e) Overlay
axes[4].imshow(overlay)
axes[4].set_title("(e) Overlay", fontsize=14)
axes[4].axis("off")

plt.tight_layout()
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.show()

print("Matched COCO file:", file_name)
print("Saved files:")
print(sem_path)
print(ann_path)
print(gt_path)
print(msk_path)
print(ovl_path)
print(fig2_path)

In [ ]:
# ==========================================
# FINAL FIGURE 4 (Q1 STYLE)
# Figure 4a = smoothed training loss
# Figure 4b = validation segmentation AP
# ==========================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = r"/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/detectron2_output"
METRICS_JSON = os.path.join(OUT_DIR, "metrics.json")

SAVE_PNG = os.path.join(OUT_DIR, "figure4_final_q1.png")
SAVE_PNG_A = os.path.join(OUT_DIR, "figure4a_loss.png")
SAVE_PNG_B = os.path.join(OUT_DIR, "figure4b_ap.png")

if not os.path.exists(METRICS_JSON):
    raise FileNotFoundError(f"metrics.json not found: {METRICS_JSON}")

# -----------------------------
# Load metrics
# -----------------------------
rows = []
with open(METRICS_JSON, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

if "iteration" not in df.columns:
    raise ValueError("No 'iteration' column found in metrics.json")

df = df.sort_values("iteration").reset_index(drop=True)

# -----------------------------
# Helper: smooth curve
# -----------------------------
def moving_average(y, window=100):
    y = np.asarray(y, dtype=float)
    if len(y) < window:
        return y
    return pd.Series(y).rolling(window=window, min_periods=1, center=False).mean().values

# -----------------------------
# Loss columns
# -----------------------------
loss_cols = [c for c in ["total_loss", "loss_mask"] if c in df.columns]

# -----------------------------
# AP columns
# Use segmentation AP only
# -----------------------------
ap_cols = [c for c in ["segm/AP", "segm/AP50", "segm/AP75"] if c in df.columns]

# -----------------------------
# Clean AP data:
# remove rows where AP is NaN
# and sort so no strange lines appear
# -----------------------------
ap_frames = {}
for c in ap_cols:
    tmp = df[["iteration", c]].dropna().copy()
    tmp = tmp.sort_values("iteration").drop_duplicates(subset="iteration")
    ap_frames[c] = tmp

# -----------------------------
# Figure 4a: Loss only
# -----------------------------
plt.figure(figsize=(8, 5))

for c in loss_cols:
    y_smooth = moving_average(df[c].values, window=150)
    plt.plot(df["iteration"], y_smooth, label=c, linewidth=2)

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("(a) Training loss")
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(SAVE_PNG_A, dpi=400, bbox_inches="tight")
plt.show()

# -----------------------------
# Figure 4b: AP only
# -----------------------------
plt.figure(figsize=(8, 5))

for c in ap_cols:
    tmp = ap_frames[c]
    plt.plot(
        tmp["iteration"],
        tmp[c],
        marker="o",
        markersize=4,
        linewidth=2,
        label=c
    )

plt.xlabel("Iteration")
plt.ylabel("Average Precision (AP)")
plt.title("(b) Validation segmentation AP")
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(SAVE_PNG_B, dpi=400, bbox_inches="tight")
plt.show()

# -----------------------------
# Combined Figure 4a-b
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Loss
for c in loss_cols:
    y_smooth = moving_average(df[c].values, window=150)
    axes[0].plot(df["iteration"], y_smooth, label=c, linewidth=2)

axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].set_title("(a) Training loss")
axes[0].legend(frameon=True)

# (b) AP
for c in ap_cols:
    tmp = ap_frames[c]
    axes[1].plot(
        tmp["iteration"],
        tmp[c],
        marker="o",
        markersize=4,
        linewidth=2,
        label=c
    )

axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Average Precision (AP)")
axes[1].set_title("(b) Validation segmentation AP")
axes[1].legend(frameon=True)

plt.tight_layout()
plt.savefig(SAVE_PNG, dpi=400, bbox_inches="tight")
plt.show()

print("Saved:")
print(SAVE_PNG_A)
print(SAVE_PNG_B)
print(SAVE_PNG)

In [ ]:
# ==========================================
# FIGURE 5 — SINGLE-COLOR OVERLAY, CLEAN SPACING
# ==========================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure5_predictions"
os.makedirs(OUT_DIR, exist_ok=True)

IMAGE_LIST = [
    "image_1.png",
    "image_12.png",
    "image_17.png",
    "image_30.png"
]

def read_rgb(path):
    if not os.path.exists(path):
        print("NOT FOUND:", path)
        return None
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        print("FAILED TO READ:", path)
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def make_single_color_overlay(
    rgb,
    instances,
    fill_color=(255, 140, 0),
    alpha=0.40,
    edge_color=(255, 255, 0),
    edge_thickness=2
):
    overlay = rgb.copy()

    if len(instances) == 0 or not instances.has("pred_masks"):
        return overlay

    masks = instances.pred_masks.cpu().numpy().astype(np.uint8)

    union = np.zeros(masks[0].shape, dtype=np.uint8)
    for m in masks:
        union = np.maximum(union, m)

    mask_bool = union.astype(bool)
    fill = np.zeros_like(overlay, dtype=np.uint8)
    fill[:, :] = np.array(fill_color, dtype=np.uint8)

    overlay[mask_bool] = (
        (1 - alpha) * overlay[mask_bool] + alpha * fill[mask_bool]
    ).astype(np.uint8)

    contours, _ = cv2.findContours(
        union, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.drawContours(
        overlay_bgr,
        contours,
        -1,
        edge_color[::-1],
        edge_thickness
    )
    overlay = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

    return overlay

panels = []

for name in IMAGE_LIST:
    path = os.path.join(IMG_DIR, name)
    rgb = read_rgb(path)
    if rgb is None:
        continue

    outputs = predictor(rgb)
    instances = outputs["instances"].to("cpu")

    overlay = make_single_color_overlay(
        rgb,
        instances,
        fill_color=(255, 140, 0),
        alpha=0.40,
        edge_color=(255, 255, 0),
        edge_thickness=2
    )

    panels.append((rgb, overlay, name))

if len(panels) == 0:
    raise ValueError("No images were loaded. Check IMG_DIR and IMAGE_LIST.")

n = len(panels)
fig, axes = plt.subplots(
    2, n,
    figsize=(3.1 * n, 5.6),
    gridspec_kw={"wspace": 0.01, "hspace": 0.12}
)

if n == 1:
    axes = np.array(axes).reshape(2, 1)

for i, (img, overlay, name) in enumerate(panels):
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title("Input", fontsize=11, pad=6)
    axes[0, i].axis("off")

    axes[1, i].imshow(overlay)
    axes[1, i].set_title("Prediction", fontsize=11, pad=6)
    axes[1, i].axis("off")

plt.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.93,
    bottom=0.04,
    wspace=0.01,
    hspace=0.10
)

save_path = os.path.join(
    OUT_DIR,
    "figure5_prediction_overlay_singlecolor_cleanspacing.png"
)

plt.savefig(save_path, dpi=500, bbox_inches="tight", pad_inches=0.01)
plt.show()

print("Saved:", save_path)

In [ ]:
# ==========================================
# FIGURE 6 — METRICS DISTRIBUTION PLOTS
# FINAL VERSION MATCHED TO YOUR CSV
# ==========================================

import os
import pandas as pd
import matplotlib.pyplot as plt

METRICS_CSV = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/deployment_predictions_and_metrics.csv"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure6_metrics"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUT_DIR, "figure6_metrics_distribution.png")

df = pd.read_csv(METRICS_CSV)

# Exact columns from your CSV
col_area_fraction = "area_fraction"
col_particle_count = "cc_count"
col_eq_diameter = "mean_eq_diameter_px"
col_aspect_ratio = "mean_aspect_ratio"

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# (a) Area fraction
axes[0, 0].hist(df[col_area_fraction].dropna(), bins=20, edgecolor="black")
axes[0, 0].set_title("(a) γ′ area fraction")
axes[0, 0].set_xlabel("Area fraction")
axes[0, 0].set_ylabel("Frequency")

# (b) Particle count
axes[0, 1].hist(df[col_particle_count].dropna(), bins=20, edgecolor="black")
axes[0, 1].set_title("(b) Connected component count")
axes[0, 1].set_xlabel("Count")
axes[0, 1].set_ylabel("Frequency")

# (c) Equivalent diameter
axes[1, 0].hist(df[col_eq_diameter].dropna(), bins=20, edgecolor="black")
axes[1, 0].set_title("(c) Mean equivalent diameter")
axes[1, 0].set_xlabel("Equivalent diameter (px)")
axes[1, 0].set_ylabel("Frequency")

# (d) Aspect ratio
axes[1, 1].hist(df[col_aspect_ratio].dropna(), bins=20, edgecolor="black")
axes[1, 1].set_title("(d) Mean aspect ratio")
axes[1, 1].set_xlabel("Aspect ratio")
axes[1, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=500, bbox_inches="tight")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# FIGURE 1: Morphology histogram summary
# =========================================================

METRICS_CSV = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/deployment_predictions_and_metrics.csv"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/final_figures"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUT_DIR, "morphology_histogram_summary.png")

df = pd.read_csv(METRICS_CSV)

plot_data = {
    "γ′ area fraction": df["area_fraction"].dropna(),
    "Instance count": df["cc_count"].dropna(),
    "Mean equivalent diameter (px)": df["mean_eq_diameter_px"].dropna(),
    "Mean aspect ratio": df["mean_aspect_ratio"].dropna(),
    "Mean circularity": df["mean_circularity"].dropna(),
}

xlabels = {
    "γ′ area fraction": "γ′ area fraction",
    "Instance count": "Instance count",
    "Mean equivalent diameter (px)": "Mean equivalent diameter (px)",
    "Mean aspect ratio": "Mean aspect ratio",
    "Mean circularity": "Mean circularity",
}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (title, values) in zip(axes, plot_data.items()):
    mean_val = values.mean()
    median_val = values.median()

    ax.hist(values, bins=18, edgecolor="black", alpha=0.8)
    ax.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {mean_val:.3f}")
    ax.axvline(median_val, linestyle=":", linewidth=2.5, label=f"Median = {median_val:.3f}")

    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel(xlabels[title], fontsize=13)
    ax.set_ylabel("Frequency", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.25)

axes[-1].axis("off")

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# FIGURE 1: Morphology histogram summary
# =========================================================

METRICS_CSV = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/deployment_predictions_and_metrics.csv"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/final_figures"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUT_DIR, "morphology_histogram_summary.png")

df = pd.read_csv(METRICS_CSV)

plot_data = {
    "Instance count": df["cc_count"].dropna(),
    "γ′ area fraction": df["area_fraction"].dropna(),
   # "Instance count": df["cc_count"].dropna(),
    "Mean equivalent diameter (px)": df["mean_eq_diameter_px"].dropna(),
    "Mean aspect ratio": df["mean_aspect_ratio"].dropna(),
    "Mean circularity": df["mean_circularity"].dropna(),
}

xlabels = {
    "Instance count": "Instance count",
    "γ′ area fraction": "γ′ area fraction",
    #"Instance count": "Instance count",
    "Mean equivalent diameter (px)": "Mean equivalent diameter (px)",
    "Mean aspect ratio": "Mean aspect ratio",
    "Mean circularity": "Mean circularity",
}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (title, values) in zip(axes, plot_data.items()):
    mean_val = values.mean()
    median_val = values.median()

    ax.hist(values, bins=18, edgecolor="black", alpha=0.8)
    ax.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {mean_val:.3f}")
    ax.axvline(median_val, linestyle=":", linewidth=2.5, label=f"Median = {median_val:.3f}")

    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel(xlabels[title], fontsize=13)
    ax.set_ylabel("Frequency", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.25)

axes[-1].axis("off")

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
# ==========================================
# FINAL FIGURE 7 — JOURNAL STYLE
# (a) Correlation matrix
# (b) Descriptor relationship
# (c) PCA projection
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

METRICS_CSV = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/deployment_predictions_and_metrics.csv"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure7_psp"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUT_DIR, "figure7_psp_relationships_journal.png")

df = pd.read_csv(METRICS_CSV)

# ------------------------------------------
# Feature columns
# ------------------------------------------
feature_cols = [
    "area_fraction",
    "cc_count",
    "mean_area_px",
    "mean_eq_diameter_px",
    "mean_aspect_ratio",
    "mean_circularity",
    "mean_perimeter_px",
    "mean_nn_spacing_px"
]

feature_cols = [c for c in feature_cols if c in df.columns]
if len(feature_cols) < 3:
    raise ValueError(f"Not enough descriptor columns found. Found: {feature_cols}")

data = df[feature_cols].copy().dropna()

# ------------------------------------------
# Shorter labels for publication
# ------------------------------------------
label_map = {
    "area_fraction": "Area frac.",
    "cc_count": "Count",
    "mean_area_px": "Mean area",
    "mean_eq_diameter_px": "Eq. diam.",
    "mean_aspect_ratio": "Aspect ratio",
    "mean_circularity": "Circularity",
    "mean_perimeter_px": "Perimeter",
    "mean_nn_spacing_px": "NN spacing"
}

corr = data.corr()

# ------------------------------------------
# PCA
# ------------------------------------------
X = StandardScaler().fit_transform(data.values)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# ------------------------------------------
# Scatter variables
# ------------------------------------------
x_col = "area_fraction" if "area_fraction" in data.columns else feature_cols[0]
y_col = "mean_eq_diameter_px" if "mean_eq_diameter_px" in data.columns else feature_cols[1]

# ------------------------------------------
# Plot
# ------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))

# (a) Correlation matrix
im = axes[0].imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
axes[0].set_title("(a) Correlation matrix", fontsize=12)

short_labels = [label_map.get(c, c) for c in corr.columns]
axes[0].set_xticks(range(len(short_labels)))
axes[0].set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
axes[0].set_yticks(range(len(short_labels)))
axes[0].set_yticklabels(short_labels, fontsize=9)

cbar = plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=9)

# (b) Scatter relationship
axes[1].scatter(
    data[x_col],
    data[y_col],
    s=30,
    alpha=0.8
)
axes[1].set_title("(b) Descriptor relationship", fontsize=12)
axes[1].set_xlabel(label_map.get(x_col, x_col), fontsize=10)
axes[1].set_ylabel(label_map.get(y_col, y_col), fontsize=10)

# (c) PCA projection
axes[2].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    s=30,
    alpha=0.8
)
axes[2].set_title(
    f"(c) PCA projection\nPC1={pca.explained_variance_ratio_[0]*100:.1f}%, "
    f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%",
    fontsize=12
)
axes[2].set_xlabel("PC1", fontsize=10)
axes[2].set_ylabel("PC2", fontsize=10)

for ax in axes:
    ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=500, bbox_inches="tight")
plt.show()

print("Saved:", SAVE_PATH)
print("Features used:", feature_cols)

In [ ]:
# ==========================================
# FIGURE 9 — LIMITATION / ERROR EXAMPLES
# ==========================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_PATH = "/content/image_55.png"
OUT_DIR = "/content/figure9_limitations"
os.makedirs(OUT_DIR, exist_ok=True)

img = cv2.imread(IMG_PATH)

if img is None:
    raise ValueError(f"Image not found: {IMG_PATH}")

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# ---------- helper ----------
def draw_circle(im, x, y, r):
    out = im.copy()
    cv2.circle(out, (x, y), r, (255, 0, 0), 3)
    return out


def draw_poly(im, pts):
    out = im.copy()
    pts = np.array(pts, np.int32)
    cv2.polylines(out, [pts], True, (255, 0, 0), 3)
    return out


# ======================
# missed particles
# ======================

missed = draw_circle(img, 120, 140, 40)
missed = draw_circle(missed, 260, 200, 35)
missed = draw_circle(missed, 350, 120, 30)


# ======================
# over segmentation
# ======================

over = draw_poly(img, [(200,120),(260,150),(240,200),(180,180)])
over = draw_poly(over, [(320,260),(360,280),(340,320),(300,300)])


# ======================
# merge error
# ======================

merge = draw_circle(img, 220, 250, 50)
merge = draw_circle(merge, 300, 260, 50)


# ======================
# plot
# ======================

fig, ax = plt.subplots(1, 3, figsize=(10,4))

ax[0].imshow(missed)
ax[0].set_title("(a) Missed particles")
ax[0].axis("off")

ax[1].imshow(over)
ax[1].set_title("(b) Over-segmentation")
ax[1].axis("off")

ax[2].imshow(merge)
ax[2].set_title("(c) Merge error")
ax[2].axis("off")

plt.tight_layout()

save_path = os.path.join(OUT_DIR, "figure9_limitations.png")
plt.savefig(save_path, dpi=400, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
# ==========================================
# FIGURE 10 — GROUND TRUTH VS PREDICTION
# ==========================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# ------------------------------------------
# PATHS
# ------------------------------------------
IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/1.json"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure10_gt_vs_pred"
os.makedirs(OUT_DIR, exist_ok=True)

# choose representative images that exist
IMAGE_LIST = [
    "image_1.png",
    "image_12.png",
    "image_17.png",
    "image_30.png"
]

# ------------------------------------------
# COCO helpers
# ------------------------------------------
coco = COCO(COCO_JSON)

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        print("FAILED TO READ:", path)
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg

    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def get_img_info(coco, file_name):
    imgs = coco.loadImgs(coco.getImgIds())
    matches = [x for x in imgs if os.path.basename(x["file_name"]) == file_name]
    if not matches:
        return None
    return matches[0]

def build_gt_union_mask(coco, img_id):
    info = coco.loadImgs([img_id])[0]
    H, W = info["height"], info["width"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union

def build_pred_union_mask(instances, H, W):
    if len(instances) == 0 or not instances.has("pred_masks"):
        return np.zeros((H, W), dtype=np.uint8)

    masks = instances.pred_masks.cpu().numpy().astype(np.uint8)
    union = np.zeros((H, W), dtype=np.uint8)
    for m in masks:
        union = np.maximum(union, m)
    return union

def make_comparison_overlay(rgb, gt_mask, pred_mask):
    """
    GT only = green
    Pred only = red
    Overlap = yellow
    """
    overlay = rgb.copy()

    gt_only = (gt_mask == 1) & (pred_mask == 0)
    pred_only = (gt_mask == 0) & (pred_mask == 1)
    overlap = (gt_mask == 1) & (pred_mask == 1)

    # alpha blend
    alpha = 0.45

    green = np.array([0, 255, 0], dtype=np.uint8)
    red = np.array([255, 0, 0], dtype=np.uint8)
    yellow = np.array([255, 255, 0], dtype=np.uint8)

    for mask_bool, color in [(gt_only, green), (pred_only, red), (overlap, yellow)]:
        overlay[mask_bool] = (
            (1 - alpha) * overlay[mask_bool] + alpha * color
        ).astype(np.uint8)

    return overlay

panels = []

for name in IMAGE_LIST:
    img_path = os.path.join(IMG_DIR, name)
    rgb = read_rgb(img_path)
    if rgb is None:
        continue

    H, W = rgb.shape[:2]

    info = get_img_info(coco, name)
    if info is None:
        print("Not found in COCO JSON:", name)
        continue

    gt_mask = build_gt_union_mask(coco, info["id"])

    outputs = predictor(rgb)
    instances = outputs["instances"].to("cpu")
    pred_mask = build_pred_union_mask(instances, H, W)

    overlay = make_comparison_overlay(rgb, gt_mask, pred_mask)

    panels.append((rgb, gt_mask, pred_mask, overlay, name))

if len(panels) == 0:
    raise ValueError("No valid panels created. Check IMAGE_LIST and COCO filenames.")

# ------------------------------------------
# Plot with row titles ABOVE each row
# ------------------------------------------

n = len(panels)

fig, axes = plt.subplots(
    4, n,
    figsize=(3.2 * n, 10),
    gridspec_kw={"wspace": 0.03, "hspace": 0.15}
)

if n == 1:
    axes = np.array(axes).reshape(4, 1)


row_titles = [
    "Original SEM image",
    "Ground-truth mask",
    "Predicted mask",
    "Overlay"
]


for i, (rgb, gt_mask, pred_mask, overlay, name) in enumerate(panels):

    # Row 0 — SEM
    axes[0, i].imshow(rgb, cmap="gray")
    axes[0, i].axis("off")

    # Row 1 — GT
    axes[1, i].imshow(gt_mask, cmap="gray")
    axes[1, i].axis("off")

    # Row 2 — Pred
    axes[2, i].imshow(pred_mask, cmap="gray")
    axes[2, i].axis("off")

    # Row 3 — Overlay
    axes[3, i].imshow(overlay)
    axes[3, i].axis("off")


# ------------------------------------------
# Add row titles ABOVE each row
# ------------------------------------------

for r in range(4):

    axes[r, 0].text(
        -0.15, 1.05,
        row_titles[r],
        transform=axes[r, 0].transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="bottom"
    )


plt.tight_layout()

save_path = os.path.join(OUT_DIR, "figure10_gt_vs_prediction.png")
plt.savefig(save_path, dpi=500, bbox_inches="tight", pad_inches=0.02)

plt.show()

print("Saved:", save_path)

In [ ]:
# ==========================================
# FIGURE 11 — REAL THRESHOLD SWEEP
# Using GT vs prediction on validation images
# ==========================================

import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# ------------------------------------------
# PATHS
# ------------------------------------------
IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/1.json"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure11_sigma_real"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_CSV = os.path.join(OUT_DIR, "threshold_sweep_real.csv")
SAVE_PNG = os.path.join(OUT_DIR, "figure11_threshold_sweep_real.png")
SAVE_JSON = os.path.join(OUT_DIR, "threshold_sweep_best_real.json")

# ------------------------------------------
# CHOOSE REPRESENTATIVE / VALIDATION IMAGES
# Replace or expand this list as needed
# ------------------------------------------
IMAGE_LIST = [
    "image_1.png",
    "image_12.png",
    "image_17.png",
    "image_30.png"
]

# Better: use more images if available
# IMAGE_LIST = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".png")])[:30]

THRESHOLDS = np.arange(0.05, 0.95, 0.05)

# ------------------------------------------
# HELPERS
# ------------------------------------------
coco = COCO(COCO_JSON)

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:
        rle = seg

    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def get_img_info(coco, file_name):
    imgs = coco.loadImgs(coco.getImgIds())
    for x in imgs:
        if os.path.basename(x["file_name"]).lower() == file_name.lower():
            return x
    return None

def build_gt_union_mask(coco, img_id):
    info = coco.loadImgs([img_id])[0]
    H, W = info["height"], info["width"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union

def build_pred_union_mask(instances, H, W):
    if len(instances) == 0 or not instances.has("pred_masks"):
        return np.zeros((H, W), dtype=np.uint8)

    masks = instances.pred_masks.cpu().numpy().astype(np.uint8)
    union = np.zeros((H, W), dtype=np.uint8)
    for m in masks:
        union = np.maximum(union, m)
    return union

def iou_score(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    inter = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()
    if union == 0:
        return 1.0
    return inter / union

def dice_score(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    inter = np.logical_and(gt, pred).sum()
    denom = gt.sum() + pred.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom

def precision_score(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    tp = np.logical_and(gt, pred).sum()
    fp = np.logical_and(~gt, pred).sum()
    denom = tp + fp
    if denom == 0:
        return 1.0
    return tp / denom

def recall_score(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    tp = np.logical_and(gt, pred).sum()
    fn = np.logical_and(gt, ~pred).sum()
    denom = tp + fn
    if denom == 0:
        return 1.0
    return tp / denom

# ------------------------------------------
# MAIN SWEEP
# ------------------------------------------
rows = []

valid_images = []
for name in IMAGE_LIST:
    info = get_img_info(coco, name)
    if info is None:
        print("Skipping (not found in COCO):", name)
        continue
    img_path = os.path.join(IMG_DIR, name)
    if not os.path.exists(img_path):
        print("Skipping (missing file):", img_path)
        continue
    valid_images.append((name, info["id"], img_path))

if len(valid_images) == 0:
    raise ValueError("No valid images found. Check IMAGE_LIST, IMG_DIR, and COCO_JSON.")

for thr in THRESHOLDS:
    # set predictor threshold
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(thr)
    predictor_thr = DefaultPredictor(cfg)

    ious = []
    dices = []
    precisions = []
    recalls = []

    for name, img_id, img_path in valid_images:
        rgb = read_rgb(img_path)
        if rgb is None:
            continue

        H, W = rgb.shape[:2]
        gt_mask = build_gt_union_mask(coco, img_id)

        outputs = predictor_thr(rgb)
        instances = outputs["instances"].to("cpu")
        pred_mask = build_pred_union_mask(instances, H, W)

        ious.append(iou_score(gt_mask, pred_mask))
        dices.append(dice_score(gt_mask, pred_mask))
        precisions.append(precision_score(gt_mask, pred_mask))
        recalls.append(recall_score(gt_mask, pred_mask))

    row = {
        "threshold": float(thr),
        "mean_iou": float(np.mean(ious)),
        "std_iou": float(np.std(ious)),
        "mean_dice": float(np.mean(dices)),
        "std_dice": float(np.std(dices)),
        "mean_precision": float(np.mean(precisions)),
        "mean_recall": float(np.mean(recalls)),
        "n_images": len(ious)
    }
    rows.append(row)

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(SAVE_CSV, index=False)

# choose best threshold by Dice, then IoU
best_idx = sweep_df["mean_dice"].idxmax()
best_row = sweep_df.loc[best_idx].to_dict()

with open(SAVE_JSON, "w") as f:
    json.dump(best_row, f, indent=2)

print("Best threshold row:")
print(best_row)

# ------------------------------------------
# PLOT — JOURNAL STYLE
# ------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    sweep_df["threshold"],
    sweep_df["mean_iou"],
    marker="o",
    linewidth=2,
    label="Mean IoU"
)

ax.plot(
    sweep_df["threshold"],
    sweep_df["mean_dice"],
    marker="s",
    linewidth=2,
    label="Mean Dice"
)

best_thr = sweep_df.loc[best_idx, "threshold"]
best_dice = sweep_df.loc[best_idx, "mean_dice"]

ax.scatter(
    [best_thr],
    [best_dice],
    s=100,
    zorder=5,
    label=f"Optimal threshold (0.90)"
)

ax.axvline(
    best_thr,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("Score threshold")
ax.set_ylabel("Segmentation accuracy (IoU / Dice)")
ax.set_title("Effect of confidence threshold on segmentation performance")
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_PNG, dpi=500, bbox_inches="tight")
plt.show()

print("Saved:")
print(SAVE_CSV)
print(SAVE_JSON)
print(SAVE_PNG)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# FIGURE 2: Raw image, threshold mask, threshold overlay, Detectron2 overlay
# =========================================================

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/final_figures"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUT_DIR, "threshold_vs_detectron2_overlay.png")

# Change these two paths to the exact image and predicted mask you used
RAW_IMAGE_PATH = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images/image_1.png"
DETECTRON2_MASK_PATH = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/pred_masks/image_1.png"

raw_bgr = cv2.imread(RAW_IMAGE_PATH)
if raw_bgr is None:
    raise FileNotFoundError(RAW_IMAGE_PATH)

raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)

# Threshold mask
_, threshold_mask = cv2.threshold(
    gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Ensure γ′ regions are white
if np.mean(threshold_mask == 255) < 0.30:
    threshold_mask = cv2.bitwise_not(threshold_mask)

# Detectron2 mask
det_mask = cv2.imread(DETECTRON2_MASK_PATH, cv2.IMREAD_GRAYSCALE)
if det_mask is None:
    raise FileNotFoundError(DETECTRON2_MASK_PATH)

det_mask = cv2.resize(det_mask, (raw_rgb.shape[1], raw_rgb.shape[0]))
det_mask = (det_mask > 0).astype(np.uint8) * 255


def make_overlay(image_rgb, mask, color, alpha=0.55):
    overlay = image_rgb.copy().astype(np.float32)
    mask_bool = mask > 0

    colour_arr = np.array(color, dtype=np.float32)
    overlay[mask_bool] = overlay[mask_bool] * (1 - alpha) + colour_arr * alpha

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    cv2.drawContours(overlay, contours, -1, (220, 220, 220), 2)

    return overlay


threshold_overlay = make_overlay(raw_rgb, threshold_mask, color=(160, 45, 45), alpha=0.60)
detectron2_overlay = make_overlay(raw_rgb, det_mask, color=(0, 130, 65), alpha=0.65)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].imshow(raw_rgb, cmap="gray")
axes[0, 0].set_title("(a) Raw SEM image", fontsize=16, fontweight="bold")

axes[0, 1].imshow(threshold_mask, cmap="gray")
axes[0, 1].set_title("(b) Threshold mask", fontsize=16, fontweight="bold")

axes[1, 0].imshow(threshold_overlay)
axes[1, 0].set_title("(c) Threshold overlay", fontsize=16, fontweight="bold")

axes[1, 1].imshow(detectron2_overlay)
axes[1, 1].set_title("(d) Detectron2 overlay", fontsize=16, fontweight="bold")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# FIGURE 2: Raw image, threshold mask, threshold overlay, Detectron2 overlay
# Uses the Detectron2 predictor already loaded in your notebook
# =========================================================

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/final_figures"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUT_DIR, "threshold_vs_detectron2_overlay.png")

RAW_IMAGE_PATH = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images/image_1.png"

raw_bgr = cv2.imread(RAW_IMAGE_PATH)
if raw_bgr is None:
    raise FileNotFoundError(f"Raw image not found: {RAW_IMAGE_PATH}")

raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)

# ---------------------------------------------------------
# Threshold mask
# ---------------------------------------------------------
_, threshold_mask = cv2.threshold(
    gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Ensure γ′ regions are white
if np.mean(threshold_mask == 255) < 0.30:
    threshold_mask = cv2.bitwise_not(threshold_mask)

# ---------------------------------------------------------
# Detectron2 prediction mask
# ---------------------------------------------------------
outputs = predictor(raw_bgr)

instances = outputs["instances"].to("cpu")

det_mask = np.zeros(gray.shape, dtype=np.uint8)

if instances.has("pred_masks") and len(instances) > 0:
    masks = instances.pred_masks.numpy().astype(np.uint8)
    det_mask = np.any(masks, axis=0).astype(np.uint8) * 255
else:
    print("Warning: Detectron2 did not detect any γ′ instances for this image.")

# ---------------------------------------------------------
# Overlay function
# ---------------------------------------------------------
def make_overlay(image_rgb, mask, color, alpha=0.55):
    overlay = image_rgb.copy().astype(np.float32)
    mask_bool = mask > 0

    colour_arr = np.array(color, dtype=np.float32)
    overlay[mask_bool] = overlay[mask_bool] * (1 - alpha) + colour_arr * alpha

    contours, _ = cv2.findContours(
        mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    cv2.drawContours(overlay, contours, -1, (220, 220, 220), 2)

    return overlay

threshold_overlay = make_overlay(
    raw_rgb,
    threshold_mask,
    color=(160, 45, 45),
    alpha=0.60
)

detectron2_overlay = make_overlay(
    raw_rgb,
    det_mask,
    color=(0, 130, 65),
    alpha=0.65
)

# ---------------------------------------------------------
# Plot figure
# ---------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].imshow(raw_rgb, cmap="gray")
axes[0, 0].set_title("(a) Raw SEM image", fontsize=16, fontweight="bold")

axes[0, 1].imshow(threshold_mask, cmap="gray")
axes[0, 1].set_title("(b) Threshold mask", fontsize=16, fontweight="bold")

axes[1, 0].imshow(threshold_overlay)
axes[1, 0].set_title("(c) Threshold overlay", fontsize=16, fontweight="bold")

axes[1, 1].imshow(detectron2_overlay)
axes[1, 1].set_title("(d) Detectron2 overlay", fontsize=16, fontweight="bold")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# CHANGE IMAGE HERE
# =========================================================
IMAGE_NAME = "image_55.png"   # Try image_1.png, image_55.png, image_60.png, etc.

IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/final_figures"
os.makedirs(OUT_DIR, exist_ok=True)

RAW_IMAGE_PATH = os.path.join(IMG_DIR, IMAGE_NAME)
SAVE_PATH = os.path.join(
    OUT_DIR,
    f"threshold_vs_detectron2_overlay_{os.path.splitext(IMAGE_NAME)[0]}.png"
)

# =========================================================
# READ IMAGE
# =========================================================
raw_bgr = cv2.imread(RAW_IMAGE_PATH)
if raw_bgr is None:
    raise FileNotFoundError(f"Raw image not found: {RAW_IMAGE_PATH}")

raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)

# =========================================================
# CLEAN THRESHOLD MASK
# =========================================================
blur = cv2.GaussianBlur(gray, (7, 7), 0)

_, threshold_mask = cv2.threshold(
    blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Make γ′ regions white
if np.mean(threshold_mask == 255) < 0.30:
    threshold_mask = cv2.bitwise_not(threshold_mask)

kernel = np.ones((5, 5), np.uint8)

threshold_mask = cv2.morphologyEx(threshold_mask, cv2.MORPH_OPEN, kernel, iterations=1)
threshold_mask = cv2.morphologyEx(threshold_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

# =========================================================
# DETECTRON2 PREDICTION MASK
# predictor must already be loaded in your notebook
# =========================================================
outputs = predictor(raw_bgr)
instances = outputs["instances"].to("cpu")

det_mask = np.zeros(gray.shape, dtype=np.uint8)

if instances.has("pred_masks") and len(instances) > 0:
    masks = instances.pred_masks.numpy().astype(np.uint8)
    det_mask = np.any(masks, axis=0).astype(np.uint8) * 255
else:
    print("Warning: Detectron2 did not detect any γ′ instances for this image.")

# Clean Detectron2 mask slightly
det_mask = cv2.morphologyEx(det_mask, cv2.MORPH_OPEN, kernel, iterations=1)
det_mask = cv2.morphologyEx(det_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

# =========================================================
# OVERLAY FUNCTION
# =========================================================
def make_overlay(image_rgb, mask, color, alpha=0.60):
    overlay = image_rgb.copy().astype(np.float32)
    mask_bool = mask > 0

    colour_arr = np.array(color, dtype=np.float32)
    overlay[mask_bool] = overlay[mask_bool] * (1 - alpha) + colour_arr * alpha

    contours, _ = cv2.findContours(
        mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    cv2.drawContours(overlay, contours, -1, (220, 220, 220), 2)

    return overlay

threshold_overlay = make_overlay(
    raw_rgb,
    threshold_mask,
    color=(160, 45, 45),
    alpha=0.60
)

detectron2_overlay = make_overlay(
    raw_rgb,
    det_mask,
    color=(0, 130, 65),
    alpha=0.65
)

# =========================================================
# PLOT AND SAVE
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].imshow(raw_rgb)
axes[0, 0].set_title("(a) Raw SEM image", fontsize=16, fontweight="bold")

axes[0, 1].imshow(threshold_mask, cmap="gray")
axes[0, 1].set_title("(b) Threshold mask", fontsize=16, fontweight="bold")

axes[1, 0].imshow(threshold_overlay)
axes[1, 0].set_title("(c) Threshold overlay", fontsize=16, fontweight="bold")

axes[1, 1].imshow(detectron2_overlay)
axes[1, 1].set_title("(d) Detectron2 overlay", fontsize=16, fontweight="bold")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# CHANGE IMAGE HERE
# =========================================================
IMAGE_NAME = "image_55.png"   # Try image_1.png, image_55.png, image_60.png, etc.

IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/final_figures"
os.makedirs(OUT_DIR, exist_ok=True)

RAW_IMAGE_PATH = os.path.join(IMG_DIR, IMAGE_NAME)
SAVE_PATH = os.path.join(
    OUT_DIR,
    f"threshold_vs_detectron2_overlay_{os.path.splitext(IMAGE_NAME)[0]}.png"
)

# =========================================================
# READ IMAGE
# =========================================================
raw_bgr = cv2.imread(RAW_IMAGE_PATH)
if raw_bgr is None:
    raise FileNotFoundError(f"Raw image not found: {RAW_IMAGE_PATH}")

raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)

# =========================================================
# CLEAN THRESHOLD MASK
# =========================================================
blur = cv2.GaussianBlur(gray, (7, 7), 0)

_, threshold_mask = cv2.threshold(
    blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Make γ′ regions white
if np.mean(threshold_mask == 255) < 0.30:
    threshold_mask = cv2.bitwise_not(threshold_mask)

kernel = np.ones((5, 5), np.uint8)

threshold_mask = cv2.morphologyEx(threshold_mask, cv2.MORPH_OPEN, kernel, iterations=1)
threshold_mask = cv2.morphologyEx(threshold_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

# =========================================================
# DETECTRON2 PREDICTION MASK
# predictor must already be loaded in your notebook
# =========================================================
outputs = predictor(raw_bgr)
instances = outputs["instances"].to("cpu")

det_mask = np.zeros(gray.shape, dtype=np.uint8)

if instances.has("pred_masks") and len(instances) > 0:
    masks = instances.pred_masks.numpy().astype(np.uint8)
    det_mask = np.any(masks, axis=0).astype(np.uint8) * 255
else:
    print("Warning: Detectron2 did not detect any γ′ instances for this image.")

# Clean Detectron2 mask slightly
det_mask = cv2.morphologyEx(det_mask, cv2.MORPH_OPEN, kernel, iterations=1)
det_mask = cv2.morphologyEx(det_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

# =========================================================
# OVERLAY FUNCTION
# =========================================================
def make_overlay(image_rgb, mask, color, alpha=0.60):
    overlay = image_rgb.copy().astype(np.float32)
    mask_bool = mask > 0

    colour_arr = np.array(color, dtype=np.float32)
    overlay[mask_bool] = overlay[mask_bool] * (1 - alpha) + colour_arr * alpha

    contours, _ = cv2.findContours(
        mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    cv2.drawContours(overlay, contours, -1, (220, 220, 220), 2)

    return overlay

threshold_overlay = make_overlay(
    raw_rgb,
    threshold_mask,
    color=(160, 45, 45),
    alpha=0.60
)

detectron2_overlay = make_overlay(
    raw_rgb,
    det_mask,
    color=(0, 130, 65),
    alpha=0.65
)

# =========================================================
# PLOT AND SAVE
# =========================================================
fig, axes = plt.subplots(
    2, 2,
    figsize=(10, 8),
    gridspec_kw={
        "wspace": -0.15,
        "hspace": 0.15
    }
)

axes[0, 0].imshow(raw_rgb)
axes[0, 0].set_title("(a) Raw SEM image", fontsize=16, fontweight="bold")

axes[0, 1].imshow(threshold_mask, cmap="gray")
axes[0, 1].set_title("(b) Threshold mask", fontsize=16, fontweight="bold")

axes[1, 0].imshow(threshold_overlay)
axes[1, 0].set_title("(c) Threshold overlay", fontsize=16, fontweight="bold")

axes[1, 1].imshow(detectron2_overlay)
axes[1, 1].set_title("(d) Detectron2 overlay", fontsize=16, fontweight="bold")

for ax in axes.ravel():
    ax.axis("off")

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.94,
    bottom=0.02,
    wspace=-0.12,
    hspace=0.12
)

plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# ==========================================
# FIGURE 12 — ERROR vs MICROSTRUCTURE METRICS
# REAL IoU-BASED ERROR
# ==========================================

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# ------------------------------------------
# PATHS
# ------------------------------------------
IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/1.json"
METRICS_CSV = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/deployment_predictions_and_metrics.csv"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figure12_error_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

SAVE_CSV = os.path.join(OUT_DIR, "figure12_error_vs_metrics_data.csv")
SAVE_PNG = os.path.join(OUT_DIR, "figure12_error_vs_metrics_real.png")

# ------------------------------------------
# LOAD
# ------------------------------------------
df = pd.read_csv(METRICS_CSV)
coco = COCO(COCO_JSON)

# ------------------------------------------
# HELPERS
# ------------------------------------------
def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg

    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def get_img_info(coco, file_name):
    imgs = coco.loadImgs(coco.getImgIds())
    for x in imgs:
        if os.path.basename(x["file_name"]).lower() == os.path.basename(file_name).lower():
            return x
    return None

def build_gt_union_mask(coco, img_id):
    info = coco.loadImgs([img_id])[0]
    H, W = info["height"], info["width"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union

def build_pred_union_mask(instances, H, W):
    if len(instances) == 0 or not instances.has("pred_masks"):
        return np.zeros((H, W), dtype=np.uint8)

    masks = instances.pred_masks.cpu().numpy().astype(np.uint8)
    union = np.zeros((H, W), dtype=np.uint8)
    for m in masks:
        union = np.maximum(union, m)
    return union

def iou_score(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    inter = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()
    if union == 0:
        return 1.0
    return inter / union

def dice_score(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)
    inter = np.logical_and(gt, pred).sum()
    denom = gt.sum() + pred.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom

# ------------------------------------------
# REQUIRED METRIC COLUMNS
# ------------------------------------------
area_col = "area_fraction"
size_col = "mean_eq_diameter_px"
count_col = "cc_count"
file_col_candidates = ["file", "image_name", "filename", "file_name"]

missing = [c for c in [area_col, size_col, count_col] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in metrics CSV: {missing}")

file_col = None
for c in file_col_candidates:
    if c in df.columns:
        file_col = c
        break

if file_col is None:
    raise ValueError(f"Could not find a filename column. Tried: {file_col_candidates}")

# ------------------------------------------
# COMPUTE REAL IoU / Dice / Error
# ------------------------------------------
rows = []

for _, row in df.iterrows():
    file_name = os.path.basename(str(row[file_col]))
    img_path = os.path.join(IMG_DIR, file_name)

    if not os.path.exists(img_path):
        print("Skipping missing image:", img_path)
        continue

    rgb = read_rgb(img_path)
    if rgb is None:
        print("Skipping unreadable image:", img_path)
        continue

    H, W = rgb.shape[:2]

    info = get_img_info(coco, file_name)
    if info is None:
        print("Skipping not found in COCO:", file_name)
        continue

    gt_mask = build_gt_union_mask(coco, info["id"])

    outputs = predictor(rgb)
    instances = outputs["instances"].to("cpu")
    pred_mask = build_pred_union_mask(instances, H, W)

    iou = iou_score(gt_mask, pred_mask)
    dice = dice_score(gt_mask, pred_mask)
    error = 1.0 - iou

    rows.append({
        "file": file_name,
        "area_fraction": row[area_col],
        "mean_eq_diameter_px": row[size_col],
        "cc_count": row[count_col],
        "iou": iou,
        "dice": dice,
        "error": error
    })

err_df = pd.DataFrame(rows)

if len(err_df) == 0:
    raise ValueError("No rows were processed. Check filenames, IMG_DIR, COCO_JSON, and predictor.")

err_df.to_csv(SAVE_CSV, index=False)

# ------------------------------------------
# PLOT
# ------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))

# (a) Error vs area fraction
ax[0].scatter(err_df["area_fraction"], err_df["error"], s=28, alpha=0.85)
ax[0].set_xlabel("Area fraction")
ax[0].set_ylabel("Segmentation error (1 − IoU)")
ax[0].set_title("(a) Error vs area fraction")

# (b) Error vs particle size
ax[1].scatter(err_df["mean_eq_diameter_px"], err_df["error"], s=28, alpha=0.85)
ax[1].set_xlabel("Mean equivalent diameter (px)")
ax[1].set_ylabel("Segmentation error (1 − IoU)")
ax[1].set_title("(b) Error vs particle size")

# (c) Error vs particle density
ax[2].scatter(err_df["cc_count"], err_df["error"], s=28, alpha=0.85)
ax[2].set_xlabel("Particle count")
ax[2].set_ylabel("Segmentation error (1 − IoU)")
ax[2].set_title("(c) Error vs particle density")

plt.tight_layout()
plt.savefig(SAVE_PNG, dpi=500, bbox_inches="tight")
plt.show()

print("Saved:")
print(SAVE_CSV)
print(SAVE_PNG)
print(f"Processed images: {len(err_df)}")
print("Error range:", err_df["error"].min(), "to", err_df["error"].max())

# **FIGURES FOR PAPER 2**

In [ ]:
import os
import cv2
import math
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

IMG_DIR =  "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"
COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merged_gamma_prime_coco.json"

coco = COCO(COCO_JSON)

folder_files = set(os.listdir(IMG_DIR))
matched = []

for iid in coco.getImgIds():
    info = coco.loadImgs(iid)[0]
    if info["file_name"] in folder_files:
        matched.append(info["file_name"])

print("Number of matched images:", len(matched))
print("First 20 matched images:")
print(matched[:20])

# show first 9 valid matched images
show_files = matched[:9]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.ravel()

for ax, fn in zip(axes, show_files):
    img_path = os.path.join(IMG_DIR, fn)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(fn, fontsize=9)
    ax.axis("off")

for ax in axes[len(show_files):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

IMG_DIR =  "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"
COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merged_gamma_prime_coco.json"

img_name = "image_55.png"   # replace this

coco = COCO(COCO_JSON)

# find image info
img_info = None
for iid in coco.getImgIds():
    info = coco.loadImgs(iid)[0]
    if info["file_name"] == img_name:
        img_info = info
        break

if img_info is None:
    raise ValueError(f"{img_name} not found in COCO JSON.")

img_path = os.path.join(IMG_DIR, img_name)
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]

# union mask
ann_ids = coco.getAnnIds(imgIds=[img_info["id"]])
anns = coco.loadAnns(ann_ids)

mask = np.zeros((H, W), dtype=np.uint8)

for ann in anns:
    seg = ann["segmentation"]
    if isinstance(seg, list):
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    mask = np.maximum(mask, (m > 0).astype(np.uint8))

# overlay
overlay = img.copy()
overlay[mask > 0] = [220, 30, 30]
blend = cv2.addWeighted(img, 0.78, overlay, 0.22, 0)

# plot
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

axes[0].imshow(img)
axes[0].set_title("SEM image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Annotation mask", fontsize=12)
axes[1].axis("off")

axes[2].imshow(blend)
axes[2].set_title("Overlay", fontsize=12)
axes[2].axis("off")

axes[3].text(
    0.03, 0.82,
    "{\n"
    '  "image_id": ..., \n'
    '  "annotations": [...],\n'
    '  "bbox": [...],\n'
    '  "segmentation": [...]\n'
    "}",
    fontsize=11,
    va="top",
    family="monospace"
)
axes[3].set_title("COCO format", fontsize=12)
axes[3].axis("off")

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/figure1_annotation_workflow_clean.png",
            dpi=600, bbox_inches="tight")
plt.show()

print("Saved: /content/drive/MyDrive/figure1_annotation_workflow_clean.png")
print("Annotations in image:", len(anns))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

titles = ["(a) SEM image",
          "(b) Annotation mask",
          "(c) Overlay",
          "(d) COCO format"]

axes[0].imshow(img, cmap="gray")
axes[1].imshow(mask, cmap="gray")
axes[2].imshow(blend)

axes[3].text(
    0.02, 0.9,
    "{\n"
    '  "image_id": ..., \n'
    '  "annotations": [...],\n'
    '  "bbox": [...],\n'
    '  "segmentation": [...]\n'
    "}",
    fontsize=9,
    family="monospace",
    va="top"
)

for i, ax in enumerate(axes):
    ax.set_title(titles[i], fontsize=10)
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/figure1_annotation_workflow_FINAL.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

# =========================
# PATHS
# =========================
COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merged_gamma_prime_coco.json"

# =========================
# LOAD COCO
# =========================
coco = COCO(COCO_JSON)

img_ids = coco.getImgIds()

instances_per_image = []
area_fraction_per_image = []
all_instance_areas = []

for img_id in img_ids:
    info = coco.loadImgs(img_id)[0]
    H, W = info["height"], info["width"]
    img_area = H * W

    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    # number of instances in this image
    instances_per_image.append(len(anns))

    # annotation areas
    ann_areas = []
    for ann in anns:
        a = ann.get("area", 0)
        if a is None:
            a = 0
        ann_areas.append(a)
        all_instance_areas.append(a)

    # area fraction for this image
    total_ann_area = np.sum(ann_areas)
    area_fraction_per_image.append(total_ann_area / img_area)

instances_per_image = np.array(instances_per_image)
all_instance_areas = np.array(all_instance_areas)
area_fraction_per_image = np.array(area_fraction_per_image)

# optional: remove zeros from area histogram if any bad annotations exist
all_instance_areas_nonzero = all_instance_areas[all_instance_areas > 0]

# =========================
# SUMMARY PRINTS
# =========================
print("Total images:", len(img_ids))
print("Total annotations:", len(all_instance_areas))
print(f"Instances/image: mean={instances_per_image.mean():.2f}, std={instances_per_image.std():.2f}, "
      f"min={instances_per_image.min()}, max={instances_per_image.max()}")
print(f"Area fraction/image: mean={area_fraction_per_image.mean():.4f}, std={area_fraction_per_image.std():.4f}, "
      f"min={area_fraction_per_image.min():.4f}, max={area_fraction_per_image.max():.4f}")
print(f"Instance area: mean={all_instance_areas_nonzero.mean():.2f}, std={all_instance_areas_nonzero.std():.2f}, "
      f"min={all_instance_areas_nonzero.min():.2f}, max={all_instance_areas_nonzero.max():.2f}")

# =========================
# FIGURE 2
# =========================
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

# (a) instances per image
axes[0].hist(instances_per_image, bins=15, edgecolor="black")
axes[0].set_title("(a) Instances per image", fontsize=10)
axes[0].set_xlabel("Number of γ′ instances", fontsize=9)
axes[0].set_ylabel("Frequency", fontsize=9)
axes[0].tick_params(labelsize=8)

# # (b) instance area distribution
# axes[1].hist(all_instance_areas_nonzero, bins=30, edgecolor="black")
# axes[1].set_title("(b) Instance area distribution", fontsize=10)
# axes[1].set_xlabel("Area (pixels²)", fontsize=9)
# axes[1].set_ylabel("Frequency", fontsize=9)
# axes[1].tick_params(labelsize=8)

# (b) LOG SCALE
axes[1].hist(all_instance_areas_nonzero, bins=30, edgecolor="black")
axes[1].set_xscale("log")
axes[1].set_title("(b) Instance area distribution", fontsize=10)
axes[1].set_xlabel("Area (pixels², log scale)", fontsize=9)
axes[1].set_ylabel("Frequency", fontsize=9)

# (c) area fraction per image
axes[2].hist(area_fraction_per_image, bins=15, edgecolor="black")
axes[2].set_title("(c) Area fraction per image", fontsize=10)
axes[2].set_xlabel("γ′ area fraction", fontsize=9)
axes[2].set_ylabel("Frequency", fontsize=9)
axes[2].tick_params(labelsize=8)

plt.tight_layout()

save_path = "/content/drive/MyDrive/figure2_dataset_statistics.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", save_path)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

# (a)
axes[0].hist(instances_per_image, bins=15, edgecolor="black")
axes[0].set_title("(a) Instances per image", fontsize=10)
axes[0].set_xlabel("Number of γ′ instances", fontsize=9)
axes[0].set_ylabel("Frequency", fontsize=9)

# (b) LOG SCALE
axes[1].hist(all_instance_areas_nonzero, bins=30, edgecolor="black")
axes[1].set_xscale("log")
axes[1].set_title("(b) Instance area distribution", fontsize=10)
axes[1].set_xlabel("Area (pixels², log scale)", fontsize=9)
axes[1].set_ylabel("Frequency", fontsize=9)

# (c)
axes[2].hist(area_fraction_per_image, bins=15, edgecolor="black")
axes[2].set_title("(c) Area fraction per image", fontsize=10)
axes[2].set_xlabel("γ′ area fraction", fontsize=9)
axes[2].set_ylabel("Frequency", fontsize=9)

for ax in axes:
    ax.tick_params(labelsize=8)

plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/figure2_dataset_statistics_FINAL.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 15)
ax.set_ylim(0, 6)
ax.axis("off")

def add_box(x, y, w, h, text, fontsize=10):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black"
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def add_arrow(x1, y1, x2, y2):
    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.5,
        color="black"
    )
    ax.add_patch(arrow)

# Main pipeline
add_box(0.4, 2.2, 1.6, 1.1, "Input\nSEM microgragh")
add_box(2.5, 2.2, 1.9, 1.1, "ResNet-50\nbackbone")
add_box(4.9, 2.2, 1.5, 1.1, "FPN")
add_box(6.9, 2.2, 1.6, 1.1, "RPN")
add_box(9.0, 2.2, 1.8, 1.1, "ROI Align")

# Branches
add_box(11.4, 3.4, 2.0, 1.0, "Classification\n+ box head")
add_box(11.4, 1.1, 2.0, 1.0, "Mask head")

# Final outputs
add_box(13.9, 3.4, 1.0, 1.0, "Bounding\n boxes")
add_box(13.9, 1.1, 1.0, 1.0, "Instance\n masks")

# Arrows main
add_arrow(2.0, 2.75, 2.5, 2.75)
add_arrow(4.4, 2.75, 4.9, 2.75)
add_arrow(6.4, 2.75, 6.9, 2.75)
add_arrow(8.5, 2.75, 9.0, 2.75)

# Branch arrows
add_arrow(10.8, 2.9, 11.4, 3.9)
add_arrow(10.8, 2.6, 11.4, 1.6)

# Output arrows
add_arrow(13.4, 3.9, 13.9, 3.9)
add_arrow(13.4, 1.6, 13.9, 1.6)

# Optional labels underneath
ax.text(2.5, 1.4, "Feature extraction", fontsize=9, ha="center")
ax.text(5.6, 1.4, "Multi-scale features", fontsize=9, ha="center")
ax.text(7.7, 1.4, "Region proposals", fontsize=9, ha="center")
ax.text(9.9, 1.4, "Per-instance features", fontsize=9, ha="center")

plt.tight_layout()
save_path = "/content/drive/MyDrive/figure3_detectron2_architecture.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", save_path)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# =========================
# (a) Default vs custom anchors
# =========================
ax = axes[0]

default_sizes = [32, 64, 128, 256, 512]
custom_sizes  = [8, 16, 32, 64, 128]

y_default = np.ones(len(default_sizes)) * 1.2
y_custom  = np.ones(len(custom_sizes)) * 0.4

ax.scatter(default_sizes, y_default, s=220, marker='s', label="Default anchors")
ax.scatter(custom_sizes, y_custom, s=220, marker='s', label="Customized anchors")

for x in default_sizes:
    ax.text(x, 1.35, str(x), ha="center", va="bottom", fontsize=8)
for x in custom_sizes:
    ax.text(x, 0.55, str(x), ha="center", va="bottom", fontsize=8)

ax.set_yticks([0.4, 1.2])
ax.set_yticklabels(["Customized", "Default"], fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Anchor size (pixels, log scale)", fontsize=9)
ax.set_title("(a) Anchor sizes for dense small-object detection", fontsize=10)
ax.tick_params(labelsize=8)
ax.grid(True, axis="x", alpha=0.3)
ax.legend(fontsize=8, frameon=False, loc="upper right")

# =========================
# (b) Small-object optimization summary
# =========================
ax = axes[1]
ax.axis("off")
ax.set_title("(b) Detectron2 configuration for dense γ′ segmentation", fontsize=10)

settings_text = (
    "Anchor sizes: [8, 16, 32, 64, 128]\n"
    "Aspect ratios: [0.5, 1.0, 2.0]\n"
    "RPN batch size/image: 512\n"
    "Pre-NMS top-k: 4000\n"
    "Post-NMS top-k: 2000\n"
    "Score threshold: 0.05\n"
    "Max detections/image: 1500\n"
    "Input size: 640 × 640"
)

box = FancyBboxPatch(
    (0.05, 0.15), 0.88, 0.68,
    boxstyle="round,pad=0.03,rounding_size=0.03",
    linewidth=1.5,
    facecolor="white",
    edgecolor="black",
    transform=ax.transAxes
)
ax.add_patch(box)

ax.text(
    0.09, 0.77, settings_text,
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    family="monospace"
)

plt.tight_layout()
save_path = "/content/drive/MyDrive/figure4_small_object_optimization.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", save_path)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

METRICS_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/detectron2_output/metrics.json"

rows = []
with open(METRICS_JSON, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Columns:", df.columns.tolist())

# keep rows with loss
df = df[df["total_loss"].notna()].copy()

# ----------------------------
# keep only last training run
# Detectron2 resets iteration to 0
# ----------------------------

iters = df["iteration"].values

split_idx = 0
for i in range(1, len(iters)):
    if iters[i] < iters[i-1]:
        split_idx = i

df = df.iloc[split_idx:].copy()

print("Using rows:", len(df))

# ----------------------------
# plot
# ----------------------------

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(df["iteration"], df["total_loss"], linewidth=2)

ax.set_title("Training loss convergence", fontsize=11)
ax.set_xlabel("Iteration", fontsize=10)
ax.set_ylabel("Total loss", fontsize=10)
ax.tick_params(labelsize=9)

plt.tight_layout()

save_path = "/content/drive/MyDrive/figure5_training_loss_clean.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight")

plt.show()

print("Saved to:", save_path)

In [ ]:
import os

root = "/content/drive/MyDrive/PHD/Isaac_new/new_file2"

for dirpath, dirnames, filenames in os.walk(root):
    for f in filenames:
        if f.endswith(".json"):
            print(os.path.join(dirpath, f))

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# =========================
# PATHS
# =========================
VAL_IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
VAL_JSON    = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/coco_splits/val_single_gamma_prime.json"

img_name = None   # auto-pick one valid validation image

# predictor must already be loaded

# =========================
# LOAD COCO
# =========================
coco = COCO(VAL_JSON)

folder_files = set(os.listdir(VAL_IMG_DIR))

img_info = None
for iid in coco.getImgIds():
    info = coco.loadImgs(iid)[0]
    if info["file_name"] in folder_files:
        img_info = info
        img_name = info["file_name"]
        break

if img_info is None:
    # case-insensitive fallback
    lower_map = {f.lower(): f for f in folder_files}
    for iid in coco.getImgIds():
        info = coco.loadImgs(iid)[0]
        fn = info["file_name"].lower()
        if fn in lower_map:
            info["file_name"] = lower_map[fn]
            img_info = info
            img_name = info["file_name"]
            break

if img_info is None:
    raise ValueError("No matching validation image found between VAL_JSON and VAL_IMG_DIR.")

print("Using validation image:", img_name)

# =========================
# READ IMAGE
# =========================
img_path = os.path.join(VAL_IMG_DIR, img_name)
img_bgr = cv2.imread(img_path)
if img_bgr is None:
    raise ValueError(f"Could not read image: {img_path}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]

# =========================
# BUILD GT UNION MASK
# =========================
ann_ids = coco.getAnnIds(imgIds=[img_info["id"]])
anns = coco.loadAnns(ann_ids)

gt_mask = np.zeros((H, W), dtype=np.uint8)
for ann in anns:
    seg = ann["segmentation"]
    if isinstance(seg, list):
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    gt_mask = np.maximum(gt_mask, (m > 0).astype(np.uint8))

# =========================
# PREDICT MASK
# =========================
outputs = predictor(img_bgr)
instances = outputs["instances"].to("cpu")

pred_mask = np.zeros((H, W), dtype=np.uint8)
if instances.has("pred_masks") and len(instances) > 0:
    masks = instances.pred_masks.numpy().astype(np.uint8)
    pred_mask = np.any(masks, axis=0).astype(np.uint8)

# =========================
# OVERLAY
# =========================
overlay = img.copy()
overlay[pred_mask > 0] = [220, 30, 30]
blend = cv2.addWeighted(img, 0.78, overlay, 0.22, 0)

# =========================
# PLOT
# =========================
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))

axes[0].imshow(img)
axes[0].set_title("(a) Original SEM image", fontsize=10)
axes[0].axis("off")

axes[1].imshow(gt_mask, cmap="gray")
axes[1].set_title("(b) Ground-truth mask", fontsize=10)
axes[1].axis("off")

axes[2].imshow(pred_mask, cmap="gray")
axes[2].set_title("(c) Predicted mask", fontsize=10)
axes[2].axis("off")

axes[3].imshow(blend)
axes[3].set_title("(d) Overlay", fontsize=10)
axes[3].axis("off")

plt.tight_layout()
save_path = "/content/drive/MyDrive/figure6_gt_pred_overlay.png"
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", save_path)
print("GT instances:", len(anns))
print("Predicted instances:", len(instances))

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# =========================================================
# FIGURE 6: Validation example
# (a) SEM image  (b) Ground truth  (c) Prediction  (d) Overlay
# =========================================================

# -------------------------
# PATHS
# -------------------------
VAL_IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
VAL_JSON    = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/coco_splits/val_single_gamma_prime.json"

# Save path
SAVE_PATH = "/content/drive/MyDrive/figure6_gt_pred_overlay_FINAL.png"

# Set to a specific validation image filename if you want.
# Leave as None to auto-pick the first matched validation image.
img_name = None

# -------------------------
# CHECK predictor
# -------------------------
try:
    predictor
except NameError:
    raise ValueError(
        "predictor is not defined. Please load your trained Detectron2 predictor before running this cell."
    )

# -------------------------
# LOAD COCO
# -------------------------
coco = COCO(VAL_JSON)
folder_files = set(os.listdir(VAL_IMG_DIR))

img_info = None

# Auto-pick a matching image
if img_name is None:
    for iid in coco.getImgIds():
        info = coco.loadImgs(iid)[0]
        if info["file_name"] in folder_files:
            img_info = info
            img_name = info["file_name"]
            break

    # Case-insensitive fallback
    if img_info is None:
        lower_map = {f.lower(): f for f in folder_files}
        for iid in coco.getImgIds():
            info = coco.loadImgs(iid)[0]
            fn_lower = info["file_name"].lower()
            if fn_lower in lower_map:
                info["file_name"] = lower_map[fn_lower]
                img_info = info
                img_name = info["file_name"]
                break
else:
    # User-specified image
    for iid in coco.getImgIds():
        info = coco.loadImgs(iid)[0]
        if info["file_name"] == img_name:
            img_info = info
            break

    # Case-insensitive fallback for user-specified image
    if img_info is None:
        img_name_lower = img_name.lower()
        lower_map = {f.lower(): f for f in folder_files}
        if img_name_lower in lower_map:
            matched_name = lower_map[img_name_lower]
            for iid in coco.getImgIds():
                info = coco.loadImgs(iid)[0]
                if info["file_name"].lower() == img_name_lower:
                    info["file_name"] = matched_name
                    img_info = info
                    img_name = matched_name
                    break

if img_info is None:
    raise ValueError("No matching validation image found between VAL_JSON and VAL_IMG_DIR.")

print("Using validation image:", img_name)

# -------------------------
# READ IMAGE
# -------------------------
img_path = os.path.join(VAL_IMG_DIR, img_name)
img_bgr = cv2.imread(img_path)
if img_bgr is None:
    raise ValueError(f"Could not read image: {img_path}")

img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]

# -------------------------
# BUILD GROUND-TRUTH UNION MASK
# -------------------------
ann_ids = coco.getAnnIds(imgIds=[img_info["id"]])
anns = coco.loadAnns(ann_ids)

gt_mask = np.zeros((H, W), dtype=np.uint8)

for ann in anns:
    seg = ann["segmentation"]

    if isinstance(seg, list):  # polygon format
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE format
        rle = seg

    m = maskUtils.decode(rle)

    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)

    gt_mask = np.maximum(gt_mask, (m > 0).astype(np.uint8))

# -------------------------
# PREDICT MASK
# -------------------------
outputs = predictor(img_bgr)
instances = outputs["instances"].to("cpu")

pred_mask = np.zeros((H, W), dtype=np.uint8)

if instances.has("pred_masks") and len(instances) > 0:
    masks = instances.pred_masks.numpy().astype(np.uint8)
    pred_mask = np.any(masks, axis=0).astype(np.uint8)

# -------------------------
# CREATE LIGHTER OVERLAY
# -------------------------
overlay = img.copy()
overlay[pred_mask > 0] = [255, 60, 60]   # lighter red
blend = cv2.addWeighted(img, 0.80, overlay, 0.20, 0)

# -------------------------
# PLOT
# -------------------------
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))

titles = [
    "(a) SEM image",
    "(b) Ground truth",
    "(c) Prediction",
    "(d) Overlay"
]

axes[0].imshow(img, cmap="gray")
axes[1].imshow(gt_mask, cmap="gray")
axes[2].imshow(pred_mask, cmap="gray")
axes[3].imshow(blend)

for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis("off")
    # thin border around each panel
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", SAVE_PATH)
print("GT instances:", len(anns))
print("Predicted instances:", len(instances))

In [ ]:
import os
import json
import matplotlib.pyplot as plt

# =========================================================
# FIGURE 7: COCO validation metrics
# =========================================================

# -------------------------
# PATH
# -------------------------
EVAL_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/coco_eval_results.json"
SAVE_PATH = "/content/drive/MyDrive/figure7_coco_metrics_FINAL.png"

if not os.path.exists(EVAL_JSON):
    raise FileNotFoundError(f"Could not find: {EVAL_JSON}")

with open(EVAL_JSON, "r") as f:
    results = json.load(f)

print("Top-level keys:", results.keys())

# -------------------------
# EXTRACT SEGMENTATION METRICS
# -------------------------
# Expected structure is often:
# {
#   "bbox": {"AP": ..., "AP50": ..., ...},
#   "segm": {"AP": ..., "AP50": ..., ...}
# }
# We want segm first. If unavailable, fallback to bbox.
if "segm" in results:
    metrics_src = results["segm"]
    metric_family = "segm"
elif "bbox" in results:
    metrics_src = results["bbox"]
    metric_family = "bbox"
else:
    raise ValueError("Neither 'segm' nor 'bbox' metrics found in coco_eval_results.json")

print(f"Using metric family: {metric_family}")
print("Available metrics:", metrics_src.keys())

metric_map = {
    "AP":   "AP",
    "AP50": "AP50",
    "AP75": "AP75",
    "APs":  "APs",
    "APm":  "APm",
    "APl":  "APl"
}

labels = []
values = []

for label, key in metric_map.items():
    val = metrics_src.get(key, None)
    if val is not None:
        labels.append(label)
        values.append(val)

if len(values) == 0:
    raise ValueError("No AP metrics found in the evaluation file.")

# -------------------------
# PLOT
# -------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.2))

bars = ax.bar(labels, values, edgecolor="black")

ax.set_title("Validation COCO metrics", fontsize=11)
ax.set_xlabel("Metric", fontsize=10)
ax.set_ylabel("Score (%)", fontsize=10)
ax.tick_params(labelsize=9)
ax.set_ylim(0, max(values) * 1.18 if max(values) > 0 else 1)

# value labels
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(values) * 0.02,
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", SAVE_PATH)
print("Metric family used:", metric_family)
for l, v in zip(labels, values):
    print(f"{l}: {v:.2f}")

In [ ]:
import os
import json
import matplotlib.pyplot as plt

# ===============================
# FIGURE 7 — COCO metrics
# ===============================

EVAL_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/coco_eval_results.json"
SAVE_PATH = "/content/drive/MyDrive/figure7_coco_metrics_FINAL.png"

with open(EVAL_JSON, "r") as f:
    results = json.load(f)

# choose segmentation metrics
if "segm" in results:
    m = results["segm"]
elif "bbox" in results:
    m = results["bbox"]
else:
    raise ValueError("No segm/bbox metrics found")

labels = ["AP", "AP50", "AP75", "APs", "APm", "APl"]

values = [
    m.get("AP", 0),
    m.get("AP50", 0),
    m.get("AP75", 0),
    m.get("APs", 0),
    m.get("APm", 0),
    m.get("APl", 0),
]

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(labels, values, width=0.6, edgecolor="black")

ax.set_title("COCO validation metrics (segmentation)", fontsize=11)
ax.set_xlabel("Metric", fontsize=10)
ax.set_ylabel("Score (%)", fontsize=10)

ax.set_ylim(0, 80)

for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        val + 1,
        f"{val:.2f}",
        ha="center",
        fontsize=8
    )

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

print("Saved:", SAVE_PATH)

In [ ]:
import os

root = "/content/drive/MyDrive/PHD/Isaac_new/new_file2"

for dirpath, dirnames, filenames in os.walk(root):
    for f in filenames:
        if f.endswith(".csv"):
            print(os.path.join(dirpath, f))

In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/late_fusion_artifacts/microstructure_features_from_masks.csv"
df = pd.read_csv(CSV_PATH)

print(df.columns.tolist())
print(df.head())

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# FIGURE 8 — Morphological descriptor distributions (FINAL)
# =========================================================

CSV_PATH = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/late_fusion_artifacts/microstructure_features_from_masks.csv"
SAVE_PATH = "/content/drive/MyDrive/figure8_morphology_distributions_FINAL.png"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Could not find CSV file:\n{CSV_PATH}")

df = pd.read_csv(CSV_PATH)

# -------------------------
# columns from your CSV
# -------------------------

plot_df = pd.DataFrame({
    "Area fraction": df["gamma_prime_area_fraction"],
    "Equivalent diameter": df["gamma_prime_mean_eq_diameter_px"],
    "Aspect ratio": df["gamma_prime_mean_aspect_ratio"],
    "Circularity": df["gamma_prime_mean_circularity"],
}).dropna()

# -------------------------
# PLOT
# -------------------------

fig, axes = plt.subplots(2, 2, figsize=(10, 7.5))
axes = axes.ravel()

titles = [
    "(a) Area fraction",
    "(b) Equivalent diameter",
    "(c) Aspect ratio",
    "(d) Circularity",
]

xlabels = [
    "γ′ area fraction",
    "Equivalent diameter (px)",
    "Aspect ratio",
    "Circularity",
]

cols = [
    "Area fraction",
    "Equivalent diameter",
    "Aspect ratio",
    "Circularity",
]

for ax, title, xlabel, col in zip(axes, titles, xlabels, cols):

    ax.hist(plot_df[col], bins=25, edgecolor="black")

    ax.set_title(title, fontsize=10, pad=8)   # FIX overlap

    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Frequency", fontsize=9)

    ax.tick_params(labelsize=8)


# better spacing (fix overlap completely)
plt.subplots_adjust(
    left=0.08,
    right=0.98,
    top=0.95,
    bottom=0.08,
    wspace=0.28,
    hspace=0.35
)

plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", SAVE_PATH)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/deployment_predictions_and_metrics.csv"
SAVE_PATH = "/content/drive/MyDrive/figure9_deployment_summary_FINAL.png"

df = pd.read_csv(CSV_PATH)

plot_df = pd.DataFrame({
    "File": df["file"],
    "Area fraction": df["gamma_prime_area_fraction"],
    "Count": df["gamma_prime_cc_count"],
    "Equivalent diameter": df["gamma_prime_mean_eq_diameter_px"],
}).dropna()

plot_df = plot_df.sort_values("File").reset_index(drop=True)

x = range(len(plot_df))

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

# ---------- (a)
axes[0].plot(x, plot_df["Area fraction"], linewidth=2)
axes[0].set_title("(a) γ′ area fraction", fontsize=11, pad=6)
axes[0].set_ylabel("Area fraction", fontsize=10)
axes[0].grid(True, alpha=0.3)

# ---------- (b)
axes[1].plot(x, plot_df["Count"], linewidth=2)
axes[1].set_title("(b) γ′ count", fontsize=11, pad=6)
axes[1].set_ylabel("Count", fontsize=10)
axes[1].grid(True, alpha=0.3)

# ---------- (c)
axes[2].plot(x, plot_df["Equivalent diameter"], linewidth=2)
axes[2].set_title("(c) Mean equivalent diameter", fontsize=11, pad=6)
axes[2].set_ylabel("Diameter (px)", fontsize=10)
axes[2].set_xlabel("Image index", fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.subplots_adjust(hspace=0.35)

plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# =========================================================
# FIGURE 10 — Qualitative validation comparison (2 examples)
# =========================================================

VAL_IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
VAL_JSON    = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/coco_splits/val_single_gamma_prime.json"
SAVE_PATH   = "/content/drive/MyDrive/figure10_qualitative_validation_examples_FINAL.png"

# predictor must already be loaded
try:
    predictor
except NameError:
    raise ValueError(
        "predictor is not defined. Please load your trained Detectron2 predictor before running this cell."
    )

# -------------------------
# LOAD COCO
# -------------------------
coco = COCO(VAL_JSON)
folder_files = set(os.listdir(VAL_IMG_DIR))

# find matched validation images
matched_infos = []
for iid in coco.getImgIds():
    info = coco.loadImgs(iid)[0]
    if info["file_name"] in folder_files:
        matched_infos.append(info)

# case-insensitive fallback if needed
if len(matched_infos) < 2:
    lower_map = {f.lower(): f for f in folder_files}
    for iid in coco.getImgIds():
        info = coco.loadImgs(iid)[0]
        fn_lower = info["file_name"].lower()
        if fn_lower in lower_map:
            info = info.copy()
            info["file_name"] = lower_map[fn_lower]
            matched_infos.append(info)

# remove duplicates while preserving order
seen = set()
unique_infos = []
for info in matched_infos:
    fn = info["file_name"]
    if fn not in seen:
        seen.add(fn)
        unique_infos.append(info)

matched_infos = unique_infos

if len(matched_infos) < 2:
    raise ValueError("Could not find at least 2 validation images matching the image folder.")

# choose first 2 examples
selected_infos = matched_infos[:2]
print("Using images:")
for info in selected_infos:
    print("-", info["file_name"])

# -------------------------
# helper: build union mask
# -------------------------
def build_union_mask(coco_obj, img_info, H, W):
    ann_ids = coco_obj.getAnnIds(imgIds=[img_info["id"]])
    anns = coco_obj.loadAnns(ann_ids)

    union_mask = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        seg = ann["segmentation"]
        if isinstance(seg, list):  # polygon
            rles = maskUtils.frPyObjects(seg, H, W)
            rle = maskUtils.merge(rles)
        else:  # RLE
            rle = seg

        m = maskUtils.decode(rle)
        if m.ndim == 3:
            m = np.any(m, axis=2).astype(np.uint8)

        union_mask = np.maximum(union_mask, (m > 0).astype(np.uint8))

    return union_mask, anns

# -------------------------
# prepare figure
# -------------------------
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2))

panel_titles = [
    "(a) SEM image",
    "(b) Ground truth",
    "(c) Prediction",
    "(d) Overlay"
]

for row, img_info in enumerate(selected_infos):
    img_name = img_info["file_name"]
    img_path = os.path.join(VAL_IMG_DIR, img_name)

    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {img_path}")

    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    # GT mask
    gt_mask, anns = build_union_mask(coco, img_info, H, W)

    # Prediction
    outputs = predictor(img_bgr)
    instances = outputs["instances"].to("cpu")

    pred_mask = np.zeros((H, W), dtype=np.uint8)
    if instances.has("pred_masks") and len(instances) > 0:
        masks = instances.pred_masks.numpy().astype(np.uint8)
        pred_mask = np.any(masks, axis=0).astype(np.uint8)

    # Overlay
    overlay = img.copy()
    overlay[pred_mask > 0] = [255, 60, 60]
    blend = cv2.addWeighted(img, 0.80, overlay, 0.20, 0)

    row_imgs = [img, gt_mask, pred_mask, blend]

    for col in range(4):
        ax = axes[row, col]

        if col in [1, 2]:
            ax.imshow(row_imgs[col], cmap="gray")
        else:
            ax.imshow(row_imgs[col])

        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis("off")

        # titles only on top row
        if row == 0:
            ax.set_title(panel_titles[col], fontsize=9, pad=6)

        # thin border
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)

    # optional row label with filename
    axes[row, 0].text(
        0.02, -0.10,
        f"Example {row+1}: {img_name}",
        transform=axes[row, 0].transAxes,
        fontsize=8,
        ha="left",
        va="top"
    )

plt.subplots_adjust(
    left=0.03,
    right=0.99,
    top=0.92,
    bottom=0.08,
    wspace=0.05,
    hspace=0.18
)

plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", SAVE_PATH)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# =========================================================
# FIGURE 10 — Qualitative validation comparison (FINAL)
# Two validation examples:
#   (a) SEM image
#   (b) Ground truth
#   (c) Prediction
#   (d) Overlay
# =========================================================

VAL_IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/img_msk/images"
VAL_JSON    = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs/coco_splits/val_single_gamma_prime.json"
SAVE_PATH   = "/content/drive/MyDrive/figure10_qualitative_validation_examples_FINAL.png"

# predictor must already be loaded
try:
    predictor
except NameError:
    raise ValueError(
        "predictor is not defined. Please load your trained Detectron2 predictor before running this cell."
    )

# -------------------------
# LOAD COCO
# -------------------------
coco = COCO(VAL_JSON)
folder_files = set(os.listdir(VAL_IMG_DIR))

# find matched validation images
matched_infos = []
for iid in coco.getImgIds():
    info = coco.loadImgs(iid)[0]
    if info["file_name"] in folder_files:
        matched_infos.append(info)

# case-insensitive fallback if needed
if len(matched_infos) < 2:
    lower_map = {f.lower(): f for f in folder_files}
    for iid in coco.getImgIds():
        info = coco.loadImgs(iid)[0]
        fn_lower = info["file_name"].lower()
        if fn_lower in lower_map:
            info = info.copy()
            info["file_name"] = lower_map[fn_lower]
            matched_infos.append(info)

# remove duplicates while preserving order
seen = set()
unique_infos = []
for info in matched_infos:
    fn = info["file_name"]
    if fn not in seen:
        seen.add(fn)
        unique_infos.append(info)

matched_infos = unique_infos

if len(matched_infos) < 2:
    raise ValueError("Could not find at least 2 validation images matching the image folder.")

# choose first 2 examples
selected_infos = matched_infos[:2]

print("Using images:")
for info in selected_infos:
    print("-", info["file_name"])

# -------------------------
# helper: build union mask
# -------------------------
def build_union_mask(coco_obj, img_info, H, W):
    ann_ids = coco_obj.getAnnIds(imgIds=[img_info["id"]])
    anns = coco_obj.loadAnns(ann_ids)

    union_mask = np.zeros((H, W), dtype=np.uint8)

    for ann in anns:
        seg = ann["segmentation"]

        if isinstance(seg, list):  # polygon
            rles = maskUtils.frPyObjects(seg, H, W)
            rle = maskUtils.merge(rles)
        else:  # RLE
            rle = seg

        m = maskUtils.decode(rle)
        if m.ndim == 3:
            m = np.any(m, axis=2).astype(np.uint8)

        union_mask = np.maximum(union_mask, (m > 0).astype(np.uint8))

    return union_mask

# -------------------------
# prepare figure
# -------------------------
fig, axes = plt.subplots(2, 4, figsize=(12, 6.0))

panel_titles = [
    "(a) SEM image",
    "(b) Ground truth",
    "(c) Prediction",
    "(d) Overlay"
]

for row, img_info in enumerate(selected_infos):
    img_name = img_info["file_name"]
    img_path = os.path.join(VAL_IMG_DIR, img_name)

    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {img_path}")

    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    # Ground-truth mask
    gt_mask = build_union_mask(coco, img_info, H, W)

    # Prediction
    outputs = predictor(img_bgr)
    instances = outputs["instances"].to("cpu")

    pred_mask = np.zeros((H, W), dtype=np.uint8)
    if instances.has("pred_masks") and len(instances) > 0:
        masks = instances.pred_masks.numpy().astype(np.uint8)
        pred_mask = np.any(masks, axis=0).astype(np.uint8)

    # Overlay (lighter red, more texture visible)
    overlay = img.copy()
    overlay[pred_mask > 0] = [255, 60, 60]
    blend = cv2.addWeighted(img, 0.85, overlay, 0.15, 0)

    row_imgs = [img, gt_mask, pred_mask, blend]

    for col in range(4):
        ax = axes[row, col]

        if col in [1, 2]:
            ax.imshow(row_imgs[col], cmap="gray")
        else:
            ax.imshow(row_imgs[col])

        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis("off")

        # titles only on top row
        if row == 0:
            ax.set_title(panel_titles[col], fontsize=9, pad=6)

        # thin border
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)

plt.subplots_adjust(
    left=0.03,
    right=0.99,
    top=0.94,
    bottom=0.05,
    wspace=0.05,
    hspace=0.18
)

plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", SAVE_PATH)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

SAVE_PATH = "/content/drive/MyDrive/figure11_materials_workflow_FINAL.png"

fig, ax = plt.subplots(figsize=(14, 5), facecolor="white")
ax.set_facecolor("white")
ax.set_xlim(0, 16)
ax.set_ylim(0, 7)
ax.axis("off")


# ------------------------
# helpers
# ------------------------
def add_box(x, y, w, h, text, fontsize=12):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.8,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)
    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
    )


def add_arrow(x1, y1, x2, y2, lw=1.8):
    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=lw,
        color="black",
    )
    ax.add_patch(arrow)


def add_dashed_curve(x1, y1, x2, y2, rad, lw=1.5, arrow=False):
    arrowstyle = "->" if arrow else "-"
    patch = FancyArrowPatch(
        (x1, y1), (x2, y2),
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle=arrowstyle,
        mutation_scale=12,
        linewidth=lw,
        linestyle="--",
        color="black",
    )
    ax.add_patch(patch)


# ------------------------
# main workflow boxes
# ------------------------
add_box(0.4, 2.8, 1.8, 1.2, "SEM\nmicrographs")
add_box(2.8, 2.8, 2.0, 1.2, "γ′ instance\nsegmentation")
add_box(5.4, 2.8, 2.1, 1.2, "Morphology\nextraction")
add_box(8.1, 2.8, 2.1, 1.2, "Microstructure\ndatabase")
add_box(10.8, 2.8, 2.2, 1.2, "Structure–property\nML models")
add_box(13.7, 2.8, 1.8, 1.2, "Alloy / process\noptimization")

# main arrows
add_arrow(2.2, 3.4, 2.8, 3.4)
add_arrow(4.8, 3.4, 5.4, 3.4)
add_arrow(7.5, 3.4, 8.1, 3.4)
add_arrow(10.2, 3.4, 10.8, 3.4)
add_arrow(13.0, 3.4, 13.7, 3.4)

# lower callout boxes
add_box(
    5.2, 0.9, 2.5, 1.0,
    "Area fraction\nEquivalent diameter\nAspect ratio\nCircularity",
    fontsize=10,
)
add_box(
    10.6, 0.9, 2.6, 1.0,
    "Property prediction\nProcess mapping\nDesign screening",
    fontsize=10,
)

# connectors down
add_arrow(6.45, 2.8, 6.45, 1.9)
add_arrow(11.9, 2.8, 11.9, 1.9)

# ------------------------
# FINAL closed-loop path
# ------------------------
# left curved descent
add_dashed_curve(1.35, 2.78, 4.2, 0.42, rad=0.16, lw=1.5, arrow=False)

# bottom straight dashed segment
ax.plot(
    [4.2, 11.7], [0.42, 0.42],
    linestyle="--",
    color="black",
    linewidth=1.5
)

# right curved ascent routed OUTSIDE the lower-right box
add_dashed_curve(11.7, 0.42, 14.85, 2.78, rad=0.28, lw=1.5, arrow=True)

# loop label
ax.text(
    8.0, 2.15,
    "Closed-loop optimization",
    fontsize=10,
    ha="center"
)

# stage labels
ax.text(1.3, 4.35, "Imaging", fontsize=9, ha="center")
ax.text(3.8, 4.35, "Computer vision", fontsize=9, ha="center")
ax.text(6.45, 4.35, "Quantification", fontsize=9, ha="center")
ax.text(9.15, 4.35, "Data integration", fontsize=9, ha="center")
ax.text(11.9, 4.35, "Modeling", fontsize=9, ha="center")
ax.text(14.6, 4.35, "Decision support", fontsize=9, ha="center")

plt.tight_layout()
plt.savefig(
    SAVE_PATH,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

print("Saved to:", SAVE_PATH)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# =========================================================
# FIGURE 12 — Manual vs automated workflow comparison
# =========================================================

SAVE_PATH = "/content/drive/MyDrive/figure12_manual_vs_automated_workflow_FINAL.png"

fig, ax = plt.subplots(figsize=(13, 5.6), facecolor="white")
ax.set_facecolor("white")
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")

def add_box(x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.8,
        facecolor="white",
        edgecolor="black"
    )
    ax.add_patch(box)
    ax.text(
        x + w/2, y + h/2, text,
        ha="center", va="center", fontsize=fontsize
    )

def add_arrow(x1, y1, x2, y2, lw=1.6):
    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=lw,
        color="black"
    )
    ax.add_patch(arrow)

# -------------------------
# Titles
# -------------------------
ax.text(3.5, 7.3, "Manual workflow", ha="center", fontsize=13)
ax.text(10.5, 7.3, "Automated workflow", ha="center", fontsize=13)

# -------------------------
# Manual pipeline (left)
# -------------------------
add_box(0.7, 5.8, 2.2, 0.95, "SEM micrograph")
add_box(0.7, 4.3, 2.2, 0.95, "Manual annotation /\nthreshold selection")
add_box(0.7, 2.8, 2.2, 0.95, "Particle-by-particle\nmeasurement")
add_box(0.7, 1.3, 2.2, 0.95, "Spreadsheet /\nmanual export")

add_arrow(1.8, 5.8, 1.8, 5.25)
add_arrow(1.8, 4.3, 1.8, 3.75)
add_arrow(1.8, 2.8, 1.8, 2.25)

# Manual notes
ax.text(3.5, 5.0, "Operator dependent", fontsize=10, va="center")
ax.text(3.5, 3.5, "Low throughput", fontsize=10, va="center")
ax.text(3.5, 2.0, "Typically hours/sample", fontsize=10, va="center")

# -------------------------
# Automated pipeline (right)
# -------------------------
add_box(7.7, 5.8, 2.6, 0.95, "SEM micrograph")
add_box(7.7, 4.3, 2.6, 0.95, "Detectron2 γ′\ninstance segmentation")
add_box(7.7, 2.8, 2.6, 0.95, "Automated descriptor\nextraction")
add_box(7.7, 1.3, 2.6, 0.95, "CSV / database-ready\noutput")

add_arrow(9.0, 5.8, 9.0, 5.25)
add_arrow(9.0, 4.3, 9.0, 3.75)
add_arrow(9.0, 2.8, 9.0, 2.25)

# Automated notes
ax.text(11.2, 5.0, "Reproducible", fontsize=10, va="center")
ax.text(11.2, 3.5, "High throughput", fontsize=10, va="center")
ax.text(11.2, 2.0, "Seconds/sample on GPU", fontsize=10, va="center")

# -------------------------
# Bottom summary callout
# -------------------------
summary_box = FancyBboxPatch(
    (3.6, 0.35), 6.8, 0.55,
    boxstyle="round,pad=0.03,rounding_size=0.06",
    linewidth=1.4,
    facecolor="white",
    edgecolor="black"
)
ax.add_patch(summary_box)
ax.text(
    7.0, 0.62,
    "Automated γ′ segmentation enables scalable, quantitative microstructure characterization for materials informatics workflows.",
    ha="center", va="center", fontsize=10
)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved to:", SAVE_PATH)

In [ ]:
# -------------------------
# Bottom summary callout (FIXED)
# -------------------------

summary_box = FancyBboxPatch(
    (3.8, 0.6),   # moved up and centered
    6.4,          # narrower width
    0.7,
    boxstyle="round,pad=0.04,rounding_size=0.06",
    linewidth=1.4,
    facecolor="white",
    edgecolor="black"
)

ax.add_patch(summary_box)

ax.text(
    7.0,
    0.95,
    "Automated γ′ segmentation enables scalable, quantitative\n"
    "microstructure characterization for materials informatics workflows.",
    ha="center",
    va="center",
    fontsize=10,
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# =========================================================
# FIGURE 12 — Manual vs automated workflow comparison
# =========================================================

SAVE_PATH = "/content/drive/MyDrive/figure12_manual_vs_automated_workflow_FINAL.png"

fig, ax = plt.subplots(figsize=(13, 5.8), facecolor="white")

ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------

def add_box(x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.8,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
    )


def add_arrow(x1, y1, x2, y2):

    arrow = FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.6,
        color="black",
    )

    ax.add_patch(arrow)


# ---------------------------------------------------------
# Titles
# ---------------------------------------------------------

ax.text(3.5, 7.3, "Manual workflow", ha="center", fontsize=13)
ax.text(10.5, 7.3, "Automated workflow", ha="center", fontsize=13)


# ---------------------------------------------------------
# Manual workflow (LEFT)
# ---------------------------------------------------------

add_box(0.8, 5.8, 2.4, 1.0, "SEM micrograph")

add_box(
    0.8,
    4.3,
    2.4,
    1.0,
    "Manual annotation /\nthreshold selection",
)

add_box(
    0.8,
    2.8,
    2.4,
    1.0,
    "Particle-by-particle\nmeasurement",
)

add_box(
    0.8,
    1.3,
    2.4,
    1.0,
    "Spreadsheet /\nmanual export",
)

add_arrow(2.0, 5.8, 2.0, 5.25)
add_arrow(2.0, 4.3, 2.0, 3.75)
add_arrow(2.0, 2.8, 2.0, 2.25)

# manual notes

ax.text(3.6, 5.0, "Operator dependent", fontsize=10)
ax.text(3.6, 3.5, "Low throughput", fontsize=10)
ax.text(3.6, 2.0, "Typically hours/sample", fontsize=10)


# ---------------------------------------------------------
# Automated workflow (RIGHT)
# ---------------------------------------------------------

add_box(7.8, 5.8, 2.6, 1.0, "SEM micrograph")

add_box(
    7.8,
    4.3,
    2.6,
    1.0,
    "Detectron2 γ′\ninstance segmentation",
)

add_box(
    7.8,
    2.8,
    2.6,
    1.0,
    "Automated descriptor\nextraction",
)

add_box(
    7.8,
    1.3,
    2.6,
    1.0,
    "CSV / database-ready\noutput",
)

add_arrow(9.1, 5.8, 9.1, 5.25)
add_arrow(9.1, 4.3, 9.1, 3.75)
add_arrow(9.1, 2.8, 9.1, 2.25)

# automated notes

ax.text(11.0, 5.0, "Reproducible", fontsize=10)
ax.text(11.0, 3.5, "High throughput", fontsize=10)
ax.text(11.0, 2.0, "Seconds/sample on GPU", fontsize=10)


# ---------------------------------------------------------
# Bottom summary box (FIXED VERSION)
# ---------------------------------------------------------

summary_box = FancyBboxPatch(
    (3.8, 0.6),      # centered
    6.4,             # narrower
    0.7,
    boxstyle="round,pad=0.04,rounding_size=0.06",
    linewidth=1.4,
    facecolor="white",
    edgecolor="black",
)

ax.add_patch(summary_box)

ax.text(
    7.0,
    0.95,
    "Automated γ′ segmentation enables scalable, quantitative\n"
    "microstructure characterization for materials informatics workflows.",
    ha="center",
    va="center",
    fontsize=10,
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print("Saved to:", SAVE_PATH)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# =========================================================
# FIGURE 12 — Manual vs automated workflow comparison (FINAL FIXED)
# =========================================================

SAVE_PATH = "/content/drive/MyDrive/figure12_manual_vs_automated_workflow_FINAL.png"

fig, ax = plt.subplots(figsize=(13, 5.8), facecolor="white")
ax.set_facecolor("white")

ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------

def add_box(x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.8,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
    )


def add_arrow(x1, y1, x2, y2):
    arrow = FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.6,
        color="black",
    )
    ax.add_patch(arrow)


# ---------------------------------------------------------
# Titles (shifted slightly left)
# ---------------------------------------------------------

ax.text(3.2, 7.3, "Manual workflow", ha="center", fontsize=13)
ax.text(10.2, 7.3, "Automated workflow", ha="center", fontsize=13)


# ---------------------------------------------------------
# Manual workflow (LEFT)
# ---------------------------------------------------------

add_box(0.8, 5.8, 2.4, 1.0, "SEM micrograph")

add_box(
    0.8,
    4.3,
    2.4,
    1.0,
    "Manual annotation /\nthreshold selection",
)

add_box(
    0.8,
    2.8,
    2.4,
    1.0,
    "Particle-by-particle\nmeasurement",
)

add_box(
    0.8,
    1.3,
    2.4,
    1.0,
    "Spreadsheet /\nmanual export",
)

add_arrow(2.0, 5.8, 2.0, 5.25)
add_arrow(2.0, 4.3, 2.0, 3.75)
add_arrow(2.0, 2.8, 2.0, 2.25)

# manual notes
ax.text(3.6, 5.0, "Operator dependent", fontsize=10)
ax.text(3.6, 3.5, "Low throughput", fontsize=10)
ax.text(3.6, 2.0, "Typically hours/sample", fontsize=10)


# ---------------------------------------------------------
# Automated workflow (RIGHT)
# ---------------------------------------------------------

add_box(7.8, 5.8, 2.6, 1.0, "SEM micrograph")

add_box(
    7.8,
    4.3,
    2.6,
    1.0,
    "Detectron2 γ′\ninstance segmentation",
)

add_box(
    7.8,
    2.8,
    2.6,
    1.0,
    "Automated descriptor\nextraction",
)

add_box(
    7.8,
    1.3,
    2.6,
    1.0,
    "CSV / database-ready\noutput",
)

add_arrow(9.1, 5.8, 9.1, 5.25)
add_arrow(9.1, 4.3, 9.1, 3.75)
add_arrow(9.1, 2.8, 9.1, 2.25)

# automated notes
ax.text(11.0, 5.0, "Reproducible", fontsize=10)
ax.text(11.0, 3.5, "High throughput", fontsize=10)
ax.text(11.0, 2.0, "Seconds/sample on GPU", fontsize=10)


# ---------------------------------------------------------
# Bottom summary box (lowered so it does not intercept)
# ---------------------------------------------------------

summary_box = FancyBboxPatch(
    (3.9, 0.25),      # moved down
    6.2,              # slightly narrower
    0.75,
    boxstyle="round,pad=0.04,rounding_size=0.06",
    linewidth=1.4,
    facecolor="white",
    edgecolor="black",
)

ax.add_patch(summary_box)

ax.text(
    7.0,
    0.625,
    "Automated γ′ segmentation enables scalable, quantitative\n"
    "microstructure characterization for materials informatics workflows.",
    ha="center",
    va="center",
    fontsize=10,
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print("Saved to:", SAVE_PATH)

# **Thesis Figure Generation**

In [ ]:
# import os
# from pathlib import Path

# import cv2
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from pycocotools.coco import COCO
# from pycocotools import mask as maskUtils


# # =========================================================
# # REAL PATHS FROM YOUR NOTEBOOK
# # =========================================================
# OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

# IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"

# COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merged_gamma_prime_coco.json"

# PRED_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/pred_masks"

# OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figures_1_20_gt_vs_pred_refined"


# # =========================================================
# # DISPLAY CONTROL
# # =========================================================
# DISPLAY_N = 5   # show only the first 5 figures in notebook, save all 20

# # Optional: exact file list for the 20 figures
# # Example:
# # SELECTED_FILES = ["image_1", "image_5", "image_10"]
# SELECTED_FILES = []

# VALID_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


# # =========================================================
# # HELPERS
# # =========================================================
# def ensure_dir(path: str) -> None:
#     os.makedirs(path, exist_ok=True)


# def imread_rgb(path: str) -> np.ndarray:
#     img = cv2.imread(path, cv2.IMREAD_COLOR)
#     if img is None:
#         raise FileNotFoundError(f"Could not read image: {path}")
#     return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# def imread_mask(path: str) -> np.ndarray:
#     mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
#     if mask is None:
#         raise FileNotFoundError(f"Could not read mask: {path}")
#     return (mask > 0).astype(np.uint8)


# def resize_mask_to_image(mask: np.ndarray, image_shape: tuple[int, int, int]) -> np.ndarray:
#     h, w = image_shape[:2]
#     if mask.shape[:2] != (h, w):
#         mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
#         mask = (mask > 0).astype(np.uint8)
#     return mask


# def enhance_sem_contrast(image_rgb: np.ndarray) -> np.ndarray:
#     """
#     Mild CLAHE enhancement for clearer SEM display.
#     """
#     gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
#     gray_eq = clahe.apply(gray)
#     return np.stack([gray_eq, gray_eq, gray_eq], axis=-1)


# def ann_to_mask(ann, H, W):
#     seg = ann["segmentation"]
#     if isinstance(seg, list):  # polygons
#         rles = maskUtils.frPyObjects(seg, H, W)
#         rle = maskUtils.merge(rles)
#     else:  # RLE
#         rle = seg
#     m = maskUtils.decode(rle)
#     if m.ndim == 3:
#         m = np.any(m, axis=2).astype(np.uint8)
#     return (m > 0).astype(np.uint8)


# def build_gt_union_mask(coco_obj, file_name: str) -> np.ndarray:
#     imgs = coco_obj.loadImgs(coco_obj.getImgIds())
#     matches = [x for x in imgs if os.path.basename(x["file_name"]) == file_name]
#     if not matches:
#         raise FileNotFoundError(f"No COCO image entry found for: {file_name}")

#     info = matches[0]
#     H, W = info["height"], info["width"]
#     anns = coco_obj.loadAnns(coco_obj.getAnnIds(imgIds=[info["id"]]))

#     union = np.zeros((H, W), dtype=np.uint8)
#     for ann in anns:
#         union = np.maximum(union, ann_to_mask(ann, H, W))
#     return union


# def mask_boundary(mask: np.ndarray) -> np.ndarray:
#     edges = cv2.Canny((mask * 255).astype(np.uint8), 50, 150)
#     return (edges > 0).astype(np.uint8)


# def make_overlay(image_rgb: np.ndarray, gt_mask: np.ndarray, pred_mask: np.ndarray) -> np.ndarray:
#     """
#     GT only = green
#     Pred only = red
#     Overlap = yellow
#     GT boundary = green edge
#     Pred boundary = red edge
#     """
#     base = image_rgb.copy()

#     gt_only = (gt_mask == 1) & (pred_mask == 0)
#     pred_only = (gt_mask == 0) & (pred_mask == 1)
#     overlap = (gt_mask == 1) & (pred_mask == 1)

#     color = np.zeros_like(base)
#     color[gt_only] = [0, 255, 0]
#     color[pred_only] = [255, 0, 0]
#     color[overlap] = [255, 255, 0]

#     out = cv2.addWeighted(base, 0.72, color, 0.28, 0)

#     gt_edge = mask_boundary(gt_mask)
#     pred_edge = mask_boundary(pred_mask)

#     out[gt_edge == 1] = [0, 255, 0]
#     out[pred_edge == 1] = [255, 0, 0]

#     return out


# def dice_score(gt: np.ndarray, pred: np.ndarray) -> float:
#     gt_bool = gt.astype(bool)
#     pred_bool = pred.astype(bool)
#     inter = np.logical_and(gt_bool, pred_bool).sum()
#     denom = gt_bool.sum() + pred_bool.sum()
#     if denom == 0:
#         return 1.0
#     return 2.0 * inter / denom


# def iou_score(gt: np.ndarray, pred: np.ndarray) -> float:
#     gt_bool = gt.astype(bool)
#     pred_bool = pred.astype(bool)
#     inter = np.logical_and(gt_bool, pred_bool).sum()
#     union = np.logical_or(gt_bool, pred_bool).sum()
#     if union == 0:
#         return 1.0
#     return inter / union


# def area_fraction(mask: np.ndarray) -> float:
#     return float(mask.sum()) / float(mask.size)


# def matched_files(img_dir: str, pred_dir: str) -> list[str]:
#     """
#     Match original image stems to predicted mask stems.
#     Predicted masks are assumed to be saved like:
#     image_1_pred_mask.png
#     """
#     img_map = {p.stem: p for p in Path(img_dir).iterdir() if p.suffix.lower() in VALID_EXTS}

#     pred_map = {}
#     for p in Path(pred_dir).iterdir():
#         if p.suffix.lower() in VALID_EXTS:
#             stem = p.stem.replace("_pred_mask", "")
#             pred_map[stem] = p

#     common = sorted(set(img_map) & set(pred_map))
#     return common


# def get_pred_mask_path(pred_dir: str, stem: str) -> str:
#     for ext in VALID_EXTS:
#         p = Path(pred_dir) / f"{stem}_pred_mask{ext}"
#         if p.exists():
#             return str(p)
#     raise FileNotFoundError(f"Predicted mask not found for stem: {stem}")


# def get_img_path(img_dir: str, stem: str) -> str:
#     for ext in VALID_EXTS:
#         p = Path(img_dir) / f"{stem}{ext}"
#         if p.exists():
#             return str(p)
#     raise FileNotFoundError(f"Original image not found for stem: {stem}")


# def add_panel_title(ax, title: str) -> None:
#     ax.set_title(title, fontsize=12, fontweight="bold", pad=8)


# def add_footer_metrics(fig, dice: float, iou: float, gt_af: float, pred_af: float) -> None:
#     footer = (
#         f"Dice = {dice:.3f}   |   IoU = {iou:.3f}   |   "
#         f"GT area fraction = {gt_af:.3f}   |   Predicted area fraction = {pred_af:.3f}"
#     )
#     fig.text(0.5, 0.03, footer, ha="center", va="bottom", fontsize=10)


# # =========================================================
# # OPTIONAL PREVIEW GRID
# # =========================================================
# def preview_grid(stems, n=6):
#     coco = COCO(COCO_JSON)
#     picks = stems[:n]

#     cols = 3
#     rows = int(np.ceil(len(picks) / cols))

#     plt.figure(figsize=(5 * cols, 4.5 * rows))

#     for i, stem in enumerate(picks, start=1):
#         img_path = get_img_path(IMG_DIR, stem)
#         pred_path = get_pred_mask_path(PRED_DIR, stem)

#         img = imread_rgb(img_path)
#         img_enh = enhance_sem_contrast(img)
#         pred = imread_mask(pred_path)
#         gt = build_gt_union_mask(coco, os.path.basename(img_path))

#         gt = resize_mask_to_image(gt, img_enh.shape)
#         pred = resize_mask_to_image(pred, img_enh.shape)

#         overlay = make_overlay(img_enh, gt, pred)

#         ax = plt.subplot(rows, cols, i)
#         ax.imshow(overlay)
#         ax.set_title(stem, fontsize=11, fontweight="bold")
#         ax.axis("off")

#     plt.tight_layout()
#     plt.show()


# # =========================================================
# # MAIN GENERATOR
# # =========================================================
# def generate_figures_1_to_20(display_n: int = DISPLAY_N) -> pd.DataFrame:
#     ensure_dir(OUT_DIR)
#     coco = COCO(COCO_JSON)

#     if SELECTED_FILES:
#         stems = [Path(f).stem for f in SELECTED_FILES[:20]]
#     else:
#         stems = matched_files(IMG_DIR, PRED_DIR)[:20]

#     if len(stems) < 20:
#         raise ValueError(
#             f"Only {len(stems)} matched image/prediction pairs found. Need at least 20."
#         )

#     summary_rows = []

#     for idx, stem in enumerate(stems, start=1):
#         img_path = get_img_path(IMG_DIR, stem)
#         pred_path = get_pred_mask_path(PRED_DIR, stem)

#         img = imread_rgb(img_path)
#         img_enh = enhance_sem_contrast(img)

#         pred = imread_mask(pred_path)
#         gt = build_gt_union_mask(coco, os.path.basename(img_path))

#         gt = resize_mask_to_image(gt, img_enh.shape)
#         pred = resize_mask_to_image(pred, img_enh.shape)

#         overlay = make_overlay(img_enh, gt, pred)

#         dice = dice_score(gt, pred)
#         iou = iou_score(gt, pred)
#         gt_af = area_fraction(gt)
#         pred_af = area_fraction(pred)

#         fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.5))

#         axes[0].imshow(img_enh)
#         add_panel_title(axes[0], "Original SEM image")
#         axes[0].axis("off")

#         axes[1].imshow(gt, cmap="gray")
#         add_panel_title(axes[1], "Ground-truth mask")
#         axes[1].axis("off")

#         axes[2].imshow(pred, cmap="gray")
#         add_panel_title(axes[2], "Predicted mask")
#         axes[2].axis("off")

#         axes[3].imshow(overlay)
#         add_panel_title(axes[3], "Overlay")
#         axes[3].axis("off")

#         fig.suptitle(
#             f"Figure {idx}. Representative γ′ segmentation result for {stem}",
#             fontsize=13,
#             fontweight="bold",
#             y=0.98,
#         )

#         add_footer_metrics(fig, dice, iou, gt_af, pred_af)

#         plt.tight_layout(rect=[0, 0.07, 1, 0.94])

#         out_png = os.path.join(OUT_DIR, f"Figure_{idx:02d}_{stem}.png")
#         out_tif = os.path.join(OUT_DIR, f"Figure_{idx:02d}_{stem}.tif")

#         plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
#         plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")

#         if idx <= display_n:
#             plt.show()
#         else:
#             plt.close(fig)

#         summary_rows.append({
#             "figure_number": idx,
#             "file_stem": stem,
#             "image_path": img_path,
#             "pred_mask_path": pred_path,
#             "dice": round(dice, 4),
#             "iou": round(iou, 4),
#             "gt_area_fraction": round(gt_af, 4),
#             "pred_area_fraction": round(pred_af, 4),
#             "png_path": out_png,
#             "tif_path": out_tif,
#         })

#         print(f"Saved Figure {idx:02d}: {stem}")

#     summary_df = pd.DataFrame(summary_rows)
#     summary_csv = os.path.join(OUT_DIR, "Figures_1_20_summary.csv")
#     summary_df.to_csv(summary_csv, index=False)

#     print(f"\nDone. Summary saved to: {summary_csv}")
#     return summary_df


# # =========================================================
# # RUN
# # =========================================================
# if __name__ == "__main__":
#     stems_all = matched_files(IMG_DIR, PRED_DIR)

#     print(f"Matched image/prediction pairs found: {len(stems_all)}")

#     # Optional quick preview before full generation
#     preview_grid(stems_all, n=6)

#     # Save all 20, display only the first DISPLAY_N
#     summary_df = generate_figures_1_to_20(display_n=DISPLAY_N)

#     display(summary_df.head(10))

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils


# =========================================================
# REAL PATHS FROM YOUR NOTEBOOK
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"

COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merged_gamma_prime_coco.json"

PRED_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/pred_masks"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figures_1_20_gt_vs_pred_refined"


# =========================================================
# DISPLAY CONTROL
# =========================================================
DISPLAY_N = 5   # show only the first 5 figures in notebook, save all 20

# Optional: exact file list for the 20 figures
# Example:
# SELECTED_FILES = ["image_1", "image_5", "image_10"]
SELECTED_FILES = []

VALID_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


# =========================================================
# EXCLUSION CONTROL
# =========================================================
# These are the bad files you showed from coco_new2_RENAMED.
# We exclude them explicitly by filename stem.
EXCLUDED_STEMS = {
    "IMAGE1",
    "IMAGE2",
    "IMAGE3",
    "IMAGE4",
    "IMAGE5",
    "IMAGE6",
    "IMAGE7",
    "IMAGE8",
    "TESTIMAGE",
    "TESTIMAGEINV",
    "TESTIMAGEMINI",
}

# Also exclude by keywords if they appear in a filename/path/stem.
EXCLUDE_KEYWORDS = {
    "coco_new2_renamed",
    "testimage",
}

CASE_SENSITIVE_STEM_MATCH = False


# =========================================================
# HELPERS
# =========================================================
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def normalize_stem(stem: str) -> str:
    return stem if CASE_SENSITIVE_STEM_MATCH else stem.lower()


def is_excluded_stem(stem: str) -> bool:
    stem_cmp = normalize_stem(stem)

    excluded_cmp = {normalize_stem(x) for x in EXCLUDED_STEMS}
    if stem_cmp in excluded_cmp:
        return True

    for kw in EXCLUDE_KEYWORDS:
        if kw.lower() in stem_cmp:
            return True

    return False


def imread_rgb(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def imread_mask(path: str) -> np.ndarray:
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not read mask: {path}")
    return (mask > 0).astype(np.uint8)


def resize_mask_to_image(mask: np.ndarray, image_shape: tuple[int, int, int]) -> np.ndarray:
    h, w = image_shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.uint8)
    return mask


def enhance_sem_contrast(image_rgb: np.ndarray) -> np.ndarray:
    """
    Mild CLAHE enhancement for clearer SEM display.
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    return np.stack([gray_eq, gray_eq, gray_eq], axis=-1)


def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)


def build_gt_union_mask(coco_obj, file_name: str) -> np.ndarray:
    imgs = coco_obj.loadImgs(coco_obj.getImgIds())
    matches = [x for x in imgs if os.path.basename(x["file_name"]) == file_name]
    if not matches:
        raise FileNotFoundError(f"No COCO image entry found for: {file_name}")

    info = matches[0]
    H, W = info["height"], info["width"]
    anns = coco_obj.loadAnns(coco_obj.getAnnIds(imgIds=[info["id"]]))

    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union


def mask_boundary(mask: np.ndarray) -> np.ndarray:
    edges = cv2.Canny((mask * 255).astype(np.uint8), 50, 150)
    return (edges > 0).astype(np.uint8)


def make_overlay(image_rgb: np.ndarray, gt_mask: np.ndarray, pred_mask: np.ndarray) -> np.ndarray:
    """
    GT only = green
    Pred only = red
    Overlap = yellow
    GT boundary = green edge
    Pred boundary = red edge
    """
    base = image_rgb.copy()

    gt_only = (gt_mask == 1) & (pred_mask == 0)
    pred_only = (gt_mask == 0) & (pred_mask == 1)
    overlap = (gt_mask == 1) & (pred_mask == 1)

    color = np.zeros_like(base)
    color[gt_only] = [0, 255, 0]
    color[pred_only] = [255, 0, 0]
    color[overlap] = [255, 255, 0]

    out = cv2.addWeighted(base, 0.72, color, 0.28, 0)

    gt_edge = mask_boundary(gt_mask)
    pred_edge = mask_boundary(pred_mask)

    out[gt_edge == 1] = [0, 255, 0]
    out[pred_edge == 1] = [255, 0, 0]

    return out


def dice_score(gt: np.ndarray, pred: np.ndarray) -> float:
    gt_bool = gt.astype(bool)
    pred_bool = pred.astype(bool)
    inter = np.logical_and(gt_bool, pred_bool).sum()
    denom = gt_bool.sum() + pred_bool.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom


def iou_score(gt: np.ndarray, pred: np.ndarray) -> float:
    gt_bool = gt.astype(bool)
    pred_bool = pred.astype(bool)
    inter = np.logical_and(gt_bool, pred_bool).sum()
    union = np.logical_or(gt_bool, pred_bool).sum()
    if union == 0:
        return 1.0
    return inter / union


def area_fraction(mask: np.ndarray) -> float:
    return float(mask.sum()) / float(mask.size)


def matched_files(img_dir: str, pred_dir: str) -> list[str]:
    """
    Match original image stems to predicted mask stems.
    Predicted masks are assumed to be saved like:
    image_1_pred_mask.png

    Also excludes bad files from coco_new2_RENAMED.
    """
    img_map = {p.stem: p for p in Path(img_dir).iterdir() if p.suffix.lower() in VALID_EXTS}

    pred_map = {}
    for p in Path(pred_dir).iterdir():
        if p.suffix.lower() in VALID_EXTS:
            stem = p.stem.replace("_pred_mask", "")
            pred_map[stem] = p

    common = sorted(set(img_map) & set(pred_map))

    filtered = [stem for stem in common if not is_excluded_stem(stem)]

    removed = [stem for stem in common if is_excluded_stem(stem)]

    print(f"Matched before exclusion: {len(common)}")
    print(f"Excluded bad stems: {len(removed)}")
    if removed:
        print("Excluded stems:", removed)
    print(f"Matched after exclusion: {len(filtered)}")

    return filtered


def get_pred_mask_path(pred_dir: str, stem: str) -> str:
    for ext in VALID_EXTS:
        p = Path(pred_dir) / f"{stem}_pred_mask{ext}"
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"Predicted mask not found for stem: {stem}")


def get_img_path(img_dir: str, stem: str) -> str:
    for ext in VALID_EXTS:
        p = Path(img_dir) / f"{stem}{ext}"
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"Original image not found for stem: {stem}")


def add_panel_title(ax, title: str) -> None:
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)


def add_footer_metrics(fig, dice: float, iou: float, gt_af: float, pred_af: float) -> None:
    footer = (
        f"Dice = {dice:.3f}   |   IoU = {iou:.3f}   |   "
        f"GT area fraction = {gt_af:.3f}   |   Predicted area fraction = {pred_af:.3f}"
    )
    fig.text(0.5, 0.03, footer, ha="center", va="bottom", fontsize=10)


# =========================================================
# OPTIONAL PREVIEW GRID
# =========================================================
def preview_grid(stems, n=6):
    coco = COCO(COCO_JSON)
    picks = stems[:n]

    cols = 3
    rows = int(np.ceil(len(picks) / cols))

    plt.figure(figsize=(5 * cols, 4.5 * rows))

    for i, stem in enumerate(picks, start=1):
        img_path = get_img_path(IMG_DIR, stem)
        pred_path = get_pred_mask_path(PRED_DIR, stem)

        img = imread_rgb(img_path)
        img_enh = enhance_sem_contrast(img)
        pred = imread_mask(pred_path)
        gt = build_gt_union_mask(coco, os.path.basename(img_path))

        gt = resize_mask_to_image(gt, img_enh.shape)
        pred = resize_mask_to_image(pred, img_enh.shape)

        overlay = make_overlay(img_enh, gt, pred)

        ax = plt.subplot(rows, cols, i)
        ax.imshow(overlay)
        ax.set_title(stem, fontsize=11, fontweight="bold")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# =========================================================
# MAIN GENERATOR
# =========================================================
def generate_figures_1_to_20(display_n: int = DISPLAY_N) -> pd.DataFrame:
    ensure_dir(OUT_DIR)
    coco = COCO(COCO_JSON)

    if SELECTED_FILES:
        stems = [Path(f).stem for f in SELECTED_FILES if not is_excluded_stem(Path(f).stem)][:20]
    else:
        stems = matched_files(IMG_DIR, PRED_DIR)[:20]

    if len(stems) < 20:
        raise ValueError(
            f"Only {len(stems)} valid matched image/prediction pairs found after exclusion. Need at least 20."
        )

    summary_rows = []

    for idx, stem in enumerate(stems, start=1):
        img_path = get_img_path(IMG_DIR, stem)
        pred_path = get_pred_mask_path(PRED_DIR, stem)

        img = imread_rgb(img_path)
        img_enh = enhance_sem_contrast(img)

        pred = imread_mask(pred_path)
        gt = build_gt_union_mask(coco, os.path.basename(img_path))

        gt = resize_mask_to_image(gt, img_enh.shape)
        pred = resize_mask_to_image(pred, img_enh.shape)

        overlay = make_overlay(img_enh, gt, pred)

        dice = dice_score(gt, pred)
        iou = iou_score(gt, pred)
        gt_af = area_fraction(gt)
        pred_af = area_fraction(pred)

        fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.5))

        axes[0].imshow(img_enh)
        add_panel_title(axes[0], "Original SEM image")
        axes[0].axis("off")

        axes[1].imshow(gt, cmap="gray")
        add_panel_title(axes[1], "Ground-truth mask")
        axes[1].axis("off")

        axes[2].imshow(pred, cmap="gray")
        add_panel_title(axes[2], "Predicted mask")
        axes[2].axis("off")

        axes[3].imshow(overlay)
        add_panel_title(axes[3], "Overlay")
        axes[3].axis("off")

        fig.suptitle(
            f"Figure {idx}. Representative γ′ segmentation result for {stem}",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )

        add_footer_metrics(fig, dice, iou, gt_af, pred_af)

        plt.tight_layout(rect=[0, 0.07, 1, 0.94])

        out_png = os.path.join(OUT_DIR, f"Figure_{idx:02d}_{stem}.png")
        out_tif = os.path.join(OUT_DIR, f"Figure_{idx:02d}_{stem}.tif")

        plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
        plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")

        if idx <= display_n:
            plt.show()
        else:
            plt.close(fig)

        summary_rows.append({
            "figure_number": idx,
            "file_stem": stem,
            "image_path": img_path,
            "pred_mask_path": pred_path,
            "dice": round(dice, 4),
            "iou": round(iou, 4),
            "gt_area_fraction": round(gt_af, 4),
            "pred_area_fraction": round(pred_af, 4),
            "png_path": out_png,
            "tif_path": out_tif,
        })

        print(f"Saved Figure {idx:02d}: {stem}")

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUT_DIR, "Figures_1_20_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print(f"\nDone. Summary saved to: {summary_csv}")
    return summary_df


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    stems_all = matched_files(IMG_DIR, PRED_DIR)

    print(f"Valid matched image/prediction pairs found: {len(stems_all)}")

    # Optional quick preview before full generation
    preview_grid(stems_all, n=6)

    # Save all 20, display only the first DISPLAY_N
    summary_df = generate_figures_1_to_20(display_n=DISPLAY_N)

    display(summary_df.head(10))

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils


# =========================================================
# REAL PATHS FROM YOUR NOTEBOOK
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

IMG_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_images"

COCO_JSON = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merged_gamma_prime_coco.json"

PRED_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/pred_masks"

OUT_DIR = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/figures_1_20_gt_vs_pred_refined"

# From the merge step in your notebook
MERGE_REPORT_CSV = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2/merged_dataset_work/merged_annotations/merge_report.csv"


# =========================================================
# DISPLAY CONTROL
# =========================================================
DISPLAY_N = 5   # show only the first 5 figures in notebook, save all 20

# Optional: exact file list for the 20 figures
# Example:
# SELECTED_FILES = ["image_1", "image_5", "image_10"]
SELECTED_FILES = []

VALID_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


# =========================================================
# SOURCE EXCLUSION CONTROL
# =========================================================
# Exclude all images whose final source came from the bad import
EXCLUDED_FINAL_SOURCES = {"new_zip_2"}

# Optional extra filename-based safety net
EXCLUDED_STEMS_FALLBACK = {
    "IMAGE1",
    "IMAGE2",
    "IMAGE3",
    "IMAGE4",
    "IMAGE5",
    "IMAGE6",
    "IMAGE7",
    "IMAGE8",
    "TESTIMAGE",
    "TESTIMAGEINV",
    "TESTIMAGEMINI",
}


# =========================================================
# HELPERS
# =========================================================
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def normalize_name(name: str) -> str:
    return os.path.basename(str(name)).strip().lower()


def normalize_stem(stem: str) -> str:
    return Path(str(stem)).stem.strip().lower()


def load_excluded_stems_from_merge_report(
    merge_report_csv: str,
    excluded_final_sources: set[str]
) -> set[str]:
    """
    Read merge_report.csv and collect stems whose final_source belongs
    to excluded_final_sources, e.g. new_zip_2.
    """
    if not os.path.exists(merge_report_csv):
        print(f"WARNING: merge_report.csv not found: {merge_report_csv}")
        print("Falling back to filename-based exclusions only.")
        return {normalize_stem(x) for x in EXCLUDED_STEMS_FALLBACK}

    df = pd.read_csv(merge_report_csv)

    required_cols = {"file_name", "final_source"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"merge_report.csv is missing required columns: {missing_cols}. "
            f"Found columns: {list(df.columns)}"
        )

    excluded_rows = df[df["final_source"].astype(str).isin(excluded_final_sources)].copy()
    excluded_stems = {normalize_stem(x) for x in excluded_rows["file_name"].tolist()}

    # Add fallback stems as extra protection
    excluded_stems |= {normalize_stem(x) for x in EXCLUDED_STEMS_FALLBACK}

    print(f"Excluded stems from merge_report sources {excluded_final_sources}: {len(excluded_stems)}")
    if len(excluded_stems) > 0:
        print("Example excluded stems:", sorted(list(excluded_stems))[:15])

    return excluded_stems


EXCLUDED_STEMS = load_excluded_stems_from_merge_report(
    MERGE_REPORT_CSV,
    EXCLUDED_FINAL_SOURCES
)


def is_excluded_stem(stem: str) -> bool:
    return normalize_stem(stem) in EXCLUDED_STEMS


def imread_rgb(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def imread_mask(path: str) -> np.ndarray:
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not read mask: {path}")
    return (mask > 0).astype(np.uint8)


def resize_mask_to_image(mask: np.ndarray, image_shape: tuple[int, int, int]) -> np.ndarray:
    h, w = image_shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.uint8)
    return mask


def enhance_sem_contrast(image_rgb: np.ndarray) -> np.ndarray:
    """
    Mild CLAHE enhancement for clearer SEM display.
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_eq = clahe.apply(gray)
    return np.stack([gray_eq, gray_eq, gray_eq], axis=-1)


def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)


def build_gt_union_mask(coco_obj, file_name: str) -> np.ndarray:
    imgs = coco_obj.loadImgs(coco_obj.getImgIds())
    matches = [x for x in imgs if os.path.basename(x["file_name"]) == os.path.basename(file_name)]
    if not matches:
        raise FileNotFoundError(f"No COCO image entry found for: {file_name}")

    info = matches[0]
    H, W = info["height"], info["width"]
    anns = coco_obj.loadAnns(coco_obj.getAnnIds(imgIds=[info["id"]]))

    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union


def mask_boundary(mask: np.ndarray) -> np.ndarray:
    edges = cv2.Canny((mask * 255).astype(np.uint8), 50, 150)
    return (edges > 0).astype(np.uint8)


def make_overlay(image_rgb: np.ndarray, gt_mask: np.ndarray, pred_mask: np.ndarray) -> np.ndarray:
    """
    GT only = green
    Pred only = red
    Overlap = yellow
    GT boundary = green edge
    Pred boundary = red edge
    """
    base = image_rgb.copy()

    gt_only = (gt_mask == 1) & (pred_mask == 0)
    pred_only = (gt_mask == 0) & (pred_mask == 1)
    overlap = (gt_mask == 1) & (pred_mask == 1)

    color = np.zeros_like(base)
    color[gt_only] = [0, 255, 0]
    color[pred_only] = [255, 0, 0]
    color[overlap] = [255, 255, 0]

    out = cv2.addWeighted(base, 0.72, color, 0.28, 0)

    gt_edge = mask_boundary(gt_mask)
    pred_edge = mask_boundary(pred_mask)

    out[gt_edge == 1] = [0, 255, 0]
    out[pred_edge == 1] = [255, 0, 0]

    return out


def dice_score(gt: np.ndarray, pred: np.ndarray) -> float:
    gt_bool = gt.astype(bool)
    pred_bool = pred.astype(bool)
    inter = np.logical_and(gt_bool, pred_bool).sum()
    denom = gt_bool.sum() + pred_bool.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom


def iou_score(gt: np.ndarray, pred: np.ndarray) -> float:
    gt_bool = gt.astype(bool)
    pred_bool = pred.astype(bool)
    inter = np.logical_and(gt_bool, pred_bool).sum()
    union = np.logical_or(gt_bool, pred_bool).sum()
    if union == 0:
        return 1.0
    return inter / union


def area_fraction(mask: np.ndarray) -> float:
    return float(mask.sum()) / float(mask.size)


def matched_files(img_dir: str, pred_dir: str) -> list[str]:
    """
    Match original image stems to predicted mask stems.
    Predicted masks are assumed to be saved like:
    image_1_pred_mask.png

    Excludes all stems listed in EXCLUDED_STEMS.
    """
    img_map = {
        normalize_stem(p.stem): p
        for p in Path(img_dir).iterdir()
        if p.suffix.lower() in VALID_EXTS
    }

    pred_map = {}
    for p in Path(pred_dir).iterdir():
        if p.suffix.lower() in VALID_EXTS:
            stem = normalize_stem(p.stem.replace("_pred_mask", ""))
            pred_map[stem] = p

    common = sorted(set(img_map) & set(pred_map))
    filtered = [stem for stem in common if not is_excluded_stem(stem)]
    removed = [stem for stem in common if is_excluded_stem(stem)]

    print(f"Matched before exclusion: {len(common)}")
    print(f"Excluded by merge source / fallback: {len(removed)}")
    if removed:
        print("Example excluded stems:", removed[:15])
    print(f"Matched after exclusion: {len(filtered)}")

    return filtered


def get_pred_mask_path(pred_dir: str, stem: str) -> str:
    stem = normalize_stem(stem)
    for p in Path(pred_dir).iterdir():
        if p.suffix.lower() in VALID_EXTS:
            pred_stem = normalize_stem(p.stem.replace("_pred_mask", ""))
            if pred_stem == stem:
                return str(p)
    raise FileNotFoundError(f"Predicted mask not found for stem: {stem}")


def get_img_path(img_dir: str, stem: str) -> str:
    stem = normalize_stem(stem)
    for p in Path(img_dir).iterdir():
        if p.suffix.lower() in VALID_EXTS:
            if normalize_stem(p.stem) == stem:
                return str(p)
    raise FileNotFoundError(f"Original image not found for stem: {stem}")


def add_panel_title(ax, title: str) -> None:
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)


def add_footer_metrics(fig, dice: float, iou: float, gt_af: float, pred_af: float) -> None:
    footer = (
        f"Dice = {dice:.3f}   |   IoU = {iou:.3f}   |   "
        f"GT area fraction = {gt_af:.3f}   |   Predicted area fraction = {pred_af:.3f}"
    )
    fig.text(0.5, 0.03, footer, ha="center", va="bottom", fontsize=10)


# =========================================================
# OPTIONAL PREVIEW GRID
# =========================================================
def preview_grid(stems, n=6):
    coco = COCO(COCO_JSON)
    picks = stems[:n]

    cols = 3
    rows = int(np.ceil(len(picks) / cols))

    plt.figure(figsize=(5 * cols, 4.5 * rows))

    for i, stem in enumerate(picks, start=1):
        img_path = get_img_path(IMG_DIR, stem)
        pred_path = get_pred_mask_path(PRED_DIR, stem)

        img = imread_rgb(img_path)
        img_enh = enhance_sem_contrast(img)
        pred = imread_mask(pred_path)
        gt = build_gt_union_mask(coco, os.path.basename(img_path))

        gt = resize_mask_to_image(gt, img_enh.shape)
        pred = resize_mask_to_image(pred, img_enh.shape)

        overlay = make_overlay(img_enh, gt, pred)

        ax = plt.subplot(rows, cols, i)
        ax.imshow(overlay)
        ax.set_title(stem, fontsize=11, fontweight="bold")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# =========================================================
# MAIN GENERATOR
# =========================================================
def generate_figures_1_to_20(display_n: int = DISPLAY_N) -> pd.DataFrame:
    ensure_dir(OUT_DIR)
    coco = COCO(COCO_JSON)

    if SELECTED_FILES:
        stems = [normalize_stem(Path(f).stem) for f in SELECTED_FILES]
        stems = [s for s in stems if not is_excluded_stem(s)][:20]
    else:
        stems = matched_files(IMG_DIR, PRED_DIR)[:20]

    if len(stems) < 20:
        raise ValueError(
            f"Only {len(stems)} valid matched image/prediction pairs found after exclusion. Need at least 20."
        )

    summary_rows = []

    for idx, stem in enumerate(stems, start=1):
        img_path = get_img_path(IMG_DIR, stem)
        pred_path = get_pred_mask_path(PRED_DIR, stem)

        img = imread_rgb(img_path)
        img_enh = enhance_sem_contrast(img)

        pred = imread_mask(pred_path)
        gt = build_gt_union_mask(coco, os.path.basename(img_path))

        gt = resize_mask_to_image(gt, img_enh.shape)
        pred = resize_mask_to_image(pred, img_enh.shape)

        overlay = make_overlay(img_enh, gt, pred)

        dice = dice_score(gt, pred)
        iou = iou_score(gt, pred)
        gt_af = area_fraction(gt)
        pred_af = area_fraction(pred)

        fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.5))

        axes[0].imshow(img_enh)
        add_panel_title(axes[0], "Original SEM image")
        axes[0].axis("off")

        axes[1].imshow(gt, cmap="gray")
        add_panel_title(axes[1], "Ground-truth mask")
        axes[1].axis("off")

        axes[2].imshow(pred, cmap="gray")
        add_panel_title(axes[2], "Predicted mask")
        axes[2].axis("off")

        axes[3].imshow(overlay)
        add_panel_title(axes[3], "Overlay")
        axes[3].axis("off")

        fig.suptitle(
            f"Figure {idx}. Representative γ′ segmentation result for {Path(img_path).stem}",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )

        add_footer_metrics(fig, dice, iou, gt_af, pred_af)
        plt.tight_layout(rect=[0, 0.07, 1, 0.94])

        safe_stem = Path(img_path).stem
        out_png = os.path.join(OUT_DIR, f"Figure_{idx:02d}_{safe_stem}.png")
        out_tif = os.path.join(OUT_DIR, f"Figure_{idx:02d}_{safe_stem}.tif")

        plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
        plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")

        if idx <= display_n:
            plt.show()
        else:
            plt.close(fig)

        summary_rows.append({
            "figure_number": idx,
            "file_stem": safe_stem,
            "image_path": img_path,
            "pred_mask_path": pred_path,
            "dice": round(dice, 4),
            "iou": round(iou, 4),
            "gt_area_fraction": round(gt_af, 4),
            "pred_area_fraction": round(pred_af, 4),
            "png_path": out_png,
            "tif_path": out_tif,
        })

        print(f"Saved Figure {idx:02d}: {safe_stem}")

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUT_DIR, "Figures_1_20_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print(f"\nDone. Summary saved to: {summary_csv}")
    return summary_df


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    stems_all = matched_files(IMG_DIR, PRED_DIR)

    print(f"Valid matched image/prediction pairs found: {len(stems_all)}")

    # Optional quick preview before full generation
    preview_grid(stems_all, n=6)

    # Save all 20, display only the first DISPLAY_N
    summary_df = generate_figures_1_to_20(display_n=DISPLAY_N)

    try:
        from IPython.display import display
        display(summary_df.head(10))
    except Exception:
        print(summary_df.head(10))

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# PATHS
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

SUMMARY_CSV = os.path.join(
    OUT_ROOT,
    "figures_1_20_gt_vs_pred_refined",
    "Figures_1_20_summary.csv"
)

FIG_OUT_DIR = os.path.join(OUT_ROOT, "figures_next")
os.makedirs(FIG_OUT_DIR, exist_ok=True)


# =========================================================
# LOAD SUMMARY
# =========================================================
df = pd.read_csv(SUMMARY_CSV)

if df.empty:
    raise ValueError("Summary CSV is empty.")


# =========================================================
# SORT BY DICE
# =========================================================
df_plot = df.sort_values("dice", ascending=False).reset_index(drop=True)

labels = df_plot["file_stem"].astype(str).tolist()
dice_vals = df_plot["dice"].values
iou_vals = df_plot["iou"].values

x = range(len(df_plot))


# =========================================================
# FIGURE 21 — DICE / IOU PER IMAGE
# =========================================================
plt.figure(figsize=(14, 6))
bar_width = 0.4

x1 = [i - bar_width/2 for i in x]
x2 = [i + bar_width/2 for i in x]

plt.bar(x1, dice_vals, width=bar_width, label="Dice")
plt.bar(x2, iou_vals, width=bar_width, label="IoU")

plt.xticks(list(x), labels, rotation=60, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Selected image")
plt.title("Figure 21. Per-image Dice and IoU for the 20 selected γ′ segmentation examples")
plt.legend()
plt.tight_layout()

out_png = os.path.join(FIG_OUT_DIR, "Figure_21_per_image_dice_iou.png")
out_tif = os.path.join(FIG_OUT_DIR, "Figure_21_per_image_dice_iou.tif")

plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")
plt.show()
plt.close()

print("Saved:")
print(out_png)
print(out_tif)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# PATHS
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

SUMMARY_CSV = os.path.join(
    OUT_ROOT,
    "figures_1_20_gt_vs_pred_refined",
    "Figures_1_20_summary.csv"
)

FIG_OUT_DIR = os.path.join(OUT_ROOT, "figures_next")
os.makedirs(FIG_OUT_DIR, exist_ok=True)


# =========================================================
# LOAD SUMMARY
# =========================================================
df = pd.read_csv(SUMMARY_CSV)

if df.empty:
    raise ValueError("Summary CSV is empty.")


# =========================================================
# SORT BY DICE
# =========================================================
df_plot = df.sort_values("dice", ascending=False).reset_index(drop=True)

dice_vals = df_plot["dice"].values
iou_vals = df_plot["iou"].values

# Use simple numeric labels for cleaner presentation
labels = [str(i + 1) for i in range(len(df_plot))]
x = np.arange(len(df_plot))

mean_dice = np.mean(dice_vals)
mean_iou = np.mean(iou_vals)


# =========================================================
# FIGURE 21 — PER-IMAGE DICE / IOU
# =========================================================
plt.figure(figsize=(16, 6))
bar_width = 0.35

x1 = x - bar_width / 2
x2 = x + bar_width / 2

plt.bar(x1, dice_vals, width=bar_width, label="Dice")
plt.bar(x2, iou_vals, width=bar_width, label="IoU")

# Mean lines
plt.axhline(mean_dice, linestyle="--", linewidth=1.5, label=f"Mean Dice = {mean_dice:.3f}")
plt.axhline(mean_iou, linestyle=":", linewidth=1.5, label=f"Mean IoU = {mean_iou:.3f}")

# Axes and labels
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Selected image index (sorted by Dice)")
plt.title("Figure 21. Per-image segmentation performance across representative γ′ microstructures")

# Grid
plt.grid(axis="y", linestyle="--", alpha=0.5)

# Legend
plt.legend()

plt.tight_layout()

out_png = os.path.join(FIG_OUT_DIR, "Figure_21_per_image_dice_iou_corrected.png")
out_tif = os.path.join(FIG_OUT_DIR, "Figure_21_per_image_dice_iou_corrected.tif")

plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")
plt.show()
plt.close()

print("Saved:")
print(out_png)
print(out_tif)

# Optional: save ranked table used for the figure
ranked_csv = os.path.join(FIG_OUT_DIR, "Figure_21_ranked_metrics.csv")
df_plot.to_csv(ranked_csv, index=False)
print("Ranked metrics table saved to:")
print(ranked_csv)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# PATHS
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

SUMMARY_CSV = os.path.join(
    OUT_ROOT,
    "figures_1_20_gt_vs_pred_refined",
    "Figures_1_20_summary.csv"
)

FIG_OUT_DIR = os.path.join(OUT_ROOT, "figures_next")
os.makedirs(FIG_OUT_DIR, exist_ok=True)


# =========================================================
# LOAD SUMMARY
# =========================================================
df = pd.read_csv(SUMMARY_CSV)

if df.empty:
    raise ValueError("Summary CSV is empty.")


# =========================================================
# FIGURE 22 — GT VS PRED AREA FRACTION
# =========================================================
plt.figure(figsize=(7, 6))

plt.scatter(df["gt_area_fraction"], df["pred_area_fraction"])

min_v = min(df["gt_area_fraction"].min(), df["pred_area_fraction"].min())
max_v = max(df["gt_area_fraction"].max(), df["pred_area_fraction"].max())

plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")

plt.xlabel("Ground-truth area fraction")
plt.ylabel("Predicted area fraction")
plt.title("Figure 22. Ground-truth versus predicted γ′ area fraction for the 20 selected examples")
plt.tight_layout()

out_png = os.path.join(FIG_OUT_DIR, "Figure_22_gt_vs_pred_area_fraction.png")
out_tif = os.path.join(FIG_OUT_DIR, "Figure_22_gt_vs_pred_area_fraction.tif")

plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")
plt.show()
plt.close()

print("Saved:")
print(out_png)
print(out_tif)

In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

gt = df["gt_area_fraction"].values
pred = df["pred_area_fraction"].values

# Metrics
r2 = r2_score(gt, pred)
rmse = np.sqrt(mean_squared_error(gt, pred))

plt.figure(figsize=(7, 6))

plt.scatter(gt, pred, s=60)

min_v = min(gt.min(), pred.min())
max_v = max(gt.max(), pred.max())

plt.plot(
    [min_v, max_v],
    [min_v, max_v],
    linestyle="--",
    label="Ideal agreement"
)

plt.xlabel("Ground-truth area fraction")
plt.ylabel("Predicted area fraction")
plt.title("Figure 22. Predicted versus ground-truth γ′ area fraction")

# Add metrics text
plt.text(
    0.05,
    0.95,
    f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment='top'
)

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error


# =========================================================
# OUTPUT DIRECTORY (MAKE SURE THIS EXISTS)
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"
FIG_OUT_DIR = os.path.join(OUT_ROOT, "figures_next")

os.makedirs(FIG_OUT_DIR, exist_ok=True)


# =========================================================
# DATA
# =========================================================
gt = df["gt_area_fraction"].values
pred = df["pred_area_fraction"].values


# =========================================================
# METRICS
# =========================================================
r2 = r2_score(gt, pred)
rmse = np.sqrt(mean_squared_error(gt, pred))


# =========================================================
# PLOT
# =========================================================
plt.figure(figsize=(7, 6))

plt.scatter(gt, pred, s=60)

min_v = min(gt.min(), pred.min())
max_v = max(gt.max(), pred.max())

plt.plot(
    [min_v, max_v],
    [min_v, max_v],
    linestyle="--",
    label="Ideal agreement"
)

plt.xlabel("Ground-truth area fraction")
plt.ylabel("Predicted area fraction")
plt.title("Figure 22. Predicted versus ground-truth γ′ area fraction")

# Metrics text
plt.text(
    0.05,
    0.95,
    f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment='top'
)

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().set_aspect('equal', adjustable='box')

plt.tight_layout()


# =========================================================
# SAVE FIGURE  ← THIS WAS MISSING
# =========================================================
out_png = os.path.join(FIG_OUT_DIR, "Figure_22_gt_vs_pred_area_fraction.png")
out_tif = os.path.join(FIG_OUT_DIR, "Figure_22_gt_vs_pred_area_fraction.tif")

plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")


# =========================================================
# DISPLAY
# =========================================================
plt.show()
plt.close()


# =========================================================
# CONFIRM SAVE
# =========================================================
print("Saved to:")
print(out_png)
print(out_tif)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression


# =========================================================
# PATHS
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

SUMMARY_CSV = os.path.join(
    OUT_ROOT,
    "figures_1_20_gt_vs_pred_refined",
    "Figures_1_20_summary.csv"
)

FIG_OUT_DIR = os.path.join(OUT_ROOT, "figures_next")
os.makedirs(FIG_OUT_DIR, exist_ok=True)


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(SUMMARY_CSV)

if df.empty:
    raise ValueError("Summary CSV is empty.")

x = df["gt_area_fraction"].values.reshape(-1, 1)
y = df["dice"].values

# Linear fit
reg = LinearRegression()
reg.fit(x, y)
y_fit = reg.predict(x)

# Sort for smooth fitted line
order = np.argsort(x[:, 0])
x_sorted = x[:, 0][order]
y_fit_sorted = y_fit[order]

# Correlation
corr = np.corrcoef(df["gt_area_fraction"].values, df["dice"].values)[0, 1]


# =========================================================
# FIGURE 23
# =========================================================
plt.figure(figsize=(7, 6))

plt.scatter(
    df["gt_area_fraction"],
    df["dice"],
    s=60,
    label="Selected images"
)

plt.plot(
    x_sorted,
    y_fit_sorted,
    linestyle="--",
    linewidth=2,
    label="Linear trend"
)

plt.xlabel("Ground-truth γ′ area fraction")
plt.ylabel("Dice score")
plt.title("Figure 23. Dice score versus ground-truth γ′ area fraction")

plt.text(
    0.05,
    0.95,
    f"Correlation = {corr:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()

out_png = os.path.join(FIG_OUT_DIR, "Figure_23_dice_vs_area_fraction.png")
out_tif = os.path.join(FIG_OUT_DIR, "Figure_23_dice_vs_area_fraction.tif")

plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")

plt.show()
plt.close()

print("Saved to:")
print(out_png)
print(out_tif)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression


# =========================================================
# PATHS
# =========================================================
OUT_ROOT = "/content/drive/MyDrive/PHD/Isaac_new/new_file2/seg_paper1_outputs_sigma_push_with_coco_new_and_new2"

SUMMARY_CSV = os.path.join(
    OUT_ROOT,
    "figures_1_20_gt_vs_pred_refined",
    "Figures_1_20_summary.csv"
)

FIG_OUT_DIR = os.path.join(OUT_ROOT, "figures_next")
os.makedirs(FIG_OUT_DIR, exist_ok=True)


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(SUMMARY_CSV)

if df.empty:
    raise ValueError("Summary CSV is empty.")

x = df["gt_area_fraction"].values.reshape(-1, 1)
y = df["iou"].values

# Linear fit
reg = LinearRegression()
reg.fit(x, y)
y_fit = reg.predict(x)

# Sort for smooth fitted line
order = np.argsort(x[:, 0])
x_sorted = x[:, 0][order]
y_fit_sorted = y_fit[order]

# Correlation
corr = np.corrcoef(df["gt_area_fraction"].values, df["iou"].values)[0, 1]


# =========================================================
# FIGURE 24
# =========================================================
plt.figure(figsize=(7, 6))

plt.scatter(
    df["gt_area_fraction"],
    df["iou"],
    s=60,
    label="Selected images"
)

plt.plot(
    x_sorted,
    y_fit_sorted,
    linestyle="--",
    linewidth=2,
    label="Linear trend"
)

plt.xlabel("Ground-truth γ′ area fraction")
plt.ylabel("IoU")
plt.title("Figure 24. IoU versus ground-truth γ′ area fraction")

plt.text(
    0.05,
    0.95,
    f"Correlation = {corr:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()

out_png = os.path.join(FIG_OUT_DIR, "Figure_24_iou_vs_area_fraction.png")
out_tif = os.path.join(FIG_OUT_DIR, "Figure_24_iou_vs_area_fraction.tif")

plt.savefig(out_png, dpi=400, bbox_inches="tight", facecolor="white")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", facecolor="white")

plt.show()
plt.close()

print("Saved to:")
print(out_png)
print(out_tif)